In [ ]:
# ================================================================
# EUR/USD NFP MULTI-FACTOR STUDY
# ================================================================
#
# ONLY EXTERNAL INPUT:
# /content/drive/MyDrive/filtered_eurusd_nfp_events.csv
#
# Embedded macro factors:
#   1. Nonfarm Payrolls: Actual vs Forecast
#   2. Average Hourly Earnings MoM: Actual vs Forecast
#   3. Unemployment Rate U3: Actual vs Forecast
#
# Tests:
#   - Macro surprise alignment
#   - EUR/USD T1 / T5 / T10 / T15 response
#   - Walk-forward logistic regression
#   - Macro-only model
#   - Macro + first-minute confirmation model
#   - Probability threshold study
#
# No look-ahead Z-score normalization.
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)

PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

MIN_TRAIN = 60
MIN_Z_HISTORY = 24

LONG_THRESHOLD = 0.60
SHORT_THRESHOLD = 0.40


# ================================================================
# 2. EMBEDDED NFP DATA
# date,actual_k,forecast_k
# ================================================================

NFP_CSV = """
date,actual,forecast
2026-08-07,-23,85
2026-07-02,57,114
2026-06-05,172,85
2026-05-08,115,65
2026-04-03,178,65
2026-03-06,-92,58
2026-02-11,130,66
2026-01-09,50,66
2025-12-16,64,51
2025-11-20,119,53
2025-09-05,22,75
2025-08-01,73,106
2025-07-03,147,111
2025-06-06,139,126
2025-05-02,177,138
2025-04-04,228,137
2025-03-07,151,159
2025-02-07,143,169
2025-01-10,256,164
2024-12-06,227,202
2024-11-01,12,106
2024-10-04,254,147
2024-09-06,142,164
2024-08-02,114,176
2024-07-05,206,191
2024-06-07,272,182
2024-05-03,175,238
2024-04-05,303,212
2024-03-08,275,198
2024-02-02,353,187
2024-01-05,216,170
2023-12-08,199,180
2023-11-03,150,180
2023-10-06,336,170
2023-09-01,187,170
2023-08-04,187,200
2023-07-07,209,225
2023-06-02,339,180
2023-05-05,253,180
2023-04-07,236,239
2023-03-10,311,205
2023-02-03,517,185
2023-01-06,223,200
2022-12-02,263,200
2022-11-04,261,200
2022-10-07,263,250
2022-09-02,315,300
2022-08-05,528,250
2022-07-08,372,268
2022-06-03,390,325
2022-05-06,428,391
2022-04-01,431,490
2022-03-04,678,400
2022-02-04,467,150
2022-01-07,199,400
2021-12-03,210,550
2021-11-05,531,450
2021-10-08,194,500
2021-09-03,235,750
2021-08-06,943,870
2021-07-02,850,700
2021-06-04,559,650
2021-05-07,266,978
2021-04-02,916,647
2021-03-05,379,182
2021-02-05,49,50
2021-01-08,-140,71
2020-12-04,245,469
2020-11-06,638,600
2020-10-02,661,850
2020-09-04,1371,1400
2020-08-07,1763,1600
2020-07-02,4800,3000
2020-06-05,2509,-8000
2020-05-08,-20537,-22000
2020-04-03,-701,-100
2020-03-06,273,175
2020-02-07,225,160
2020-01-10,145,164
2019-12-06,266,186
2019-11-01,128,89
2019-10-04,136,140
2019-09-06,130,160
2019-08-02,164,164
2019-07-05,224,160
2019-06-07,75,185
2019-05-03,263,181
2019-04-05,196,175
2019-03-08,20,181
2019-02-01,304,165
2019-01-04,312,178
2018-12-07,155,200
2018-11-02,250,193
2018-10-05,134,185
2018-09-07,201,191
2018-08-03,157,193
2018-07-06,213,200
2018-06-01,223,189
2018-05-04,164,189
2018-04-06,103,193
2018-03-09,313,200
2018-02-02,200,184
2018-01-05,148,190
2017-12-08,228,200
2017-11-03,261,310
2017-10-06,-33,90
2017-09-01,156,180
2017-08-04,209,183
2017-07-07,222,179
2017-06-02,138,185
2017-05-05,211,185
2017-04-07,98,180
2017-03-10,235,200
2017-02-03,227,175
2017-01-06,156,178
2016-12-02,178,175
2016-11-04,161,175
2016-10-07,156,175
2016-09-02,151,180
2016-08-05,255,180
2016-07-08,287,175
2016-06-03,38,164
2016-05-06,160,202
2016-04-01,215,205
2016-03-04,242,190
2016-02-05,151,190
2016-01-08,292,200
2015-12-04,211,200
2015-11-06,271,180
2015-10-02,142,203
2015-09-04,173,220
2015-08-07,215,223
2015-07-02,223,230
2015-06-05,280,225
2015-05-08,223,224
2015-04-03,126,245
2015-03-06,295,240
2015-02-06,257,234
2015-01-09,252,240
2014-12-05,321,225
2014-11-07,214,231
2014-10-03,248,215
2014-09-05,142,225
2014-08-01,209,233
2014-07-03,288,212
2014-06-06,217,218
2014-05-02,288,210
2014-04-04,192,200
2014-03-07,175,149
2014-02-07,113,185
2014-01-10,74,196
2013-12-06,203,180
2013-11-08,204,125
2013-10-22,148,180
2013-09-06,169,180
2013-08-02,162,184
2013-07-05,195,165
2013-06-07,175,170
2013-05-03,165,145
2013-04-05,88,200
2013-03-08,236,160
2013-02-01,157,160
2013-01-04,155,150
2012-12-07,146,93
2012-11-02,171,125
2012-10-05,114,113
2012-09-07,96,125
2012-08-03,163,100
2012-07-06,80,90
2012-06-01,69,150
2012-05-04,115,170
2012-04-06,120,203
2012-03-09,227,210
2012-02-03,243,150
2012-01-06,200,150
2011-12-02,120,120
2011-11-04,80,95
2011-10-07,103,53
2011-09-02,0,74
2011-08-05,117,95
2011-07-08,18,89
2011-06-03,54,169
2011-05-06,244,185
2011-04-01,216,188
2011-03-04,192,180
2011-02-04,36,150
2011-01-07,103,150
2010-12-03,39,140
2010-11-05,151,60
2010-10-08,-95,3
2010-09-03,-54,-118
2010-08-06,-131,-65
2010-07-02,-125,-100
2010-06-04,431,500
2010-05-07,290,187
2010-04-02,162,190
2010-03-05,-36,-55
2010-02-05,-20,15
2010-01-08,-85,0
"""


# ================================================================
# 3. EMBEDDED AHE MOM DATA
# date,actual_pct,forecast_pct
# ================================================================

AHE_CSV = """
date,actual,forecast
2026-08-07,0.1,0.3
2026-07-02,0.3,0.3
2026-06-05,0.3,0.3
2026-05-08,0.2,0.3
2026-04-03,0.2,0.3
2026-03-06,0.4,0.3
2026-02-11,0.4,0.3
2026-01-09,0.3,0.3
2025-12-16,0.1,0.3
2025-11-20,0.2,0.3
2025-09-05,0.3,0.3
2025-08-01,0.3,0.3
2025-07-03,0.2,0.3
2025-06-06,0.4,0.3
2025-05-02,0.2,0.3
2025-04-04,0.3,0.3
2025-03-07,0.3,0.3
2025-02-07,0.5,0.3
2025-01-10,0.3,0.3
2024-12-06,0.4,0.3
2024-11-01,0.4,0.3
2024-10-04,0.4,0.3
2024-09-06,0.4,0.3
2024-08-02,0.2,0.3
2024-07-05,0.3,0.3
2024-06-07,0.4,0.3
2024-05-03,0.2,0.3
2024-04-05,0.3,0.3
2024-03-08,0.1,0.2
2024-02-02,0.6,0.3
2024-01-05,0.4,0.3
2023-12-08,0.4,0.3
2023-11-03,0.2,0.3
2023-10-06,0.2,0.3
2023-09-01,0.2,0.3
2023-08-04,0.4,0.3
2023-07-07,0.4,0.3
2023-06-02,0.3,0.4
2023-05-05,0.5,0.3
2023-04-07,0.3,0.3
2023-03-10,0.2,0.3
2023-02-03,0.3,0.3
2023-01-06,0.3,0.4
2022-12-02,0.6,0.3
2022-11-04,0.4,0.3
2022-10-07,0.3,0.3
2022-09-02,0.3,0.4
2022-08-05,0.5,0.3
2022-07-08,0.3,0.3
2022-06-03,0.3,0.4
2022-05-06,0.3,0.4
2022-04-01,0.4,0.4
2022-03-04,0.0,0.5
2022-02-04,0.7,0.5
2022-01-07,0.6,0.4
2021-12-03,0.3,0.4
2021-11-05,0.4,0.4
2021-10-08,0.6,0.4
2021-09-03,0.6,0.3
2021-08-06,0.4,0.3
2021-07-02,0.3,0.4
2021-06-04,0.5,0.2
2021-05-07,0.7,0.1
2021-04-02,-0.1,0.1
2021-03-05,0.2,0.2
2021-02-05,0.2,0.3
2021-01-08,0.8,0.2
2020-12-04,0.3,0.1
2020-11-06,0.1,0.2
2020-10-02,0.1,0.2
2020-08-07,0.2,-0.5
2020-07-02,-1.2,-0.7
2020-06-05,-1.0,1.0
2020-05-08,4.7,0.4
2020-04-03,0.4,0.2
2020-03-06,0.3,0.3
2020-02-07,0.2,0.3
2020-01-10,0.1,0.3
2019-12-06,0.2,0.3
2019-11-01,0.2,0.3
2019-10-04,0.0,0.3
2019-09-06,0.4,0.3
2019-08-02,0.3,0.2
2019-07-05,0.2,0.3
2019-06-07,0.2,0.3
2019-05-03,0.2,0.3
2019-04-05,0.1,0.2
2019-03-08,0.4,0.3
2019-02-01,0.1,0.3
2019-01-04,0.4,0.3
2018-12-07,0.2,0.3
2018-11-02,0.2,0.2
2018-10-05,0.3,0.3
2018-09-07,0.4,0.3
2018-08-03,0.3,0.3
2018-07-06,0.2,0.3
2018-06-01,0.3,0.2
2018-05-04,0.1,0.2
2018-04-06,0.3,0.3
2018-03-09,0.1,0.2
2018-02-02,0.3,0.3
2018-01-05,0.3,0.3
2017-12-08,0.2,0.3
2017-11-03,0.0,0.2
2017-10-06,0.5,0.3
2017-09-01,0.1,0.2
2017-08-04,0.3,0.3
2017-07-07,0.2,0.3
2017-06-02,0.2,0.2
2017-05-05,0.3,0.3
2017-04-07,0.2,0.2
2017-03-10,0.2,0.3
2017-02-03,0.1,0.3
2017-01-06,0.4,0.3
2016-12-02,-0.1,0.2
2016-11-04,0.4,0.3
2016-10-07,0.2,0.2
2016-09-02,0.1,0.2
2016-08-05,0.3,0.2
2016-07-08,0.1,0.2
2016-06-03,0.2,0.2
2016-05-06,0.3,0.3
2016-04-01,0.3,0.2
2016-03-04,-0.1,0.2
2016-02-05,0.5,0.3
2016-01-08,0.0,0.2
2015-12-04,0.2,0.2
2015-11-06,0.4,0.2
2015-10-02,0.0,0.2
2015-09-04,0.3,0.2
2015-08-07,0.2,0.2
2015-07-02,0.0,0.2
2015-06-05,0.3,0.2
2015-05-08,0.1,0.2
2015-04-03,0.3,0.2
2015-03-06,0.1,0.2
2015-02-06,0.5,0.3
2015-01-09,-0.2,0.2
2014-12-05,0.4,0.2
2014-11-07,0.1,0.2
2014-10-03,0.0,0.2
2014-09-05,0.2,0.2
2014-08-01,0.0,0.2
2014-07-03,0.2,0.2
2014-06-06,0.2,0.2
2014-05-02,0.0,0.2
2014-04-04,0.0,0.2
2014-03-07,0.4,0.2
2014-02-07,0.2,0.2
2014-01-10,0.1,0.2
2013-12-06,0.2,0.2
2013-11-08,0.1,0.2
2013-10-22,0.1,0.2
2013-09-06,0.2,0.2
2013-08-02,-0.1,0.2
2013-07-05,0.4,0.2
2013-06-07,0.0,0.2
2013-05-03,0.2,0.2
2013-04-05,0.0,0.2
2013-03-08,0.2,0.2
2013-02-01,0.2,0.2
2013-01-04,0.3,0.2
2012-12-07,0.2,0.2
2012-11-02,0.0,0.2
2012-10-05,0.3,0.2
2012-09-07,0.0,0.2
2012-08-03,0.1,0.2
2012-07-06,0.3,0.2
2012-06-01,0.1,0.2
2012-05-04,0.0,0.2
2012-04-06,0.2,0.2
2012-03-09,0.1,0.2
2012-02-03,0.2,0.2
2012-01-06,0.2,0.2
2011-12-02,-0.1,0.2
2011-11-04,0.2,0.2
2011-10-07,0.2,0.2
2011-09-02,-0.1,0.2
2011-08-05,0.4,0.2
2011-07-08,0.0,0.2
2011-06-03,0.3,0.2
2011-05-06,0.1,0.2
2011-04-01,0.0,0.2
2011-03-04,0.0,0.2
2011-02-04,0.4,0.2
2011-01-07,0.1,0.1
2010-12-03,0.0,0.1
2010-11-05,0.2,0.1
2010-10-08,0.0,0.1
2010-09-03,0.3,0.1
2010-08-06,0.2,0.1
2010-07-02,-0.1,0.1
2010-06-04,0.3,0.1
2010-05-07,0.0,0.2
2010-04-02,-0.1,0.2
2010-03-05,0.1,0.2
2010-02-05,0.3,0.2
2010-01-08,0.2,0.2
"""


# ================================================================
# 4. EMBEDDED U3 UNEMPLOYMENT DATA
# date,actual_pct,forecast_pct
# ================================================================

U3_CSV = """
date,actual,forecast
2026-08-07,4.1,4.2
2026-07-02,4.2,4.3
2026-06-05,4.3,4.3
2026-05-08,4.3,4.3
2026-04-03,4.3,4.4
2026-03-06,4.4,4.3
2026-02-11,4.3,4.4
2026-01-09,4.4,4.5
2025-12-16,4.6,4.5
2025-11-20,4.4,4.3
2025-09-05,4.3,4.3
2025-08-01,4.2,4.2
2025-07-03,4.1,4.3
2025-06-06,4.2,4.2
2025-05-02,4.2,4.2
2025-04-04,4.2,4.1
2025-03-07,4.1,4.0
2025-02-07,4.0,4.1
2025-01-10,4.1,4.2
2024-12-06,4.2,4.2
2024-11-01,4.1,4.1
2024-10-04,4.1,4.2
2024-09-06,4.2,4.2
2024-08-02,4.3,4.1
2024-07-05,4.1,4.0
2024-06-07,4.0,3.9
2024-05-03,3.9,3.8
2024-04-05,3.8,3.9
2024-03-08,3.9,3.7
2024-02-02,3.7,3.8
2024-01-05,3.7,3.8
2023-12-08,3.7,3.9
2023-11-03,3.9,3.8
2023-10-06,3.8,3.7
2023-09-01,3.8,3.5
2023-08-04,3.5,3.6
2023-07-07,3.6,3.6
2023-06-02,3.7,3.5
2023-05-05,3.4,3.6
2023-04-07,3.5,3.6
2023-03-10,3.6,3.4
2023-02-03,3.4,3.6
2023-01-06,3.5,3.7
2022-12-02,3.7,3.7
2022-11-04,3.7,3.6
2022-10-07,3.5,3.7
2022-09-02,3.7,3.5
2022-08-05,3.5,3.6
2022-07-08,3.6,3.6
2022-06-03,3.6,3.5
2022-05-06,3.6,3.5
2022-04-01,3.6,3.7
2022-03-04,3.8,3.9
2022-02-04,4.0,3.9
2022-01-07,3.9,4.1
2021-12-03,4.2,4.5
2021-11-05,4.6,4.7
2021-10-08,4.8,5.1
2021-09-03,5.2,5.2
2021-08-06,5.4,5.7
2021-07-02,5.9,5.7
2021-06-04,5.8,5.9
2021-05-07,6.1,5.8
2021-04-02,6.0,6.0
2021-03-05,6.2,6.3
2021-02-05,6.3,6.7
2021-01-08,6.7,6.8
2020-12-04,6.7,6.8
2020-11-06,6.9,7.7
2020-10-02,7.9,8.2
2020-09-04,8.4,9.8
2020-08-07,10.2,10.5
2020-07-02,11.1,12.3
2020-06-05,13.3,19.7
2020-05-08,14.7,16.0
2020-04-03,4.4,3.8
2020-03-06,3.5,3.6
2020-02-07,3.6,3.5
2020-01-10,3.5,3.5
2019-12-06,3.5,3.6
2019-11-01,3.6,3.6
2019-10-04,3.5,3.7
2019-09-06,3.7,3.7
2019-08-02,3.7,3.7
2019-07-05,3.7,3.6
2019-06-07,3.6,3.6
2019-05-03,3.6,3.8
2019-04-05,3.8,3.8
2019-03-08,3.8,3.9
2019-02-01,4.0,3.9
2019-01-04,3.9,3.7
2018-12-07,3.7,3.7
2018-11-02,3.7,3.7
2018-10-05,3.7,3.8
2018-09-07,3.9,3.8
2018-08-03,3.9,3.9
2018-07-06,4.0,3.8
2018-06-01,3.8,3.9
2018-05-04,3.9,4.0
2018-04-06,4.1,4.0
2018-03-09,4.1,4.0
2018-02-02,4.1,4.1
2018-01-05,4.1,4.1
2017-12-08,4.1,4.1
2017-11-03,4.1,4.2
2017-10-06,4.2,4.4
2017-09-01,4.4,4.3
2017-08-04,4.3,4.3
2017-07-07,4.4,4.3
2017-06-02,4.3,4.4
2017-05-05,4.4,4.6
2017-04-07,4.5,4.7
2017-03-10,4.7,4.7
2017-02-03,4.8,4.7
2017-01-06,4.7,4.7
2016-12-02,4.6,4.9
2016-11-04,4.9,4.9
2016-10-07,5.0,4.9
2016-09-02,4.9,4.8
2016-08-05,4.9,4.8
2016-07-08,4.9,4.8
2016-06-03,4.7,4.9
2016-05-06,5.0,5.0
2016-04-01,5.0,4.9
2016-03-04,4.9,4.9
2016-02-05,4.9,5.0
2016-01-08,5.0,5.0
2015-12-04,5.0,5.0
2015-11-06,5.0,5.1
2015-10-02,5.1,5.1
2015-09-04,5.1,5.2
2015-08-07,5.3,5.3
2015-07-02,5.3,5.4
2015-06-05,5.5,5.4
2015-05-08,5.4,5.4
2015-04-03,5.5,5.5
2015-03-06,5.5,5.6
2015-02-06,5.7,5.6
2015-01-09,5.6,5.7
2014-12-05,5.8,5.8
2014-11-07,5.8,5.9
2014-10-03,5.9,6.1
2014-09-05,6.1,6.1
2014-08-01,6.2,6.1
2014-07-03,6.1,6.3
2014-06-06,6.3,6.4
2014-05-02,6.3,6.6
2014-04-04,6.7,6.6
2014-03-07,6.7,6.6
2014-02-07,6.6,6.7
2014-01-10,6.7,7.0
2013-12-06,7.0,7.2
2013-11-08,7.3,7.3
2013-10-22,7.2,7.3
2013-09-06,7.3,7.4
2013-08-02,7.4,7.5
2013-07-05,7.6,7.5
2013-06-07,7.6,7.5
2013-05-03,7.5,7.6
2013-04-05,7.6,7.7
2013-03-08,7.7,7.9
2013-02-01,7.9,7.8
2013-01-04,7.8,7.7
2012-12-07,7.7,7.9
2012-11-02,7.9,7.9
2012-10-05,7.8,8.2
2012-09-07,8.1,8.3
2012-08-03,8.3,8.2
2012-07-06,8.2,8.2
2012-06-01,8.2,8.1
2012-05-04,8.1,8.2
2012-04-06,8.2,8.3
2012-03-09,8.3,8.3
2012-02-03,8.3,8.5
2012-01-06,8.5,8.7
2011-12-02,8.6,9.0
2011-11-04,9.0,9.1
2011-10-07,9.1,9.1
2011-09-02,9.1,9.1
2011-08-05,9.1,9.2
2011-07-08,9.2,9.1
2011-06-03,9.1,9.0
2011-05-06,9.0,8.8
2011-04-01,8.8,8.9
2011-03-04,8.9,9.1
2011-02-04,9.0,9.6
2011-01-07,9.4,9.7
2010-12-03,9.8,9.6
2010-11-05,9.6,9.6
2010-10-08,9.6,9.7
2010-09-03,9.6,9.6
2010-08-06,9.5,9.6
2010-07-02,9.5,9.8
2010-06-04,9.7,9.8
2010-05-07,9.9,9.7
2010-04-02,9.7,9.7
2010-03-05,9.7,9.8
2010-02-05,9.7,10.0
2010-01-08,10.0,10.1
"""


# ================================================================
# 5. LOAD EMBEDDED MACRO DATA
# ================================================================

nfp = pd.read_csv(
    StringIO(NFP_CSV.strip())
)

ahe = pd.read_csv(
    StringIO(AHE_CSV.strip())
)

u3 = pd.read_csv(
    StringIO(U3_CSV.strip())
)


for df in [nfp, ahe, u3]:
    df["date"] = pd.to_datetime(
        df["date"]
    )


nfp = nfp.rename(
    columns={
        "actual": "NFP_Actual",
        "forecast": "NFP_Forecast"
    }
)

ahe = ahe.rename(
    columns={
        "actual": "AHE_Actual",
        "forecast": "AHE_Forecast"
    }
)

u3 = u3.rename(
    columns={
        "actual": "U3_Actual",
        "forecast": "U3_Forecast"
    }
)


# ================================================================
# 6. MERGE THREE RELEASE SERIES
# ================================================================

macro = pd.merge(
    nfp,
    ahe,
    on="date",
    how="inner"
)

macro = pd.merge(
    macro,
    u3,
    on="date",
    how="inner"
)


macro = (
    macro
    .sort_values("date")
    .reset_index(drop=True)
)


# ================================================================
# 7. CALCULATE TRUE SURPRISES
# ================================================================

macro["NFP_Surprise"] = (
    macro["NFP_Actual"]
    -
    macro["NFP_Forecast"]
)


macro["AHE_Surprise"] = (
    macro["AHE_Actual"]
    -
    macro["AHE_Forecast"]
)


macro["U3_Surprise"] = (
    macro["U3_Actual"]
    -
    macro["U3_Forecast"]
)


# ------------------------------------------------
# USD-normalize unemployment.
#
# U3 actual BELOW forecast = stronger USD signal.
# ------------------------------------------------

macro["U3_USD_Surprise"] = (
    -macro["U3_Surprise"]
)


print("=" * 80)
print("MACRO DATA")
print("=" * 80)

print(
    "NFP rows:",
    len(nfp)
)

print(
    "AHE rows:",
    len(ahe)
)

print(
    "U3 rows:",
    len(u3)
)

print(
    "Three-factor matched rows:",
    len(macro)
)

print(
    "Date range:",
    macro["date"].min().date(),
    "->",
    macro["date"].max().date()
)


print("\nLatest macro rows:")

print(
    macro.tail(15)
    .round(3)
    .to_string(index=False)
)


# ================================================================
# 8. LOAD EUR/USD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


required_price_columns = {
    "Datetime",
    "Close"
}

missing = (
    required_price_columns
    -
    set(prices.columns)
)

if missing:

    raise ValueError(
        f"Price file missing columns: {missing}"
    )


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)

prices["Close"] = pd.to_numeric(
    prices["Close"],
    errors="coerce"
)


prices = (
    prices
    .dropna(
        subset=[
            "Datetime",
            "Close"
        ]
    )
    .sort_values("Datetime")
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("PRICE DATA")
print("=" * 80)

print(
    "Rows:",
    len(prices)
)

print(
    "Range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 9. EVENT WINDOW EXTRACTION
# ================================================================

def extract_window(
    price_df,
    release_date,
    release_time
):

    release_date = (
        pd.Timestamp(release_date)
        .date()
    )


    day = price_df[
        price_df["Datetime"].dt.date
        ==
        release_date
    ].copy()


    if day.empty:
        return None


    candidates = day[
        day["Datetime"]
        .dt.strftime("%H:%M")
        ==
        release_time
    ]


    if candidates.empty:
        return None


    start = candidates.iloc[0][
        "Datetime"
    ]


    # Exact 16 timestamps:
    # T0 through T15
    desired = [
        start
        +
        pd.Timedelta(minutes=i)
        for i in range(16)
    ]


    indexed = (
        day
        .drop_duplicates(
            subset="Datetime",
            keep="first"
        )
        .set_index("Datetime")
    )


    if not all(
        timestamp in indexed.index
        for timestamp in desired
    ):
        return None


    closes = np.array(
        [
            indexed.loc[
                timestamp,
                "Close"
            ]
            for timestamp in desired
        ],
        dtype=float
    )


    t0 = closes[0]


    pip_displacement = (
        closes - t0
    ) * 10000


    record = {
        "date":
            pd.Timestamp(release_date)
    }


    for i in range(16):

        record[f"T{i}"] = (
            pip_displacement[i]
        )


    return record


# ================================================================
# 10. AUTOMATIC TIMEZONE / RELEASE-TIME TEST
# ================================================================
#
# Possible timestamp conventions:
#
# 08:30 = New York time
# 12:30 = UTC during US daylight saving
# 13:30 = UTC during US standard time
# 14:30 / 15:30 = possible European-local timestamps
#
# Since DST means a single UTC time cannot cover the whole year,
# we ALSO test dynamic New York -> UTC mappings below.
# ================================================================

STATIC_TIMES = [
    "08:30",
    "12:30",
    "13:30",
    "14:30",
    "15:30"
]


static_counts = {}


for test_time in STATIC_TIMES:

    count = 0

    for d in macro["date"]:

        if extract_window(
            prices,
            d,
            test_time
        ) is not None:

            count += 1

    static_counts[test_time] = count


print("\n" + "=" * 80)
print("STATIC RELEASE-TIME MATCH TEST")
print("=" * 80)


for k, v in static_counts.items():

    print(
        f"{k}: {v} matched releases"
    )


# ================================================================
# 11. DYNAMIC UTC TIME
# ================================================================
#
# NFP releases are 08:30 America/New_York.
#
# If price timestamps are UTC:
# EDT -> 12:30 UTC
# EST -> 13:30 UTC
#
# ================================================================

def new_york_release_to_utc_time(date):

    dt = pd.Timestamp(
        date
    ).tz_localize(
        "America/New_York"
    )

    dt = dt.replace(
        hour=8,
        minute=30,
        second=0
    )

    utc = dt.tz_convert(
        "UTC"
    )

    return utc.strftime(
        "%H:%M"
    )


dynamic_utc_count = 0


for d in macro["date"]:

    t = new_york_release_to_utc_time(
        d
    )

    if extract_window(
        prices,
        d,
        t
    ) is not None:

        dynamic_utc_count += 1


print(
    "\nDynamic NY->UTC:",
    dynamic_utc_count,
    "matched releases"
)


# ================================================================
# 12. CHOOSE BEST MATCHING MODE
# ================================================================

best_static_time = max(
    static_counts,
    key=static_counts.get
)

best_static_count = (
    static_counts[
        best_static_time
    ]
)


if dynamic_utc_count > best_static_count:

    MATCH_MODE = "DYNAMIC_UTC"

    print(
        "\nSelected timestamp mode:",
        MATCH_MODE
    )

else:

    MATCH_MODE = best_static_time

    print(
        "\nSelected timestamp mode:",
        MATCH_MODE
    )


# ================================================================
# 13. BUILD T0-T15 WINDOWS
# ================================================================

windows = []


for d in macro["date"]:

    if MATCH_MODE == "DYNAMIC_UTC":

        time_string = (
            new_york_release_to_utc_time(
                d
            )
        )

    else:

        time_string = MATCH_MODE


    result = extract_window(
        prices,
        d,
        time_string
    )


    if result is not None:

        windows.append(
            result
        )


windows = pd.DataFrame(
    windows
)


print(
    "\nExtracted event windows:",
    len(windows)
)


if len(windows) < 50:

    raise ValueError(
        "\nFewer than 50 EUR/USD release windows matched.\n"
        "STOPPING because the timestamp alignment is probably wrong.\n"
        "Send me the STATIC RELEASE-TIME MATCH TEST output."
    )


# ================================================================
# 14. MERGE MACRO + EUR/USD
# ================================================================

data = pd.merge(
    macro,
    windows,
    on="date",
    how="inner"
)


data = (
    data
    .sort_values("date")
    .reset_index(drop=True)
)


print(
    "Final matched observations:",
    len(data)
)


# ================================================================
# 15. NO-LOOKAHEAD HISTORICAL Z-SCORES
# ================================================================

def expanding_zscore(
    series,
    min_periods=24
):

    s = pd.to_numeric(
        series,
        errors="coerce"
    )


    past_mean = (
        s.expanding(
            min_periods=min_periods
        )
        .mean()
        .shift(1)
    )


    past_std = (
        s.expanding(
            min_periods=min_periods
        )
        .std()
        .shift(1)
    )


    z = (
        s - past_mean
    ) / past_std


    return z.replace(
        [np.inf, -np.inf],
        np.nan
    )


data["NFP_Z"] = expanding_zscore(
    data["NFP_Surprise"],
    MIN_Z_HISTORY
)

data["AHE_Z"] = expanding_zscore(
    data["AHE_Surprise"],
    MIN_Z_HISTORY
)

data["U3_Z"] = expanding_zscore(
    data["U3_USD_Surprise"],
    MIN_Z_HISTORY
)


# ================================================================
# 16. CAP EXTREME COVID Z-SCORES
# ================================================================

for feature in [
    "NFP_Z",
    "AHE_Z",
    "U3_Z"
]:

    data[feature] = (
        data[feature]
        .clip(
            lower=-3,
            upper=3
        )
    )


# ================================================================
# 17. COMPOSITE MACRO SCORE
# ================================================================

data["Macro_Composite"] = (

      data["NFP_Z"]
    + data["AHE_Z"]
    + data["U3_Z"]

) / 3


# ================================================================
# 18. DISCRETE ALIGNMENT SCORE
# ================================================================
#
# +3 = all three USD positive
# -3 = all three USD negative
#
# ================================================================

data["NFP_Sign"] = np.sign(
    data["NFP_Surprise"]
)

data["AHE_Sign"] = np.sign(
    data["AHE_Surprise"]
)

data["U3_USD_Sign"] = np.sign(
    data["U3_USD_Surprise"]
)


data["Alignment_Score"] = (

      data["NFP_Sign"]
    + data["AHE_Sign"]
    + data["U3_USD_Sign"]

)


# ================================================================
# 19. PRICE TARGETS
# ================================================================

data["T1_Pips"] = data["T1"]
data["T5_Pips"] = data["T5"]
data["T10_Pips"] = data["T10"]
data["T15_Pips"] = data["T15"]


data["Up_T1"] = (
    data["T1"] > 0
).astype(int)

data["Up_T5"] = (
    data["T5"] > 0
).astype(int)

data["Up_T10"] = (
    data["T10"] > 0
).astype(int)

data["Up_T15"] = (
    data["T15"] > 0
).astype(int)


# ================================================================
# 20. RETURNS AFTER WAITING FOR T1
# ================================================================

data["T1_to_T5"] = (
    data["T5"]
    -
    data["T1"]
)

data["T1_to_T10"] = (
    data["T10"]
    -
    data["T1"]
)

data["T1_to_T15"] = (
    data["T15"]
    -
    data["T1"]
)


data["Up_T1_to_T5"] = (
    data["T1_to_T5"] > 0
).astype(int)

data["Up_T1_to_T10"] = (
    data["T1_to_T10"] > 0
).astype(int)

data["Up_T1_to_T15"] = (
    data["T1_to_T15"] > 0
).astype(int)


# ================================================================
# 21. FIRST-MINUTE REVERSAL
# ================================================================

data["T1_T15_Reversal"] = (

    np.sign(
        data["T1"]
    )

    !=

    np.sign(
        data["T15"]
    )

).astype(int)


# ================================================================
# 22. MACRO DIRECTION CORRECTNESS
# ================================================================
#
# Positive USD macro should generally mean EUR/USD DOWN.
#
# ================================================================

data["Macro_USD_Direction"] = np.sign(
    data["Macro_Composite"]
)


data["Expected_EURUSD_Direction"] = (
    -data["Macro_USD_Direction"]
)


data["Actual_T15_Direction"] = np.sign(
    data["T15"]
)


data["Macro_Direction_Correct"] = (

    data["Expected_EURUSD_Direction"]
    ==
    data["Actual_T15_Direction"]

).astype(int)


# ================================================================
# 23. ALIGNMENT ANALYSIS
# ================================================================

print("\n" + "=" * 80)
print("ALIGNMENT SCORE RESULTS")
print("=" * 80)


alignment_stats = (

    data
    .groupby(
        "Alignment_Score"
    )
    .agg(

        N=(
            "T15",
            "count"
        ),

        Mean_T1=(
            "T1",
            "mean"
        ),

        Mean_T5=(
            "T5",
            "mean"
        ),

        Mean_T10=(
            "T10",
            "mean"
        ),

        Mean_T15=(
            "T15",
            "mean"
        ),

        Median_T15=(
            "T15",
            "median"
        ),

        Mean_Abs_T15=(
            "T15",
            lambda x:
                x.abs().mean()
        ),

        EURUSD_Up_Pct=(
            "Up_T15",
            "mean"
        ),

        Reversal_Pct=(
            "T1_T15_Reversal",
            "mean"
        )
    )

    .reset_index()
)


alignment_stats[
    "EURUSD_Up_Pct"
] *= 100

alignment_stats[
    "Reversal_Pct"
] *= 100


print(
    alignment_stats
    .round(2)
    .to_string(index=False)
)


# ================================================================
# 24. CONCORDANCE REGIMES
# ================================================================

data["Concordance"] = np.select(

    [

        data[
            "Alignment_Score"
        ] == 3,

        data[
            "Alignment_Score"
        ] == -3,

        data[
            "Alignment_Score"
        ].abs() <= 1

    ],

    [

        "FULL_USD_BULL",

        "FULL_USD_BEAR",

        "CONFLICTED"

    ],

    default="PARTIAL"
)


concordance_stats = (

    data
    .groupby(
        "Concordance"
    )
    .agg(

        N=(
            "T15",
            "count"
        ),

        Mean_T1=(
            "T1",
            "mean"
        ),

        Mean_T5=(
            "T5",
            "mean"
        ),

        Mean_T10=(
            "T10",
            "mean"
        ),

        Mean_T15=(
            "T15",
            "mean"
        ),

        Median_T15=(
            "T15",
            "median"
        ),

        MeanAbs_T15=(
            "T15",
            lambda x:
                x.abs().mean()
        ),

        ReversalPct=(
            "T1_T15_Reversal",
            "mean"
        )

    )

    .reset_index()
)


concordance_stats[
    "ReversalPct"
] *= 100


print("\n" + "=" * 80)
print("CONCORDANCE RESULTS")
print("=" * 80)


print(
    concordance_stats
    .round(2)
    .to_string(index=False)
)


# ================================================================
# 25. SIMPLE CORRELATIONS
# ================================================================

print("\n" + "=" * 80)
print("SURPRISE / PRICE CORRELATIONS")
print("=" * 80)


correlation_columns = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z",
    "Macro_Composite",

    "T1",
    "T5",
    "T10",
    "T15",
    "T1_to_T15"

]


correlations = (
    data[
        correlation_columns
    ]
    .corr()
)


print(

    correlations.loc[

        [
            "NFP_Z",
            "AHE_Z",
            "U3_Z",
            "Macro_Composite"
        ],

        [
            "T1",
            "T5",
            "T10",
            "T15",
            "T1_to_T15"
        ]

    ]

    .round(3)

    .to_string()
)


# ================================================================
# 26. WALK-FORWARD LOGISTIC REGRESSION
# ================================================================

def walk_forward_model(
    df,
    features,
    target,
    min_train=60
):

    predictions = []
    coefficients = []


    for i in range(
        min_train,
        len(df)
    ):


        train = (

            df.iloc[:i]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        test = (

            df.iloc[[i]]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        if len(test) == 0:
            continue


        if len(train) < min_train:
            continue


        if (
            train[target]
            .nunique()
            <
            2
        ):
            continue


        X_train = train[
            features
        ]

        y_train = train[
            target
        ]

        X_test = test[
            features
        ]


        model = Pipeline([

            (
                "scale",
                StandardScaler()
            ),

            (
                "logit",

                LogisticRegression(

                    C=0.5,

                    penalty="l2",

                    solver="liblinear",

                    class_weight="balanced",

                    max_iter=5000

                )
            )
        ])


        model.fit(
            X_train,
            y_train
        )


        probability_up = (

            model
            .predict_proba(
                X_test
            )[0, 1]

        )


        predictions.append({

            "date":
                test[
                    "date"
                ].iloc[0],

            "Actual":
                int(
                    test[
                        target
                    ].iloc[0]
                ),

            "Probability_Up":
                probability_up

        })


        coef_values = (

            model
            .named_steps[
                "logit"
            ]
            .coef_[0]

        )


        coef_row = {

            "date":
                test[
                    "date"
                ].iloc[0]

        }


        for feature, value in zip(
            features,
            coef_values
        ):

            coef_row[
                feature
            ] = value


        coefficients.append(
            coef_row
        )


    return (
        pd.DataFrame(predictions),
        pd.DataFrame(coefficients)
    )


# ================================================================
# 27. MODEL A — MACRO ONLY -> T15
# ================================================================

MACRO_FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z"

]


pred_macro, coef_macro = (
    walk_forward_model(

        data,

        MACRO_FEATURES,

        "Up_T15",

        MIN_TRAIN

    )
)


# ================================================================
# 28. MODEL B — MACRO + T1 -> REMAINING MOVE
# ================================================================

T1_FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z",
    "T1"

]


pred_t1, coef_t1 = (
    walk_forward_model(

        data,

        T1_FEATURES,

        "Up_T1_to_T15",

        MIN_TRAIN

    )
)


# ================================================================
# 29. MODEL EVALUATION
# ================================================================

def evaluate_predictions(
    predictions,
    name
):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)


    if predictions.empty:

        print(
            "No predictions produced."
        )

        return


    y = predictions[
        "Actual"
    ]


    p = predictions[
        "Probability_Up"
    ]


    binary_prediction = (
        p >= 0.5
    ).astype(int)


    accuracy = accuracy_score(
        y,
        binary_prediction
    )


    balanced = (
        balanced_accuracy_score(
            y,
            binary_prediction
        )
    )


    if y.nunique() > 1:

        auc = roc_auc_score(
            y,
            p
        )

    else:

        auc = np.nan


    print(
        "OOS observations:",
        len(predictions)
    )


    print(
        f"Accuracy:             "
        f"{accuracy:.1%}"
    )


    print(
        f"Balanced accuracy:    "
        f"{balanced:.1%}"
    )


    print(
        f"ROC AUC:              "
        f"{auc:.3f}"
    )


evaluate_predictions(
    pred_macro,
    "MODEL A — MACRO ONLY -> T15"
)


evaluate_predictions(
    pred_t1,
    "MODEL B — MACRO + T1 -> T1 TO T15"
)


# ================================================================
# 30. ADD REAL PIP OUTCOMES
# ================================================================

pred_macro = pd.merge(

    pred_macro,

    data[
        [
            "date",
            "T15"
        ]
    ],

    on="date",

    how="left"
)


pred_t1 = pd.merge(

    pred_t1,

    data[
        [
            "date",
            "T1_to_T15"
        ]
    ],

    on="date",

    how="left"
)


# ================================================================
# 31. THRESHOLD STUDY
# ================================================================

def threshold_study(
    predictions,
    pip_column
):

    rows = []


    thresholds = [

        0.50,
        0.55,
        0.60,
        0.65,
        0.70,
        0.75

    ]


    for confidence in thresholds:


        longs = predictions[

            predictions[
                "Probability_Up"
            ]
            >=
            confidence

        ].copy()


        shorts = predictions[

            predictions[
                "Probability_Up"
            ]
            <=
            (
                1
                -
                confidence
            )

        ].copy()


        longs[
            "Trade_Pips"
        ] = longs[
            pip_column
        ]


        shorts[
            "Trade_Pips"
        ] = (
            -shorts[
                pip_column
            ]
        )


        trades = pd.concat(

            [
                longs,
                shorts
            ],

            ignore_index=True
        )


        if len(trades) == 0:
            continue


        rows.append({

            "Confidence":
                confidence,

            "Trades":
                len(trades),

            "Trade_Frequency":
                (
                    len(trades)
                    /
                    len(predictions)
                ),

            "Win_Rate":
                (
                    trades[
                        "Trade_Pips"
                    ]
                    >
                    0
                ).mean(),

            "Mean_Pips":
                trades[
                    "Trade_Pips"
                ].mean(),

            "Median_Pips":
                trades[
                    "Trade_Pips"
                ].median(),

            "Total_Pips":
                trades[
                    "Trade_Pips"
                ].sum(),

            "Mean_Winner":
                trades.loc[
                    trades[
                        "Trade_Pips"
                    ] > 0,
                    "Trade_Pips"
                ].mean(),

            "Mean_Loser":
                trades.loc[
                    trades[
                        "Trade_Pips"
                    ] < 0,
                    "Trade_Pips"
                ].mean()

        })


    return pd.DataFrame(
        rows
    )


macro_thresholds = (
    threshold_study(

        pred_macro,

        "T15"

    )
)


t1_thresholds = (
    threshold_study(

        pred_t1,

        "T1_to_T15"

    )
)


print("\n" + "=" * 80)
print("MODEL A — MACRO ONLY THRESHOLD STUDY")
print("=" * 80)


print(

    macro_thresholds
    .round(3)
    .to_string(index=False)

)


print("\n" + "=" * 80)
print("MODEL B — MACRO + T1 THRESHOLD STUDY")
print("=" * 80)


print(

    t1_thresholds
    .round(3)
    .to_string(index=False)

)


# ================================================================
# 32. COEFFICIENT STABILITY
# ================================================================

def coefficient_summary(
    coef_df,
    title
):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


    if coef_df.empty:

        print(
            "No coefficient data."
        )

        return


    features = [

        c
        for c in coef_df.columns
        if c != "date"

    ]


    rows = []


    for feature in features:


        series = coef_df[
            feature
        ]


        rows.append({

            "Feature":
                feature,

            "Mean_Coef":
                series.mean(),

            "Median_Coef":
                series.median(),

            "Positive_Pct":
                (
                    series > 0
                ).mean(),

            "Negative_Pct":
                (
                    series < 0
                ).mean()

        })


    result = pd.DataFrame(
        rows
    )


    result[
        "Abs_Mean"
    ] = (

        result[
            "Mean_Coef"
        ].abs()

    )


    result = (
        result
        .sort_values(
            "Abs_Mean",
            ascending=False
        )
    )


    print(

        result[
            [
                "Feature",
                "Mean_Coef",
                "Median_Coef",
                "Positive_Pct",
                "Negative_Pct"
            ]
        ]

        .round(3)

        .to_string(index=False)

    )


coefficient_summary(
    coef_macro,
    "MODEL A COEFFICIENTS"
)


coefficient_summary(
    coef_t1,
    "MODEL B COEFFICIENTS"
)


# ================================================================
# 33. MACRO COMPOSITE BUCKET ANALYSIS
# ================================================================

valid_composite = data.dropna(
    subset=[
        "Macro_Composite"
    ]
).copy()


valid_composite[
    "Composite_Bucket"
] = pd.qcut(

    valid_composite[
        "Macro_Composite"
    ],

    q=5,

    duplicates="drop"

)


bucket_stats = (

    valid_composite

    .groupby(
        "Composite_Bucket",
        observed=True
    )

    .agg(

        N=(
            "T15",
            "count"
        ),

        Avg_Macro_Score=(
            "Macro_Composite",
            "mean"
        ),

        Mean_T1=(
            "T1",
            "mean"
        ),

        Mean_T5=(
            "T5",
            "mean"
        ),

        Mean_T10=(
            "T10",
            "mean"
        ),

        Mean_T15=(
            "T15",
            "mean"
        ),

        Median_T15=(
            "T15",
            "median"
        )

    )

    .reset_index()

)


print("\n" + "=" * 80)
print("MACRO COMPOSITE QUINTILES")
print("=" * 80)


print(

    bucket_stats
    .round(3)
    .to_string(index=False)

)


# ================================================================
# 34. PLOT — MACRO COMPOSITE VS T15
# ================================================================

plt.figure(
    figsize=(10, 6)
)


plot_df = data.dropna(
    subset=[
        "Macro_Composite",
        "T15"
    ]
)


plt.scatter(
    plot_df[
        "Macro_Composite"
    ],
    plot_df[
        "T15"
    ],
    alpha=0.65
)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Three-Factor USD Macro Composite"
)


plt.ylabel(
    "EUR/USD T15 displacement (pips)"
)


plt.title(
    "NFP + Wages + Unemployment Surprise vs EUR/USD T+15"
)


plt.grid(
    True,
    alpha=0.3
)


plt.show()


# ================================================================
# 35. PLOT — MEAN T0-T15 TRAJECTORY
# FULL USD BULL VS FULL USD BEAR
# ================================================================

time_cols = [
    f"T{i}"
    for i in range(16)
]

minutes = list(
    range(16)
)


usd_bull = data[
    data[
        "Concordance"
    ]
    ==
    "FULL_USD_BULL"
]


usd_bear = data[
    data[
        "Concordance"
    ]
    ==
    "FULL_USD_BEAR"
]


if len(usd_bull) > 0:

    plt.figure(
        figsize=(10, 6)
    )


    plt.plot(

        minutes,

        usd_bull[
            time_cols
        ]
        .mean()
        .values,

        marker="o"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes after release"
    )


    plt.ylabel(
        "EUR/USD pip displacement"
    )


    plt.title(
        f"Fully USD-Bullish Macro Releases — Mean Path [n={len(usd_bull)}]"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


if len(usd_bear) > 0:

    plt.figure(
        figsize=(10, 6)
    )


    plt.plot(

        minutes,

        usd_bear[
            time_cols
        ]
        .mean()
        .values,

        marker="o"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes after release"
    )


    plt.ylabel(
        "EUR/USD pip displacement"
    )


    plt.title(
        f"Fully USD-Bearish Macro Releases — Mean Path [n={len(usd_bear)}]"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


# ================================================================
# 36. LATEST WALK-FORWARD SIGNALS
# ================================================================

def add_signal(
    df,
    long_threshold=0.60,
    short_threshold=0.40
):

    result = df.copy()


    result[
        "Signal"
    ] = np.select(

        [

            result[
                "Probability_Up"
            ]
            >=
            long_threshold,

            result[
                "Probability_Up"
            ]
            <=
            short_threshold

        ],

        [

            "LONG EUR/USD",

            "SHORT EUR/USD"

        ],

        default="NO TRADE"

    )


    return result


pred_macro = add_signal(
    pred_macro
)

pred_t1 = add_signal(
    pred_t1
)


print("\n" + "=" * 80)
print("LATEST MODEL A SIGNALS")
print("=" * 80)


print(

    pred_macro
    .tail(20)
    .round(3)
    .to_string(index=False)

)


print("\n" + "=" * 80)
print("LATEST MODEL B SIGNALS")
print("=" * 80)


print(

    pred_t1
    .tail(20)
    .round(3)
    .to_string(index=False)

)


# ================================================================
# 37. SAVE OUTPUTS
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "nfp_three_factor_master.csv",

    index=False

)


alignment_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "alignment_stats.csv",

    index=False

)


concordance_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "concordance_stats.csv",

    index=False

)


macro_thresholds.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_only_thresholds.csv",

    index=False

)


t1_thresholds.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_plus_t1_thresholds.csv",

    index=False

)


pred_macro.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_only_oos_predictions.csv",

    index=False

)


pred_t1.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_plus_t1_oos_predictions.csv",

    index=False

)


# ================================================================
# 38. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("FINISHED")
print("=" * 80)


print(
    "\nMatched macro/price releases:",
    len(data)
)


print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)


print(
    """

WHAT TO SEND BACK TO ME:

1. STATIC RELEASE-TIME MATCH TEST

2. Selected timestamp mode + final matched observations

3. ALIGNMENT SCORE RESULTS

4. CONCORDANCE RESULTS

5. MODEL A — MACRO ONLY -> T15

6. MODEL B — MACRO + T1 -> T1 TO T15

7. MODEL A THRESHOLD STUDY

8. MODEL B THRESHOLD STUDY

9. MODEL A COEFFICIENTS

10. MODEL B COEFFICIENTS

Those outputs will tell us whether:

- headline NFP contributes independently,
- wage surprises improve the signal,
- unemployment surprises improve it,
- full macro alignment reduces false breaks,
- waiting for T1 improves the edge,
- and whether higher model confidence actually produces
  better EUR/USD trades.

"""
)

In [ ]:
# ================================================================
# SELF-CONTAINED NFP TIMESTAMP / TIMEZONE DIAGNOSTIC
#
# Only external input:
# /content/drive/MyDrive/filtered_eurusd_nfp_events.csv
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from io import StringIO
from google.colab import drive


# ================================================================
# 1. MOUNT DRIVE
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)

PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


# ================================================================
# 2. EMBEDDED NFP RELEASE DATES
# ================================================================

NFP_DATES = """
date
2026-08-07
2026-07-02
2026-06-05
2026-05-08
2026-04-03
2026-03-06
2026-02-11
2026-01-09
2025-12-16
2025-11-20
2025-09-05
2025-08-01
2025-07-03
2025-06-06
2025-05-02
2025-04-04
2025-03-07
2025-02-07
2025-01-10
2024-12-06
2024-11-01
2024-10-04
2024-09-06
2024-08-02
2024-07-05
2024-06-07
2024-05-03
2024-04-05
2024-03-08
2024-02-02
2024-01-05
2023-12-08
2023-11-03
2023-10-06
2023-09-01
2023-08-04
2023-07-07
2023-06-02
2023-05-05
2023-04-07
2023-03-10
2023-02-03
2023-01-06
2022-12-02
2022-11-04
2022-10-07
2022-09-02
2022-08-05
2022-07-08
2022-06-03
2022-05-06
2022-04-01
2022-03-04
2022-02-04
2022-01-07
2021-12-03
2021-11-05
2021-10-08
2021-09-03
2021-08-06
2021-07-02
2021-06-04
2021-05-07
2021-04-02
2021-03-05
2021-02-05
2021-01-08
2020-12-04
2020-11-06
2020-10-02
2020-09-04
2020-08-07
2020-07-02
2020-06-05
2020-05-08
2020-04-03
2020-03-06
2020-02-07
2020-01-10
2019-12-06
2019-11-01
2019-10-04
2019-09-06
2019-08-02
2019-07-05
2019-06-07
2019-05-03
2019-04-05
2019-03-08
2019-02-01
2019-01-04
2018-12-07
2018-11-02
2018-10-05
2018-09-07
2018-08-03
2018-07-06
2018-06-01
2018-05-04
2018-04-06
2018-03-09
2018-02-02
2018-01-05
2017-12-08
2017-11-03
2017-10-06
2017-09-01
2017-08-04
2017-07-07
2017-06-02
2017-05-05
2017-04-07
2017-03-10
2017-02-03
2017-01-06
2016-12-02
2016-11-04
2016-10-07
2016-09-02
2016-08-05
2016-07-08
2016-06-03
2016-05-06
2016-04-01
2016-03-04
2016-02-05
2016-01-08
2015-12-04
2015-11-06
2015-10-02
2015-09-04
2015-08-07
2015-07-02
2015-06-05
2015-05-08
2015-04-03
2015-03-06
2015-02-06
2015-01-09
2014-12-05
2014-11-07
2014-10-03
2014-09-05
2014-08-01
2014-07-03
2014-06-06
2014-05-02
2014-04-04
2014-03-07
2014-02-07
2014-01-10
2013-12-06
2013-11-08
2013-10-22
2013-09-06
2013-08-02
2013-07-05
2013-06-07
2013-05-03
2013-04-05
2013-03-08
2013-02-01
2013-01-04
2012-12-07
2012-11-02
2012-10-05
2012-09-07
2012-08-03
2012-07-06
2012-06-01
2012-05-04
2012-04-06
2012-03-09
2012-02-03
2012-01-06
2011-12-02
2011-11-04
2011-10-07
2011-09-02
2011-08-05
2011-07-08
2011-06-03
2011-05-06
2011-04-01
2011-03-04
2011-02-04
2011-01-07
2010-12-03
2010-11-05
2010-10-08
2010-09-03
2010-08-06
2010-07-02
2010-06-04
2010-05-07
2010-04-02
2010-03-05
2010-02-05
2010-01-08
"""


release_df = pd.read_csv(
    StringIO(NFP_DATES.strip())
)

release_df["date"] = pd.to_datetime(
    release_df["date"]
)

release_dates = (
    release_df["date"]
    .dt.date
    .tolist()
)


# ================================================================
# 3. LOAD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)

prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)

for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:
    if col in prices.columns:
        prices[col] = pd.to_numeric(
            prices[col],
            errors="coerce"
        )

prices = (
    prices
    .dropna(
        subset=[
            "Datetime",
            "Close"
        ]
    )
    .sort_values("Datetime")
    .reset_index(drop=True)
)


print("=" * 80)
print("PRICE DATA")
print("=" * 80)

print(
    "Range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)

print(
    "Rows:",
    len(prices)
)

print(
    "NFP release dates:",
    len(release_dates)
)


# ================================================================
# 4. CANDIDATE CLOCK TIMES
# ================================================================
#
# Scan every 30 minutes from 07:00 through 16:30.
# ================================================================

candidate_times = []

for hour in range(7, 17):

    for minute in [
        0,
        30
    ]:

        candidate_times.append(
            f"{hour:02d}:{minute:02d}"
        )


# ================================================================
# 5. EVENT METRIC FUNCTION
# ================================================================

def get_event_metrics(
    price_df,
    release_date,
    time_string
):

    day = price_df[
        price_df["Datetime"].dt.date
        ==
        release_date
    ].copy()

    if day.empty:
        return None


    event_rows = day[
        day["Datetime"]
        .dt.strftime("%H:%M")
        ==
        time_string
    ]

    if event_rows.empty:
        return None


    event_time = (
        event_rows.iloc[0]["Datetime"]
    )


    indexed = (
        day
        .drop_duplicates(
            subset="Datetime",
            keep="first"
        )
        .set_index("Datetime")
    )


    required_times = {
        "T0":
            event_time,

        "T1":
            event_time
            + pd.Timedelta(minutes=1),

        "T5":
            event_time
            + pd.Timedelta(minutes=5),

        "T15":
            event_time
            + pd.Timedelta(minutes=15)
    }


    if not all(
        t in indexed.index
        for t in required_times.values()
    ):
        return None


    p0 = indexed.loc[
        required_times["T0"],
        "Close"
    ]

    p1 = indexed.loc[
        required_times["T1"],
        "Close"
    ]

    p5 = indexed.loc[
        required_times["T5"],
        "Close"
    ]

    p15 = indexed.loc[
        required_times["T15"],
        "Close"
    ]


    result = {
        "Abs_T1":
            abs(p1 - p0) * 10000,

        "Abs_T5":
            abs(p5 - p0) * 10000,

        "Abs_T15":
            abs(p15 - p0) * 10000
    }


    # Event-bar high-low range
    if (
        "High" in indexed.columns
        and
        "Low" in indexed.columns
    ):

        high = indexed.loc[
            required_times["T0"],
            "High"
        ]

        low = indexed.loc[
            required_times["T0"],
            "Low"
        ]


        if (
            pd.notna(high)
            and
            pd.notna(low)
        ):

            result[
                "Event_Bar_Range"
            ] = (
                high - low
            ) * 10000

        else:

            result[
                "Event_Bar_Range"
            ] = np.nan

    else:

        result[
            "Event_Bar_Range"
        ] = np.nan


    return result


# ================================================================
# 6. RUN DIAGNOSTIC
# ================================================================

rows = []


for time_string in candidate_times:

    metrics = []


    for release_date in release_dates:

        result = get_event_metrics(
            prices,
            release_date,
            time_string
        )


        if result is not None:

            metrics.append(
                result
            )


    if len(metrics) == 0:
        continue


    temp = pd.DataFrame(
        metrics
    )


    rows.append({

        "Time":
            time_string,

        "N":
            len(temp),

        "Median_Abs_T1":
            temp[
                "Abs_T1"
            ].median(),

        "Mean_Abs_T1":
            temp[
                "Abs_T1"
            ].mean(),

        "Median_Abs_T5":
            temp[
                "Abs_T5"
            ].median(),

        "Mean_Abs_T5":
            temp[
                "Abs_T5"
            ].mean(),

        "Median_Abs_T15":
            temp[
                "Abs_T15"
            ].median(),

        "Mean_Abs_T15":
            temp[
                "Abs_T15"
            ].mean(),

        "Median_Event_Bar_Range":
            temp[
                "Event_Bar_Range"
            ].median(),

        "Mean_Event_Bar_Range":
            temp[
                "Event_Bar_Range"
            ].mean()
    })


diagnostic = pd.DataFrame(
    rows
)


# ================================================================
# 7. FULL RESULTS
# ================================================================

print("\n" + "=" * 80)
print("ALL CANDIDATE TIMES")
print("=" * 80)


print(
    diagnostic
    .round(2)
    .to_string(index=False)
)


# ================================================================
# 8. RANK BY MEDIAN T1
# ================================================================

ranked_t1 = (
    diagnostic
    .sort_values(
        "Median_Abs_T1",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("RANKED BY MEDIAN ABSOLUTE T1 MOVE")
print("=" * 80)


print(
    ranked_t1[
        [
            "Time",
            "N",
            "Median_Abs_T1",
            "Mean_Abs_T1",
            "Median_Event_Bar_Range",
            "Median_Abs_T5",
            "Median_Abs_T15"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


# ================================================================
# 9. RANK BY EVENT BAR RANGE
# ================================================================

ranked_range = (
    diagnostic
    .sort_values(
        "Median_Event_Bar_Range",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("RANKED BY EVENT-MINUTE HIGH-LOW RANGE")
print("=" * 80)


print(
    ranked_range[
        [
            "Time",
            "N",
            "Median_Event_Bar_Range",
            "Mean_Event_Bar_Range",
            "Median_Abs_T1",
            "Median_Abs_T5",
            "Median_Abs_T15"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


# ================================================================
# 10. CHART
# ================================================================

plot_df = (
    diagnostic
    .sort_values("Time")
)


plt.figure(
    figsize=(14, 6)
)

plt.plot(
    plot_df["Time"],
    plot_df["Median_Abs_T1"],
    marker="o"
)

plt.xticks(
    rotation=45
)

plt.xlabel(
    "Clock time in HistData CSV"
)

plt.ylabel(
    "Median absolute T1 move (pips)"
)

plt.title(
    "NFP-Day EUR/USD First-Minute Volatility by CSV Clock Time"
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ================================================================
# 11. TOP 5
# ================================================================

print("\n" + "=" * 80)
print("TOP 5 CANDIDATE NFP TIMES")
print("=" * 80)


print(
    ranked_t1.head(5)
    .round(2)
    .to_string(index=False)
)

In [ ]:
# ================================================================
# EUR/USD NFP THREE-FACTOR EVENT STUDY — CLEAN FROM-SCRATCH VERSION
# ================================================================
#
# INPUTS:
#
# 1. EUR/USD M1 prices:
#    /content/drive/MyDrive/filtered_eurusd_nfp_events.csv
#
# 2. Macro table previously created from your supplied:
#       NFP Actual / Forecast
#       AHE MoM Actual / Forecast
#       U3 Actual / Forecast
#
#    /content/drive/MyDrive/nfp_three_factor_results/
#    nfp_three_factor_master.csv
#
#
# CONFIRMED:
# HistData clock time 08:30 = NFP volatility event.
#
#
# IMPORTANT CHANGE:
#
# We now use:
#
#    08:29 CLOSE = PRE-RELEASE PRICE
#
# and measure:
#
#    08:30 event close
#    08:31
#    ...
#    08:45
#
# This captures the actual announcement candle.
#
#
# OUTPUT:
#
# - Direction analysis
# - Move magnitude analysis
# - MFE / MAE
# - Reversal / continuation
# - Concordant / conflicted macro regimes
# - Interaction model
# - 20 / 30 pip move model
# - Out-of-sample walk-forward tests
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LogisticRegression,
    Ridge
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error
)


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_v3"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_TIME = "08:30"

MIN_TRAIN = 60

MIN_Z_HISTORY = 24


CONFIDENCE_LEVELS = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


# ================================================================
# 3. IMPORTANT:
# KEEP ONLY MACRO COLUMNS
#
# This prevents old T0/T15 columns from the previous experiment
# entering the new merge.
# ================================================================

macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"
]


missing_macro = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing_macro:

    print(
        "\nColumns in macro file:"
    )

    print(
        macro_raw.columns.tolist()
    )

    raise ValueError(
        f"Missing macro columns: {missing_macro}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values(
        "date"
    )

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


print("=" * 80)
print("MACRO INPUT")
print("=" * 80)

print(
    "Macro releases:",
    len(macro)
)

print(
    "Date range:",
    macro["date"].min(),
    "->",
    macro["date"].max()
)


# ================================================================
# 4. LOAD EUR/USD M1
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values(
        "Datetime"
    )

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


print("\n" + "=" * 80)
print("PRICE INPUT")
print("=" * 80)

print(
    "Rows:",
    len(prices)
)

print(
    "Price range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 5. BUILD FAST DATETIME INDEX
# ================================================================

price_indexed = (
    prices
    .set_index(
        "Datetime"
    )
    .sort_index()
)


# ================================================================
# 6. EXTRACT ONE NFP EVENT
# ================================================================
#
# PRE:
#   08:29 close
#
# EVENT:
#   08:30
#
# T1:
#   08:31
#
# ...
#
# T15:
#   08:45
#
# ================================================================

def extract_nfp_event(
    release_date
):

    release_date = pd.Timestamp(
        release_date
    )


    event_dt = release_date.replace(
        hour=8,
        minute=30,
        second=0
    )


    pre_dt = (
        event_dt
        -
        pd.Timedelta(minutes=1)
    )


    timestamps = [

        pre_dt

    ] + [

        event_dt
        +
        pd.Timedelta(minutes=i)

        for i in range(16)

    ]


    if not all(
        t in price_indexed.index
        for t in timestamps
    ):

        return None


    pre_close = float(
        price_indexed.loc[
            pre_dt,
            "Close"
        ]
    )


    record = {

        "date":
            release_date.normalize(),

        "Pre_Close":
            pre_close
    }


    # ============================================================
    # CLOSE DISPLACEMENT
    # ============================================================

    for i in range(16):

        timestamp = (
            event_dt
            +
            pd.Timedelta(minutes=i)
        )


        close = float(
            price_indexed.loc[
                timestamp,
                "Close"
            ]
        )


        record[
            f"C{i}_Pips"
        ] = (
            close
            -
            pre_close
        ) * 10000


    # ============================================================
    # EVENT CANDLE
    # ============================================================

    event_row = price_indexed.loc[
        event_dt
    ]


    event_open = float(
        event_row["Open"]
    )

    event_high = float(
        event_row["High"]
    )

    event_low = float(
        event_row["Low"]
    )

    event_close = float(
        event_row["Close"]
    )


    record[
        "Event_Open_Gap_Pips"
    ] = (
        event_open
        -
        pre_close
    ) * 10000


    record[
        "Event_Close_Pips"
    ] = (
        event_close
        -
        pre_close
    ) * 10000


    record[
        "Event_Range_Pips"
    ] = (
        event_high
        -
        event_low
    ) * 10000


    record[
        "Event_High_From_Pre"
    ] = (
        event_high
        -
        pre_close
    ) * 10000


    record[
        "Event_Low_From_Pre"
    ] = (
        event_low
        -
        pre_close
    ) * 10000


    # ============================================================
    # 08:30 -> 08:45 HIGH / LOW PATH
    # ============================================================

    highs = []
    lows = []


    for i in range(16):

        timestamp = (
            event_dt
            +
            pd.Timedelta(minutes=i)
        )


        row = price_indexed.loc[
            timestamp
        ]


        highs.append(

            (
                float(row["High"])
                -
                pre_close
            )

            * 10000

        )


        lows.append(

            (
                float(row["Low"])
                -
                pre_close
            )

            * 10000

        )


    record[
        "Max_Up_15m"
    ] = max(
        highs
    )


    record[
        "Max_Down_15m"
    ] = min(
        lows
    )


    record[
        "Total_Range_15m"
    ] = (
        max(highs)
        -
        min(lows)
    )


    # ============================================================
    # KEY CLOSE HORIZONS
    # ============================================================

    record[
        "T1_Pips"
    ] = record[
        "C1_Pips"
    ]


    record[
        "T5_Pips"
    ] = record[
        "C5_Pips"
    ]


    record[
        "T10_Pips"
    ] = record[
        "C10_Pips"
    ]


    record[
        "T15_Pips"
    ] = record[
        "C15_Pips"
    ]


    return record


# ================================================================
# 7. BUILD ALL EVENT WINDOWS
# ================================================================

events = []


for release_date in macro[
    "date"
]:

    event = extract_nfp_event(
        release_date
    )

    if event is not None:

        events.append(
            event
        )


event_df = pd.DataFrame(
    events
)


print("\n" + "=" * 80)
print("EVENT EXTRACTION")
print("=" * 80)

print(
    "Events extracted:",
    len(event_df)
)


if len(event_df) < 100:

    raise ValueError(
        "Too few NFP event windows extracted."
    )


# ================================================================
# 8. CLEAN MERGE
# ================================================================
#
# Since macro contains ONLY macro columns,
# there can be no T15 collision now.
#
# ================================================================

data = pd.merge(

    macro,

    event_df,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values(
        "date"
    )

    .reset_index(
        drop=True
    )

)


# ================================================================
# 9. HARD VALIDATION
# ================================================================

required_price_cols = [

    "Event_Close_Pips",
    "Event_Range_Pips",

    "T1_Pips",
    "T5_Pips",
    "T10_Pips",
    "T15_Pips",

    "Max_Up_15m",
    "Max_Down_15m",
    "Total_Range_15m"
]


missing_price = [

    c
    for c in required_price_cols
    if c not in data.columns

]


if missing_price:

    raise ValueError(
        f"Missing price columns: {missing_price}"
    )


print(
    "Final matched events:",
    len(data)
)

print(
    "Date range:",
    data["date"].min(),
    "->",
    data["date"].max()
)


print("\nSample:")

print(

    data[
        [
            "date",

            "NFP_Surprise",
            "AHE_Surprise",
            "U3_USD_Surprise",

            "Event_Close_Pips",
            "T1_Pips",
            "T5_Pips",
            "T10_Pips",
            "T15_Pips",
            "Event_Range_Pips"
        ]
    ]

    .tail(10)

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 10. HISTORICAL Z-SCORE — NO FUTURE LEAKAGE
# ================================================================

def historical_zscore(
    series,
    minimum_history=24
):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )


    mean = (

        series

        .expanding(
            min_periods=
                minimum_history
        )

        .mean()

        .shift(1)

    )


    std = (

        series

        .expanding(
            min_periods=
                minimum_history
        )

        .std()

        .shift(1)

    )


    z = (
        series
        -
        mean
    ) / std


    return z.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


data["NFP_Z"] = (
    historical_zscore(
        data[
            "NFP_Surprise"
        ],
        MIN_Z_HISTORY
    )
)


data["AHE_Z"] = (
    historical_zscore(
        data[
            "AHE_Surprise"
        ],
        MIN_Z_HISTORY
    )
)


data["U3_Z"] = (
    historical_zscore(
        data[
            "U3_USD_Surprise"
        ],
        MIN_Z_HISTORY
    )
)


# ================================================================
# 11. CAP EXTREME COVID INFLUENCE
# ================================================================

for col in [

    "NFP_Z",
    "AHE_Z",
    "U3_Z"

]:

    data[col] = (
        data[col]
        .clip(
            -3,
            3
        )
    )


# ================================================================
# 12. BASIC MACRO SCORE
# ================================================================

data[
    "Macro_Composite"
] = (

      data["NFP_Z"]

    + data["AHE_Z"]

    + data["U3_Z"]

) / 3


# ================================================================
# 13. DISCRETE MACRO ALIGNMENT
# ================================================================

data[
    "NFP_Sign"
] = np.sign(
    data[
        "NFP_Surprise"
    ]
)


data[
    "AHE_Sign"
] = np.sign(
    data[
        "AHE_Surprise"
    ]
)


data[
    "U3_Sign"
] = np.sign(
    data[
        "U3_USD_Surprise"
    ]
)


data[
    "Alignment_Score"
] = (

      data["NFP_Sign"]

    + data["AHE_Sign"]

    + data["U3_Sign"]

)


# ================================================================
# 14. REGIME
# ================================================================

data[
    "Regime"
] = np.select(

    [

        data[
            "Alignment_Score"
        ] == 3,

        data[
            "Alignment_Score"
        ] == -3,

        data[
            "Alignment_Score"
        ].abs() <= 1

    ],

    [

        "FULL_USD_BULL",

        "FULL_USD_BEAR",

        "CONFLICTED"

    ],

    default=
        "PARTIAL"

)


# ================================================================
# 15. PRICE BEHAVIOUR TARGETS
# ================================================================

data[
    "Initial_Direction"
] = np.sign(
    data[
        "Event_Close_Pips"
    ]
)


data[
    "T15_Direction"
] = np.sign(
    data[
        "T15_Pips"
    ]
)


data[
    "EURUSD_Up_T15"
] = (
    data[
        "T15_Pips"
    ] > 0
).astype(int)


# ================================================================
# 16. REVERSAL
# ================================================================

data[
    "Reversal_By_T15"
] = (

    (

        data[
            "Initial_Direction"
        ]

        !=

        data[
            "T15_Direction"
        ]

    )

    &

    (
        data[
            "Initial_Direction"
        ] != 0
    )

    &

    (
        data[
            "T15_Direction"
        ] != 0
    )

).astype(int)


# ================================================================
# 17. CONTINUATION
# ================================================================

data[
    "Continuation_By_T15"
] = (

    (

        data[
            "Initial_Direction"
        ]

        ==

        data[
            "T15_Direction"
        ]

    )

    &

    (

        data[
            "T15_Pips"
        ].abs()

        >

        data[
            "Event_Close_Pips"
        ].abs()

    )

).astype(int)


# ================================================================
# 18. MOVE SIZE TARGETS
# ================================================================

for threshold in [

    10,
    20,
    30,
    40

]:

    data[
        f"AbsMove_{threshold}"
    ] = (

        data[
            "T15_Pips"
        ].abs()

        >=

        threshold

    ).astype(int)


# ================================================================
# 19. MFE / MAE FROM MACRO-EXPECTED DIRECTION
# ================================================================
#
# Positive macro = stronger USD
# Stronger USD = expected EUR/USD DOWN.
#
# ================================================================

data[
    "Expected_EURUSD_Direction"
] = (

    -np.sign(
        data[
            "Macro_Composite"
        ]
    )

)


def macro_excursion(row):

    direction = row[
        "Expected_EURUSD_Direction"
    ]


    if direction > 0:

        # LONG EURUSD

        mfe = row[
            "Max_Up_15m"
        ]

        mae = (
            -row[
                "Max_Down_15m"
            ]
        )


    elif direction < 0:

        # SHORT EURUSD

        mfe = (
            -row[
                "Max_Down_15m"
            ]
        )

        mae = row[
            "Max_Up_15m"
        ]


    else:

        mfe = np.nan
        mae = np.nan


    return pd.Series(
        {
            "Macro_MFE":
                mfe,

            "Macro_MAE":
                mae
        }
    )


data[
    [
        "Macro_MFE",
        "Macro_MAE"
    ]
] = data.apply(
    macro_excursion,
    axis=1
)


# ================================================================
# 20. OVERALL NFP BEHAVIOUR
# ================================================================

print("\n" + "=" * 80)
print("OVERALL NFP PRICE BEHAVIOUR")
print("=" * 80)


print(
    f"Median event candle range: "
    f"{data['Event_Range_Pips'].median():.2f} pips"
)


print(
    f"Mean event candle range:   "
    f"{data['Event_Range_Pips'].mean():.2f} pips"
)


print(
    f"Median |T1|:               "
    f"{data['T1_Pips'].abs().median():.2f} pips"
)


print(
    f"Median |T5|:               "
    f"{data['T5_Pips'].abs().median():.2f} pips"
)


print(
    f"Median |T10|:              "
    f"{data['T10_Pips'].abs().median():.2f} pips"
)


print(
    f"Median |T15|:              "
    f"{data['T15_Pips'].abs().median():.2f} pips"
)


print(
    f"Mean |T15|:                "
    f"{data['T15_Pips'].abs().mean():.2f} pips"
)


print(
    f"Median full 15m range:     "
    f"{data['Total_Range_15m'].median():.2f} pips"
)


print(
    f"Reversal rate:             "
    f"{data['Reversal_By_T15'].mean():.1%}"
)


print(
    f"Continuation rate:         "
    f"{data['Continuation_By_T15'].mean():.1%}"
)


for threshold in [

    10,
    20,
    30,
    40

]:

    print(

        f"|T15| >= "
        f"{threshold:>2} pips:        "

        f"{data[f'AbsMove_{threshold}'].mean():.1%}"

    )


# ================================================================
# 21. ALIGNMENT ANALYSIS
# ================================================================

alignment_stats = (

    data

    .groupby(
        "Alignment_Score"
    )

    .agg(

        N=(
            "T15_Pips",
            "count"
        ),

        Mean_Event_Close=(
            "Event_Close_Pips",
            "mean"
        ),

        Median_Event_Range=(
            "Event_Range_Pips",
            "median"
        ),

        Mean_T1=(
            "T1_Pips",
            "mean"
        ),

        Mean_T5=(
            "T5_Pips",
            "mean"
        ),

        Mean_T10=(
            "T10_Pips",
            "mean"
        ),

        Mean_T15=(
            "T15_Pips",
            "mean"
        ),

        Median_T15=(
            "T15_Pips",
            "median"
        ),

        MeanAbs_T15=(

            "T15_Pips",

            lambda x:
                x.abs().mean()

        ),

        Median_Total_Range=(
            "Total_Range_15m",
            "median"
        ),

        ReversalPct=(
            "Reversal_By_T15",
            "mean"
        ),

        ContinuationPct=(
            "Continuation_By_T15",
            "mean"
        ),

        Move20Pct=(
            "AbsMove_20",
            "mean"
        ),

        Move30Pct=(
            "AbsMove_30",
            "mean"
        )

    )

    .reset_index()

)


for col in [

    "ReversalPct",
    "ContinuationPct",
    "Move20Pct",
    "Move30Pct"

]:

    alignment_stats[col] *= 100


print("\n" + "=" * 80)
print("ALIGNMENT SCORE ANALYSIS")
print("=" * 80)


print(

    alignment_stats

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 22. REGIME ANALYSIS
# ================================================================

regime_stats = (

    data

    .groupby(
        "Regime"
    )

    .agg(

        N=(
            "T15_Pips",
            "count"
        ),

        Median_Event_Range=(
            "Event_Range_Pips",
            "median"
        ),

        Mean_Event_Close=(
            "Event_Close_Pips",
            "mean"
        ),

        Mean_T15=(
            "T15_Pips",
            "mean"
        ),

        Median_T15=(
            "T15_Pips",
            "median"
        ),

        MeanAbs_T15=(

            "T15_Pips",

            lambda x:
                x.abs().mean()

        ),

        Median_Total_Range=(
            "Total_Range_15m",
            "median"
        ),

        ReversalPct=(
            "Reversal_By_T15",
            "mean"
        ),

        ContinuationPct=(
            "Continuation_By_T15",
            "mean"
        ),

        Move20Pct=(
            "AbsMove_20",
            "mean"
        ),

        Move30Pct=(
            "AbsMove_30",
            "mean"
        )

    )

    .reset_index()

)


for col in [

    "ReversalPct",
    "ContinuationPct",
    "Move20Pct",
    "Move30Pct"

]:

    regime_stats[col] *= 100


print("\n" + "=" * 80)
print("MACRO REGIME ANALYSIS")
print("=" * 80)


print(

    regime_stats

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 23. INTERACTION FEATURES
# ================================================================

data[
    "NFP_x_AHE"
] = (
    data["NFP_Z"]
    *
    data["AHE_Z"]
)


data[
    "NFP_x_U3"
] = (
    data["NFP_Z"]
    *
    data["U3_Z"]
)


data[
    "AHE_x_U3"
] = (
    data["AHE_Z"]
    *
    data["U3_Z"]
)


data[
    "Abs_NFP"
] = data[
    "NFP_Z"
].abs()


data[
    "Abs_AHE"
] = data[
    "AHE_Z"
].abs()


data[
    "Abs_U3"
] = data[
    "U3_Z"
].abs()


# ================================================================
# 24. NUMBER OF MACRO DISAGREEMENTS
# ================================================================

data[
    "Conflict_Count"
] = (

      (
          data["NFP_Sign"]
          !=
          data["AHE_Sign"]
      ).astype(int)

    + (
          data["NFP_Sign"]
          !=
          data["U3_Sign"]
      ).astype(int)

    + (
          data["AHE_Sign"]
          !=
          data["U3_Sign"]
      ).astype(int)

)


# ================================================================
# 25. FEATURE SETS
# ================================================================

BASE_FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z"

]


INTERACTION_FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z",

    "NFP_x_AHE",
    "NFP_x_U3",
    "AHE_x_U3",

    "Abs_NFP",
    "Abs_AHE",
    "Abs_U3",

    "Conflict_Count"

]


# ================================================================
# 26. WALK-FORWARD CLASSIFICATION
# ================================================================

def walk_forward_classifier(

    df,
    features,
    target,
    min_train=60

):

    predictions = []


    for i in range(
        min_train,
        len(df)
    ):


        train = (

            df.iloc[:i]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        test = (

            df.iloc[[i]]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        if len(test) == 0:

            continue


        if len(train) < min_train:

            continue


        if train[
            target
        ].nunique() < 2:

            continue


        model = Pipeline([

            (

                "scale",

                StandardScaler()

            ),

            (

                "logit",

                LogisticRegression(

                    C=0.25,

                    penalty="l2",

                    solver="liblinear",

                    class_weight="balanced",

                    max_iter=5000

                )

            )

        ])


        model.fit(

            train[
                features
            ],

            train[
                target
            ]

        )


        probability = (

            model

            .predict_proba(

                test[
                    features
                ]

            )[0, 1]

        )


        predictions.append({

            "date":

                test[
                    "date"
                ].iloc[0],

            "Actual":

                int(
                    test[
                        target
                    ].iloc[0]
                ),

            "Probability":

                probability

        })


    return pd.DataFrame(
        predictions
    )


# ================================================================
# 27. DIRECTION MODELS
# ================================================================

direction_basic = (
    walk_forward_classifier(

        data,

        BASE_FEATURES,

        "EURUSD_Up_T15",

        MIN_TRAIN

    )
)


direction_interaction = (
    walk_forward_classifier(

        data,

        INTERACTION_FEATURES,

        "EURUSD_Up_T15",

        MIN_TRAIN

    )
)


# ================================================================
# 28. MOVE MAGNITUDE MODELS
# ================================================================

move20_model = (
    walk_forward_classifier(

        data,

        INTERACTION_FEATURES,

        "AbsMove_20",

        MIN_TRAIN

    )
)


move30_model = (
    walk_forward_classifier(

        data,

        INTERACTION_FEATURES,

        "AbsMove_30",

        MIN_TRAIN

    )
)


# ================================================================
# 29. REVERSAL MODEL
# ================================================================

REVERSAL_FEATURES = (

    INTERACTION_FEATURES

    + [

        "Event_Close_Pips",
        "Event_Range_Pips"

    ]

)


reversal_model = (
    walk_forward_classifier(

        data,

        REVERSAL_FEATURES,

        "Reversal_By_T15",

        MIN_TRAIN

    )
)


# ================================================================
# 30. MODEL EVALUATION
# ================================================================

def evaluate_classifier(
    predictions,
    name
):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)


    if predictions.empty:

        print(
            "No predictions."
        )

        return


    y = predictions[
        "Actual"
    ]


    probability = predictions[
        "Probability"
    ]


    prediction = (
        probability
        >=
        0.5
    ).astype(int)


    print(
        "OOS observations:",
        len(predictions)
    )


    print(

        f"Accuracy:           "
        f"{accuracy_score(y, prediction):.1%}"

    )


    print(

        f"Balanced accuracy:  "
        f"{balanced_accuracy_score(y, prediction):.1%}"

    )


    if y.nunique() > 1:

        print(

            f"ROC AUC:            "
            f"{roc_auc_score(y, probability):.3f}"

        )


evaluate_classifier(

    direction_basic,

    "DIRECTION — BASIC 3 FACTORS"

)


evaluate_classifier(

    direction_interaction,

    "DIRECTION — INTERACTIONS"

)


evaluate_classifier(

    move20_model,

    "ABS(T15) >= 20 PIPS"

)


evaluate_classifier(

    move30_model,

    "ABS(T15) >= 30 PIPS"

)


evaluate_classifier(

    reversal_model,

    "EVENT MOVE REVERSES BY T15"

)


# ================================================================
# 31. WALK-FORWARD T15 REGRESSION
# ================================================================

def walk_forward_ridge(

    df,
    features,
    target,
    min_train=60

):

    predictions = []


    for i in range(
        min_train,
        len(df)
    ):


        train = (

            df.iloc[:i]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        test = (

            df.iloc[[i]]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        if len(test) == 0:

            continue


        if len(train) < min_train:

            continue


        model = Pipeline([

            (

                "scale",

                StandardScaler()

            ),

            (

                "ridge",

                Ridge(
                    alpha=10
                )

            )

        ])


        model.fit(

            train[
                features
            ],

            train[
                target
            ]

        )


        prediction = (

            model

            .predict(

                test[
                    features
                ]

            )[0]

        )


        predictions.append({

            "date":

                test[
                    "date"
                ].iloc[0],

            "Actual":

                float(
                    test[
                        target
                    ].iloc[0]
                ),

            "Prediction":

                float(
                    prediction
                )

        })


    return pd.DataFrame(
        predictions
    )


ridge = walk_forward_ridge(

    data,

    INTERACTION_FEATURES,

    "T15_Pips",

    MIN_TRAIN

)


print("\n" + "=" * 80)
print("T15 PIP DISPLACEMENT REGRESSION")
print("=" * 80)


if not ridge.empty:


    mae = mean_absolute_error(

        ridge[
            "Actual"
        ],

        ridge[
            "Prediction"
        ]

    )


    rmse = np.sqrt(

        mean_squared_error(

            ridge[
                "Actual"
            ],

            ridge[
                "Prediction"
            ]

        )

    )


    directional_accuracy = (

        np.sign(
            ridge[
                "Actual"
            ]
        )

        ==

        np.sign(
            ridge[
                "Prediction"
            ]
        )

    ).mean()


    print(

        "OOS observations:",
        len(ridge)

    )


    print(

        f"MAE:                  "
        f"{mae:.2f} pips"

    )


    print(

        f"RMSE:                 "
        f"{rmse:.2f} pips"

    )


    print(

        f"Directional accuracy: "
        f"{directional_accuracy:.1%}"

    )


# ================================================================
# 32. DIRECTION CONFIDENCE / TRADE STUDY
# ================================================================

direction_trade = pd.merge(

    direction_interaction,

    data[
        [
            "date",
            "T15_Pips"
        ]
    ],

    on="date",

    how="left"

)


def threshold_study(
    df
):

    rows = []


    for confidence in CONFIDENCE_LEVELS:


        longs = df[

            df[
                "Probability"
            ]

            >=

            confidence

        ].copy()


        shorts = df[

            df[
                "Probability"
            ]

            <=

            (
                1
                -
                confidence
            )

        ].copy()


        longs[
            "Strategy_Pips"
        ] = longs[
            "T15_Pips"
        ]


        shorts[
            "Strategy_Pips"
        ] = (
            -shorts[
                "T15_Pips"
            ]
        )


        trades = pd.concat(

            [
                longs,
                shorts
            ],

            ignore_index=True

        )


        if trades.empty:

            continue


        rows.append({

            "Confidence":

                confidence,

            "Trades":

                len(trades),

            "Frequency":

                len(trades)
                /
                len(df),

            "Win_Rate":

                (
                    trades[
                        "Strategy_Pips"
                    ] > 0
                ).mean(),

            "Mean_Pips":

                trades[
                    "Strategy_Pips"
                ].mean(),

            "Median_Pips":

                trades[
                    "Strategy_Pips"
                ].median(),

            "Total_Pips":

                trades[
                    "Strategy_Pips"
                ].sum()

        })


    return pd.DataFrame(
        rows
    )


threshold_results = (
    threshold_study(
        direction_trade
    )
)


print("\n" + "=" * 80)
print("DIRECTION INTERACTION MODEL — CONFIDENCE STUDY")
print("=" * 80)


print(

    threshold_results

    .round(3)

    .to_string(
        index=False
    )

)


# ================================================================
# 33. MACRO SURPRISE MAGNITUDE
# ================================================================

data[
    "Macro_Strength"
] = np.sqrt(

      data[
          "NFP_Z"
      ] ** 2

    + data[
          "AHE_Z"
      ] ** 2

    + data[
          "U3_Z"
      ] ** 2

)


strength = (

    data

    .dropna(
        subset=[
            "Macro_Strength"
        ]
    )

    .copy()

)


strength[
    "Strength_Quintile"
] = pd.qcut(

    strength[
        "Macro_Strength"
    ],

    q=5,

    duplicates="drop"

)


strength_stats = (

    strength

    .groupby(
        "Strength_Quintile",
        observed=True
    )

    .agg(

        N=(
            "T15_Pips",
            "count"
        ),

        Mean_Strength=(
            "Macro_Strength",
            "mean"
        ),

        Median_Event_Range=(
            "Event_Range_Pips",
            "median"
        ),

        Median_Abs_T15=(

            "T15_Pips",

            lambda x:
                x.abs().median()

        ),

        Mean_Abs_T15=(

            "T15_Pips",

            lambda x:
                x.abs().mean()

        ),

        Move20Pct=(
            "AbsMove_20",
            "mean"
        ),

        Move30Pct=(
            "AbsMove_30",
            "mean"
        ),

        ReversalPct=(
            "Reversal_By_T15",
            "mean"
        )

    )

    .reset_index()

)


for col in [

    "Move20Pct",
    "Move30Pct",
    "ReversalPct"

]:

    strength_stats[
        col
    ] *= 100


print("\n" + "=" * 80)
print("MACRO SURPRISE MAGNITUDE VS MOVE SIZE")
print("=" * 80)


print(

    strength_stats

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 34. MEAN T0-T15 PATH BY MACRO REGIME
# ================================================================

close_columns = [

    f"C{i}_Pips"

    for i in range(16)

]


for regime_name in [

    "FULL_USD_BULL",
    "FULL_USD_BEAR",
    "CONFLICTED"

]:


    subset = data[

        data[
            "Regime"
        ]

        ==

        regime_name

    ]


    if len(subset) == 0:

        continue


    mean_path = (

        subset[
            close_columns
        ]

        .mean()

        .values

    )


    plt.figure(
        figsize=(11, 6)
    )


    plt.plot(

        range(16),

        mean_path,

        marker="o"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes from 08:30"
    )


    plt.ylabel(
        "EUR/USD pips from 08:29 close"
    )


    plt.title(

        f"{regime_name} — Mean NFP Path "
        f"[n={len(subset)}]"

    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


# ================================================================
# 35. SAVE EVERYTHING
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "nfp_event_master_v3.csv",

    index=False

)


alignment_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "alignment_stats.csv",

    index=False

)


regime_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "regime_stats.csv",

    index=False

)


strength_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "strength_stats.csv",

    index=False

)


direction_basic.to_csv(

    f"{OUTPUT_DIR}/"
    "direction_basic_oos.csv",

    index=False

)


direction_interaction.to_csv(

    f"{OUTPUT_DIR}/"
    "direction_interaction_oos.csv",

    index=False

)


move20_model.to_csv(

    f"{OUTPUT_DIR}/"
    "move20_oos.csv",

    index=False

)


move30_model.to_csv(

    f"{OUTPUT_DIR}/"
    "move30_oos.csv",

    index=False

)


reversal_model.to_csv(

    f"{OUTPUT_DIR}/"
    "reversal_oos.csv",

    index=False

)


ridge.to_csv(

    f"{OUTPUT_DIR}/"
    "t15_ridge_oos.csv",

    index=False

)


threshold_results.to_csv(

    f"{OUTPUT_DIR}/"
    "direction_thresholds.csv",

    index=False

)


# ================================================================
# 36. FINISH
# ================================================================

print("\n" + "=" * 80)
print("FINISHED")
print("=" * 80)


print(

    "Final event sample:",
    len(data)

)


print(

    "Saved to:",
    OUTPUT_DIR

)


print(
"""

SEND ME THESE OUTPUT SECTIONS:

1. OVERALL NFP PRICE BEHAVIOUR

2. ALIGNMENT SCORE ANALYSIS

3. MACRO REGIME ANALYSIS

4. DIRECTION — BASIC 3 FACTORS

5. DIRECTION — INTERACTIONS

6. ABS(T15) >= 20 PIPS

7. ABS(T15) >= 30 PIPS

8. EVENT MOVE REVERSES BY T15

9. T15 PIP DISPLACEMENT REGRESSION

10. DIRECTION INTERACTION MODEL — CONFIDENCE STUDY

11. MACRO SURPRISE MAGNITUDE VS MOVE SIZE

"""
)

In [ ]:
# ================================================================
# MASTER EUR/USD NFP THREE-FACTOR STUDY
# ================================================================
#
# PURPOSE
# -------
# Full study of:
#
#   NFP surprise
#   + AHE MoM surprise
#   + U3 unemployment surprise
#
# versus EUR/USD from the 08:29 pre-release close through T240.
#
#
# CONFIRMED EVENT CLOCK IN HISTDATA:
#
#   08:30 = NFP release
#
#
# BASELINE:
#
#   08:29 close = 0 pips
#
#
# ANALYSIS INCLUDED
# -----------------
#
# 1. Overall NFP price behaviour
# 2. Alignment score -3 ... +3
# 3. Full USD bullish / bearish regimes
# 4. T1 -> T240 return curves
# 5. Optimal descriptive exit horizon
# 6. Optimal exit separately by direction
# 7. Incremental return after T15
# 8. MFE / MAE by horizon
# 9. Reversal and continuation rates
# 10. Macro surprise strength vs persistence
# 11. Individual historical price overlays
# 12. Mean / median overlay paths
# 13. Walk-forward direction models at multiple horizons
# 14. Chronological train/test horizon-selection test
# 15. Rolling expanding walk-forward horizon test
#
#
# IMPORTANT:
#
# Any "best historical exit minute" is descriptive/in-sample.
#
# The chronological tests later in the script are more important
# when evaluating whether that exit horizon is actually robust.
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_master_T240_study"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30

MAX_MINUTE = 240

MIN_Z_HISTORY = 24

MIN_MODEL_TRAIN = 60


REPORT_HORIZONS = [
    1,
    5,
    10,
    15,
    20,
    30,
    45,
    60,
    90,
    120,
    180,
    240
]


MODEL_HORIZONS = [
    5,
    10,
    15,
    30,
    45,
    60,
    90,
    120,
    180,
    240
]


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


# ================================================================
# 3. KEEP MACRO COLUMNS ONLY
# ================================================================

macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"

]


missing = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values(
        "date"
    )

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


print("=" * 90)
print("MACRO DATA")
print("=" * 90)

print(
    "Releases:",
    len(macro)
)

print(
    "Range:",
    macro["date"].min(),
    "->",
    macro["date"].max()
)


# ================================================================
# 4. LOAD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values(
        "Datetime"
    )

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


price_index = (

    prices

    .set_index(
        "Datetime"
    )

    .sort_index()

)


print("\n" + "=" * 90)
print("PRICE DATA")
print("=" * 90)

print(
    "Rows:",
    len(prices)
)

print(
    "Range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 5. BUILD TRUE MACRO SURPRISE SIGNS
# ================================================================
#
# Positive U3_USD_Surprise means USD positive because we already
# inverted unemployment when creating the macro dataset.
#
# ================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)

macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)

macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment_Score"] = (

      macro["NFP_Sign"]

    + macro["AHE_Sign"]

    + macro["U3_Sign"]

)


macro["Regime"] = np.select(

    [

        macro[
            "Alignment_Score"
        ] == 3,

        macro[
            "Alignment_Score"
        ] == -3,

        macro[
            "Alignment_Score"
        ].abs() <= 1

    ],

    [

        "FULL_USD_BULL",

        "FULL_USD_BEAR",

        "CONFLICTED"

    ],

    default="PARTIAL"

)


# ================================================================
# 6. HISTORICAL Z-SCORES
# ================================================================
#
# NO FUTURE LEAKAGE.
#
# Each observation uses only previous releases to estimate
# the mean and standard deviation.
#
# ================================================================

def historical_zscore(
    series,
    minimum_history=24
):

    s = pd.to_numeric(
        series,
        errors="coerce"
    )


    historical_mean = (

        s

        .expanding(
            min_periods=minimum_history
        )

        .mean()

        .shift(1)

    )


    historical_std = (

        s

        .expanding(
            min_periods=minimum_history
        )

        .std()

        .shift(1)

    )


    z = (
        s
        -
        historical_mean
    ) / historical_std


    return z.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


macro["NFP_Z"] = historical_zscore(
    macro["NFP_Surprise"],
    MIN_Z_HISTORY
)

macro["AHE_Z"] = historical_zscore(
    macro["AHE_Surprise"],
    MIN_Z_HISTORY
)

macro["U3_Z"] = historical_zscore(
    macro["U3_USD_Surprise"],
    MIN_Z_HISTORY
)


# cap extreme historical outliers

for col in [
    "NFP_Z",
    "AHE_Z",
    "U3_Z"
]:

    macro[col] = (
        macro[col]
        .clip(-3, 3)
    )


macro["Macro_Composite"] = (

      macro["NFP_Z"]

    + macro["AHE_Z"]

    + macro["U3_Z"]

) / 3


macro["Macro_Strength"] = np.sqrt(

      macro["NFP_Z"] ** 2

    + macro["AHE_Z"] ** 2

    + macro["U3_Z"] ** 2

)


# ================================================================
# 7. INTERACTION FEATURES
# ================================================================

macro["NFP_x_AHE"] = (
    macro["NFP_Z"]
    *
    macro["AHE_Z"]
)

macro["NFP_x_U3"] = (
    macro["NFP_Z"]
    *
    macro["U3_Z"]
)

macro["AHE_x_U3"] = (
    macro["AHE_Z"]
    *
    macro["U3_Z"]
)


macro["Abs_NFP"] = (
    macro["NFP_Z"].abs()
)

macro["Abs_AHE"] = (
    macro["AHE_Z"].abs()
)

macro["Abs_U3"] = (
    macro["U3_Z"].abs()
)


macro["Conflict_Count"] = (

      (
          macro["NFP_Sign"]
          !=
          macro["AHE_Sign"]
      ).astype(int)

    + (
          macro["NFP_Sign"]
          !=
          macro["U3_Sign"]
      ).astype(int)

    + (
          macro["AHE_Sign"]
          !=
          macro["U3_Sign"]
      ).astype(int)

)


# ================================================================
# 8. EXTRACT FULL NFP PATH
# ================================================================
#
# 08:29 close is the pre-release baseline.
#
# T0 = 08:30 close
#
# T240 = 12:30 close
#
# ================================================================

def extract_event(
    release_date
):

    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    pre_dt = (
        event_dt
        -
        pd.Timedelta(minutes=1)
    )


    if pre_dt not in price_index.index:

        return None


    pre_close = float(

        price_index.loc[
            pre_dt,
            "Close"
        ]

    )


    record = {

        "date":
            date,

        "Pre_Close":
            pre_close

    }


    running_high = -np.inf
    running_low = np.inf


    for minute in range(
        MAX_MINUTE + 1
    ):


        timestamp = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if timestamp not in price_index.index:

            record[
                f"T{minute}"
            ] = np.nan

            record[
                f"RUN_HIGH_T{minute}"
            ] = np.nan

            record[
                f"RUN_LOW_T{minute}"
            ] = np.nan

            continue


        row = price_index.loc[
            timestamp
        ]


        close = float(
            row["Close"]
        )

        high = float(
            row["High"]
        )

        low = float(
            row["Low"]
        )


        close_pips = (

            close
            -
            pre_close

        ) * 10000


        high_pips = (

            high
            -
            pre_close

        ) * 10000


        low_pips = (

            low
            -
            pre_close

        ) * 10000


        running_high = max(
            running_high,
            high_pips
        )

        running_low = min(
            running_low,
            low_pips
        )


        record[
            f"T{minute}"
        ] = close_pips


        record[
            f"RUN_HIGH_T{minute}"
        ] = running_high


        record[
            f"RUN_LOW_T{minute}"
        ] = running_low


    # event candle specifically

    if event_dt in price_index.index:

        event_row = price_index.loc[
            event_dt
        ]

        record[
            "Event_Range_Pips"
        ] = (

            float(
                event_row["High"]
            )

            -

            float(
                event_row["Low"]
            )

        ) * 10000


        record[
            "Event_Close_Pips"
        ] = (

            float(
                event_row["Close"]
            )

            -

            pre_close

        ) * 10000


    else:

        record[
            "Event_Range_Pips"
        ] = np.nan

        record[
            "Event_Close_Pips"
        ] = np.nan


    return record


# ================================================================
# 9. EXTRACT ALL EVENTS
# ================================================================

event_rows = []


for release_date in macro["date"]:

    event = extract_event(
        release_date
    )

    if event is not None:

        event_rows.append(
            event
        )


events = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    events,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values(
        "date"
    )

    .reset_index(
        drop=True
    )

)


print("\n" + "=" * 90)
print("EVENT EXTRACTION")
print("=" * 90)

print(
    "Matched releases:",
    len(data)
)

print(
    "Range:",
    data["date"].min(),
    "->",
    data["date"].max()
)


# ================================================================
# 10. PRICE BEHAVIOUR VARIABLES
# ================================================================

data["Initial_Direction"] = np.sign(
    data["T0"]
)


for horizon in REPORT_HORIZONS:

    data[
        f"Direction_T{horizon}"
    ] = np.sign(
        data[
            f"T{horizon}"
        ]
    )


    data[
        f"Reversal_T{horizon}"
    ] = (

        (
            data["Initial_Direction"]
            !=
            data[
                f"Direction_T{horizon}"
            ]
        )

        &

        (
            data["Initial_Direction"]
            != 0
        )

        &

        (
            data[
                f"Direction_T{horizon}"
            ]
            != 0
        )

    ).astype(int)


# ================================================================
# 11. NORMALIZE ALL ALIGNED TRADES
# ================================================================
#
# FULL USD BULL:
#
# expected SHORT EURUSD
# Trade_Direction = -1
#
#
# FULL USD BEAR:
#
# expected LONG EURUSD
# Trade_Direction = +1
#
# ================================================================

data["Trade_Direction"] = np.select(

    [

        data["Regime"]
        ==
        "FULL_USD_BEAR",

        data["Regime"]
        ==
        "FULL_USD_BULL"

    ],

    [
        1,
        -1
    ],

    default=np.nan

)


for minute in range(
    MAX_MINUTE + 1
):

    data[
        f"Strategy_T{minute}"
    ] = (

        data[
            f"T{minute}"
        ]

        *

        data[
            "Trade_Direction"
        ]

    )


aligned = data[
    data[
        "Trade_Direction"
    ].notna()
].copy()


print("\nFULL USD BULL events:",

      (
          aligned["Regime"]
          ==
          "FULL_USD_BULL"
      ).sum()
)


print(
    "FULL USD BEAR events:",

    (
        aligned["Regime"]
        ==
        "FULL_USD_BEAR"
    ).sum()
)


print(
    "Total fully aligned:",
    len(aligned)
)


# ================================================================
# 12. OVERALL NFP BEHAVIOUR BY HORIZON
# ================================================================

overall_rows = []


for horizon in REPORT_HORIZONS:

    movement = data[
        f"T{horizon}"
    ].dropna()


    overall_rows.append({

        "Horizon":
            horizon,

        "N":
            len(movement),

        "Median_Abs_Move":
            movement.abs().median(),

        "Mean_Abs_Move":
            movement.abs().mean(),

        "Median_Move":
            movement.median(),

        "Mean_Move":
            movement.mean(),

        "Reversal_Rate":
            data.loc[
                movement.index,
                f"Reversal_T{horizon}"
            ].mean()

    })


overall_horizon = pd.DataFrame(
    overall_rows
)


print("\n" + "=" * 90)
print("ALL NFP RELEASES — PRICE BEHAVIOUR BY HORIZON")
print("=" * 90)


print(

    overall_horizon

    .assign(
        Reversal_Rate=lambda x:
            x["Reversal_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 13. ALIGNMENT SCORE BY HORIZON
# ================================================================

alignment_rows = []


for score in sorted(
    data[
        "Alignment_Score"
    ].dropna().unique()
):


    subset = data[
        data[
            "Alignment_Score"
        ]
        ==
        score
    ]


    for horizon in REPORT_HORIZONS:

        movement = subset[
            f"T{horizon}"
        ].dropna()


        if movement.empty:
            continue


        alignment_rows.append({

            "Alignment":
                score,

            "Horizon":
                horizon,

            "N":
                len(movement),

            "Mean_Move":
                movement.mean(),

            "Median_Move":
                movement.median(),

            "Mean_Abs_Move":
                movement.abs().mean(),

            "Pct_EURUSD_Up":
                (
                    movement > 0
                ).mean()

        })


alignment_horizon = pd.DataFrame(
    alignment_rows
)


print("\n" + "=" * 90)
print("ALIGNMENT SCORE — SELECTED HORIZONS")
print("=" * 90)


print(

    alignment_horizon[

        alignment_horizon[
            "Horizon"
        ].isin(
            [
                15,
                30,
                60,
                120,
                240
            ]
        )

    ]

    .assign(
        Pct_EURUSD_Up=lambda x:
            x["Pct_EURUSD_Up"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 14. FULL ALIGNMENT RETURN EVERY MINUTE
# ================================================================

minute_stats = []


for minute in range(
    1,
    MAX_MINUTE + 1
):

    returns = aligned[
        f"Strategy_T{minute}"
    ].dropna()


    if returns.empty:
        continue


    minute_stats.append({

        "Minute":
            minute,

        "N":
            len(returns),

        "Mean_Return":
            returns.mean(),

        "Median_Return":
            returns.median(),

        "Win_Rate":
            (
                returns > 0
            ).mean(),

        "Std_Return":
            returns.std()

    })


minute_stats = pd.DataFrame(
    minute_stats
)


# ================================================================
# 15. DESCRIPTIVE BEST EXIT
# ================================================================

best_mean = minute_stats.loc[

    minute_stats[
        "Mean_Return"
    ].idxmax()

]


best_median = minute_stats.loc[

    minute_stats[
        "Median_Return"
    ].idxmax()

]


best_win = minute_stats.loc[

    minute_stats[
        "Win_Rate"
    ].idxmax()

]


BEST_FULL_SAMPLE_MINUTE = int(
    best_mean[
        "Minute"
    ]
)


print("\n" + "=" * 90)
print("DESCRIPTIVE BEST FULL-ALIGNMENT EXIT")
print("=" * 90)


print(
    f"Best mean-return minute: "
    f"T{BEST_FULL_SAMPLE_MINUTE}"
)


print(
    f"Mean return:             "
    f"{best_mean['Mean_Return']:.2f} pips"
)


print(
    f"Median return:           "
    f"{best_mean['Median_Return']:.2f} pips"
)


print(
    f"Win rate:                "
    f"{best_mean['Win_Rate']:.1%}"
)


print()


print(
    f"Best MEDIAN minute:      "
    f"T{int(best_median['Minute'])}"
)


print(
    f"Best raw WIN-RATE minute:"
    f" T{int(best_win['Minute'])}"
)


# ================================================================
# 16. FULL ALIGNMENT KEY HORIZONS
# ================================================================

aligned_horizon_rows = []


for horizon in REPORT_HORIZONS:

    returns = aligned[
        f"Strategy_T{horizon}"
    ].dropna()


    if returns.empty:
        continue


    aligned_horizon_rows.append({

        "Horizon":
            horizon,

        "N":
            len(returns),

        "Mean_Return":
            returns.mean(),

        "Median_Return":
            returns.median(),

        "Win_Rate":
            (
                returns > 0
            ).mean(),

        "Mean_Winner":
            returns[
                returns > 0
            ].mean(),

        "Mean_Loser":
            returns[
                returns < 0
            ].mean()

    })


aligned_horizon = pd.DataFrame(
    aligned_horizon_rows
)


print("\n" + "=" * 90)
print("FULL ALIGNMENT — RETURN BY HORIZON")
print("=" * 90)


print(

    aligned_horizon

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 17. USD BULL VS USD BEAR SEPARATELY
# ================================================================

direction_rows = []


for regime_name in [
    "FULL_USD_BULL",
    "FULL_USD_BEAR"
]:


    subset = aligned[
        aligned[
            "Regime"
        ]
        ==
        regime_name
    ]


    for horizon in REPORT_HORIZONS:


        returns = subset[
            f"Strategy_T{horizon}"
        ].dropna()


        if returns.empty:
            continue


        direction_rows.append({

            "Regime":
                regime_name,

            "Horizon":
                horizon,

            "N":
                len(returns),

            "Mean_Return":
                returns.mean(),

            "Median_Return":
                returns.median(),

            "Win_Rate":
                (
                    returns > 0
                ).mean()

        })


direction_horizon = pd.DataFrame(
    direction_rows
)


print("\n" + "=" * 90)
print("FULL ALIGNMENT — RESULTS BY DIRECTION")
print("=" * 90)


print(

    direction_horizon

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 18. BEST EXIT SEPARATELY BY DIRECTION
# ================================================================

print("\n" + "=" * 90)
print("DESCRIPTIVE BEST EXIT BY DIRECTION")
print("=" * 90)


best_direction_minutes = {}


for regime_name in [
    "FULL_USD_BULL",
    "FULL_USD_BEAR"
]:


    subset = aligned[
        aligned[
            "Regime"
        ]
        ==
        regime_name
    ]


    rows = []


    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        r = subset[
            f"Strategy_T{minute}"
        ].dropna()


        if r.empty:
            continue


        rows.append({

            "Minute":
                minute,

            "Mean_Return":
                r.mean(),

            "Median_Return":
                r.median(),

            "Win_Rate":
                (
                    r > 0
                ).mean()

        })


    temp = pd.DataFrame(
        rows
    )


    best = temp.loc[

        temp[
            "Mean_Return"
        ].idxmax()

    ]


    best_direction_minutes[
        regime_name
    ] = int(
        best[
            "Minute"
        ]
    )


    print()


    print(
        regime_name
    )


    print(
        f"Best minute:   "
        f"T{int(best['Minute'])}"
    )


    print(
        f"Mean return:   "
        f"{best['Mean_Return']:.2f} pips"
    )


    print(
        f"Median return: "
        f"{best['Median_Return']:.2f} pips"
    )


    print(
        f"Win rate:      "
        f"{best['Win_Rate']:.1%}"
    )


# ================================================================
# 19. INCREMENTAL RETURN AFTER T15
# ================================================================

increment_rows = []


t15 = aligned[
    "Strategy_T15"
]


for horizon in [
    30,
    45,
    60,
    90,
    120,
    180,
    240
]:


    later = aligned[
        f"Strategy_T{horizon}"
    ]


    mask = (
        t15.notna()
        &
        later.notna()
    )


    extra = (

        later[
            mask
        ]

        -

        t15[
            mask
        ]

    )


    increment_rows.append({

        "T15_to_Horizon":
            horizon,

        "N":
            len(extra),

        "Mean_Extra_Pips":
            extra.mean(),

        "Median_Extra_Pips":
            extra.median(),

        "Pct_Improved":
            (
                extra > 0
            ).mean()

    })


incremental = pd.DataFrame(
    increment_rows
)


print("\n" + "=" * 90)
print("INCREMENTAL RETURN AFTER T15")
print("=" * 90)


print(

    incremental

    .assign(
        Pct_Improved=lambda x:
            x["Pct_Improved"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 20. MFE AND MAE BY HORIZON
# ================================================================

excursion_rows = []


for horizon in REPORT_HORIZONS:


    mfe_values = []
    mae_values = []


    for _, row in aligned.iterrows():


        direction = row[
            "Trade_Direction"
        ]


        high = row[
            f"RUN_HIGH_T{horizon}"
        ]

        low = row[
            f"RUN_LOW_T{horizon}"
        ]


        if pd.isna(high) or pd.isna(low):

            continue


        if direction > 0:

            # long EURUSD

            mfe = high

            mae = (
                -low
            )


        else:

            # short EURUSD

            mfe = (
                -low
            )

            mae = high


        mfe_values.append(
            mfe
        )

        mae_values.append(
            mae
        )


    excursion_rows.append({

        "Horizon":
            horizon,

        "N":
            len(mfe_values),

        "Mean_MFE":
            np.mean(
                mfe_values
            ),

        "Median_MFE":
            np.median(
                mfe_values
            ),

        "Mean_MAE":
            np.mean(
                mae_values
            ),

        "Median_MAE":
            np.median(
                mae_values
            )

    })


excursion = pd.DataFrame(
    excursion_rows
)


print("\n" + "=" * 90)
print("FULL ALIGNMENT — MFE / MAE BY HORIZON")
print("=" * 90)


print(

    excursion

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 21. REVERSAL / CONTINUATION STUDY
# ================================================================

reversal_rows = []


for horizon in REPORT_HORIZONS:


    valid = data[
        [
            "Initial_Direction",
            f"Direction_T{horizon}",
            f"T{horizon}"
        ]
    ].dropna()


    reversal = (

        (
            valid[
                "Initial_Direction"
            ]

            !=

            valid[
                f"Direction_T{horizon}"
            ]
        )

        &

        (
            valid[
                "Initial_Direction"
            ]
            != 0
        )

        &

        (
            valid[
                f"Direction_T{horizon}"
            ]
            != 0
        )

    )


    continuation = (

        (
            valid[
                "Initial_Direction"
            ]

            ==

            valid[
                f"Direction_T{horizon}"
            ]
        )

        &

        (
            valid[
                f"T{horizon}"
            ].abs()

            >

            data.loc[
                valid.index,
                "T0"
            ].abs()
        )

    )


    reversal_rows.append({

        "Horizon":
            horizon,

        "N":
            len(valid),

        "Reversal_Rate":
            reversal.mean(),

        "Continuation_Rate":
            continuation.mean()

    })


reversal_table = pd.DataFrame(
    reversal_rows
)


print("\n" + "=" * 90)
print("ALL RELEASES — REVERSAL / CONTINUATION")
print("=" * 90)


print(

    reversal_table

    .assign(

        Reversal_Rate=lambda x:
            x["Reversal_Rate"]
            *
            100,

        Continuation_Rate=lambda x:
            x["Continuation_Rate"]
            *
            100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 22. MACRO STRENGTH VS PERSISTENCE
# ================================================================

strength_data = data.dropna(
    subset=[
        "Macro_Strength"
    ]
).copy()


strength_data[
    "Strength_Quintile"
] = pd.qcut(

    strength_data[
        "Macro_Strength"
    ],

    q=5,

    duplicates="drop"

)


strength_rows = []


for bucket, subset in strength_data.groupby(
    "Strength_Quintile",
    observed=True
):


    for horizon in [
        15,
        30,
        60,
        120,
        240
    ]:


        move = subset[
            f"T{horizon}"
        ].dropna()


        if move.empty:
            continue


        strength_rows.append({

            "Strength_Quintile":
                str(bucket),

            "Horizon":
                horizon,

            "N":
                len(move),

            "Mean_Strength":
                subset.loc[
                    move.index,
                    "Macro_Strength"
                ].mean(),

            "Median_Abs_Move":
                move.abs().median(),

            "Mean_Abs_Move":
                move.abs().mean()

        })


strength_persistence = pd.DataFrame(
    strength_rows
)


print("\n" + "=" * 90)
print("MACRO SURPRISE STRENGTH VS MOVE PERSISTENCE")
print("=" * 90)


print(

    strength_persistence

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 23. WALK-FORWARD DIRECTION MODEL
# ================================================================

BASE_FEATURES = [
    "NFP_Z",
    "AHE_Z",
    "U3_Z"
]


INTERACTION_FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z",

    "NFP_x_AHE",
    "NFP_x_U3",
    "AHE_x_U3",

    "Abs_NFP",
    "Abs_AHE",
    "Abs_U3",

    "Conflict_Count"

]


def walk_forward_classifier(
    df,
    features,
    target,
    min_train=60
):


    predictions = []


    for i in range(
        min_train,
        len(df)
    ):


        train = (

            df.iloc[:i]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        test = (

            df.iloc[[i]]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        if len(test) == 0:
            continue


        if len(train) < min_train:
            continue


        if train[
            target
        ].nunique() < 2:

            continue


        model = Pipeline([

            (
                "scale",
                StandardScaler()
            ),

            (
                "logit",

                LogisticRegression(

                    C=0.25,

                    penalty="l2",

                    solver="liblinear",

                    class_weight="balanced",

                    max_iter=5000

                )
            )

        ])


        model.fit(

            train[
                features
            ],

            train[
                target
            ]

        )


        probability = (

            model

            .predict_proba(

                test[
                    features
                ]

            )[0, 1]

        )


        predictions.append({

            "date":
                test[
                    "date"
                ].iloc[0],

            "Actual":
                int(
                    test[
                        target
                    ].iloc[0]
                ),

            "Probability_Up":
                probability

        })


    return pd.DataFrame(
        predictions
    )


# ================================================================
# 24. MODEL DIRECTION AT MULTIPLE HORIZONS
# ================================================================

model_results = []


for horizon in MODEL_HORIZONS:


    target = (
        f"EURUSD_Up_T{horizon}"
    )


    data[
        target
    ] = (

        data[
            f"T{horizon}"
        ]

        >

        0

    ).astype(int)


    predictions = walk_forward_classifier(

        data,

        BASE_FEATURES,

        target,

        MIN_MODEL_TRAIN

    )


    if predictions.empty:
        continue


    y = predictions[
        "Actual"
    ]


    p = predictions[
        "Probability_Up"
    ]


    prediction = (
        p >= 0.5
    ).astype(int)


    result = {

        "Horizon":
            horizon,

        "N":
            len(predictions),

        "Accuracy":
            accuracy_score(
                y,
                prediction
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y,
                prediction
            )

    }


    if y.nunique() > 1:

        result[
            "ROC_AUC"
        ] = roc_auc_score(
            y,
            p
        )

    else:

        result[
            "ROC_AUC"
        ] = np.nan


    model_results.append(
        result
    )


model_horizon_results = pd.DataFrame(
    model_results
)


print("\n" + "=" * 90)
print("WALK-FORWARD 3-FACTOR DIRECTION MODEL BY HORIZON")
print("=" * 90)


print(

    model_horizon_results

    .assign(

        Accuracy=lambda x:
            x["Accuracy"]
            *
            100,

        Balanced_Accuracy=lambda x:
            x[
                "Balanced_Accuracy"
            ]
            *
            100

    )

    .round(3)

    .to_string(
        index=False
    )

)


# ================================================================
# 25. CHRONOLOGICAL 70/30 EXIT TEST
# ================================================================
#
# Find the best exit using first 70% ONLY.
#
# Evaluate that selected minute on later 30%.
#
# ================================================================

aligned = (

    aligned

    .sort_values(
        "date"
    )

    .reset_index(
        drop=True
    )

)


split = int(
    len(aligned)
    *
    0.70
)


train = aligned.iloc[
    :split
].copy()


test = aligned.iloc[
    split:
].copy()


train_results = []


for minute in range(
    1,
    MAX_MINUTE + 1
):


    returns = train[
        f"Strategy_T{minute}"
    ].dropna()


    if returns.empty:
        continue


    train_results.append({

        "Minute":
            minute,

        "Mean_Return":
            returns.mean(),

        "Median_Return":
            returns.median(),

        "Win_Rate":
            (
                returns > 0
            ).mean()

    })


train_results = pd.DataFrame(
    train_results
)


TRAIN_SELECTED_MINUTE = int(

    train_results.loc[

        train_results[
            "Mean_Return"
        ].idxmax(),

        "Minute"

    ]

)


test_returns = test[
    f"Strategy_T{TRAIN_SELECTED_MINUTE}"
].dropna()


print("\n" + "=" * 90)
print("CHRONOLOGICAL 70/30 HORIZON TEST")
print("=" * 90)


print(
    "Training events:",
    len(train)
)

print(
    "Testing events:",
    len(test)
)


print(
    f"\nExit chosen using first 70% ONLY:"
    f" T{TRAIN_SELECTED_MINUTE}"
)


print(
    f"Later 30% mean return:   "
    f"{test_returns.mean():.2f} pips"
)


print(
    f"Later 30% median return: "
    f"{test_returns.median():.2f} pips"
)


print(
    f"Later 30% win rate:      "
    f"{(test_returns > 0).mean():.1%}"
)


# ================================================================
# 26. EXPANDING WALK-FORWARD EXIT SELECTION
# ================================================================
#
# Stronger robustness test:
#
# For every aligned event after enough history exists:
#
# 1. Look only at PRIOR aligned releases.
# 2. Find historical best mean-return minute.
# 3. Use THAT exit minute on the next event.
#
# ================================================================

MIN_ALIGNED_HISTORY = 10


wf_exit_rows = []


for i in range(
    MIN_ALIGNED_HISTORY,
    len(aligned)
):


    historical = aligned.iloc[
        :i
    ]


    current = aligned.iloc[
        i
    ]


    candidate_rows = []


    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        historical_returns = historical[
            f"Strategy_T{minute}"
        ].dropna()


        if len(
            historical_returns
        ) < MIN_ALIGNED_HISTORY:

            continue


        candidate_rows.append({

            "Minute":
                minute,

            "Mean_Return":
                historical_returns.mean()

        })


    candidate_df = pd.DataFrame(
        candidate_rows
    )


    if candidate_df.empty:
        continue


    selected_minute = int(

        candidate_df.loc[

            candidate_df[
                "Mean_Return"
            ].idxmax(),

            "Minute"

        ]

    )


    realized = current[
        f"Strategy_T{selected_minute}"
    ]


    if pd.isna(realized):
        continue


    wf_exit_rows.append({

        "date":
            current["date"],

        "Regime":
            current["Regime"],

        "Selected_Minute":
            selected_minute,

        "Realized_Return":
            realized

    })


walk_forward_exit = pd.DataFrame(
    wf_exit_rows
)


print("\n" + "=" * 90)
print("EXPANDING WALK-FORWARD EXIT-SELECTION TEST")
print("=" * 90)


if not walk_forward_exit.empty:


    print(
        "OOS aligned trades:",
        len(
            walk_forward_exit
        )
    )


    print(
        f"Mean return:   "
        f"{walk_forward_exit['Realized_Return'].mean():.2f} pips"
    )


    print(
        f"Median return: "
        f"{walk_forward_exit['Realized_Return'].median():.2f} pips"
    )


    print(
        f"Win rate:      "
        f"{(walk_forward_exit['Realized_Return'] > 0).mean():.1%}"
    )


    print(
        f"Median selected exit: "
        f"T{walk_forward_exit['Selected_Minute'].median():.0f}"
    )


    print(
        "\nMost frequently selected exits:"
    )


    print(

        walk_forward_exit[
            "Selected_Minute"
        ]

        .value_counts()

        .head(15)

        .to_string()

    )


# ================================================================
# 27. CHOOSE OVERLAY GRAPH HORIZON
# ================================================================
#
# Show enough data to include:
#
# - descriptive full-sample peak
# - chronological training-selected peak
# - at least 60 minutes
# - plus 30 minutes afterward to reveal decay
#
# ================================================================

OVERLAY_MINUTE = min(

    MAX_MINUTE,

    max(

        60,

        BEST_FULL_SAMPLE_MINUTE
        +
        30,

        TRAIN_SELECTED_MINUTE
        +
        30

    )

)


print("\n" + "=" * 90)
print("OVERLAY GRAPH TIMEFRAME")
print("=" * 90)


print(
    f"Full-sample peak: "
    f"T{BEST_FULL_SAMPLE_MINUTE}"
)


print(
    f"Training-only selected peak: "
    f"T{TRAIN_SELECTED_MINUTE}"
)


print(
    f"Overlay range: "
    f"T0 -> T{OVERLAY_MINUTE}"
)


# ================================================================
# 28. FULL USD BULL OVERLAY
# ================================================================

bull = aligned[

    aligned[
        "Regime"
    ]
    ==
    "FULL_USD_BULL"

].copy()


minutes = np.arange(
    0,
    OVERLAY_MINUTE + 1
)


plt.figure(
    figsize=(16, 8)
)


for _, row in bull.iterrows():


    path = np.array([

        row[
            f"T{minute}"
        ]

        for minute in minutes

    ])


    plt.plot(

        minutes,

        path,

        alpha=0.20,

        linewidth=1

    )


bull_mean = np.array([

    bull[
        f"T{minute}"
    ].mean()

    for minute in minutes

])


bull_median = np.array([

    bull[
        f"T{minute}"
    ].median()

    for minute in minutes

])


plt.plot(

    minutes,

    bull_mean,

    linewidth=4,

    label="Mean"

)


plt.plot(

    minutes,

    bull_median,

    linewidth=3,

    linestyle="--",

    label="Median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":",
    label="T15"
)


plt.axvline(

    best_direction_minutes[
        "FULL_USD_BULL"
    ],

    linestyle="--",

    label=(
        "Best historical mean "
        f"T{best_direction_minutes['FULL_USD_BULL']}"
    )

)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "EUR/USD displacement from 08:29 close (pips)"
)


plt.title(

    "FULL USD BULL (+3)\n"
    "Every historical EUR/USD path"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 29. FULL USD BEAR OVERLAY
# ================================================================

bear = aligned[

    aligned[
        "Regime"
    ]
    ==
    "FULL_USD_BEAR"

].copy()


plt.figure(
    figsize=(16, 8)
)


for _, row in bear.iterrows():


    path = np.array([

        row[
            f"T{minute}"
        ]

        for minute in minutes

    ])


    plt.plot(

        minutes,

        path,

        alpha=0.20,

        linewidth=1

    )


bear_mean = np.array([

    bear[
        f"T{minute}"
    ].mean()

    for minute in minutes

])


bear_median = np.array([

    bear[
        f"T{minute}"
    ].median()

    for minute in minutes

])


plt.plot(

    minutes,

    bear_mean,

    linewidth=4,

    label="Mean"

)


plt.plot(

    minutes,

    bear_median,

    linewidth=3,

    linestyle="--",

    label="Median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":",
    label="T15"
)


plt.axvline(

    best_direction_minutes[
        "FULL_USD_BEAR"
    ],

    linestyle="--",

    label=(
        "Best historical mean "
        f"T{best_direction_minutes['FULL_USD_BEAR']}"
    )

)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "EUR/USD displacement from 08:29 close (pips)"
)


plt.title(

    "FULL USD BEAR (-3)\n"
    "Every historical EUR/USD path"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 30. NORMALIZED FULL ALIGNMENT OVERLAY
# ================================================================
#
# All trades are flipped so:
#
# POSITIVE = macro-expected direction was profitable.
#
# This makes bull and bear events directly comparable.
#
# ================================================================

plt.figure(
    figsize=(16, 8)
)


for _, row in aligned.iterrows():


    path = np.array([

        row[
            f"Strategy_T{minute}"
        ]

        for minute in minutes

    ])


    plt.plot(

        minutes,

        path,

        alpha=0.16,

        linewidth=1

    )


combined_mean = np.array([

    aligned[
        f"Strategy_T{minute}"
    ].mean()

    for minute in minutes

])


combined_median = np.array([

    aligned[
        f"Strategy_T{minute}"
    ].median()

    for minute in minutes

])


plt.plot(

    minutes,

    combined_mean,

    linewidth=4,

    label="Mean macro-direction return"

)


plt.plot(

    minutes,

    combined_median,

    linewidth=3,

    linestyle="--",

    label="Median macro-direction return"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":",
    label="T15"
)


plt.axvline(

    BEST_FULL_SAMPLE_MINUTE,

    linestyle="--",

    label=(
        f"Full-sample peak T"
        f"{BEST_FULL_SAMPLE_MINUTE}"
    )

)


plt.axvline(

    TRAIN_SELECTED_MINUTE,

    linestyle="-.",

    label=(
        f"Training-selected T"
        f"{TRAIN_SELECTED_MINUTE}"
    )

)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "Return in expected macro direction (pips)"
)


plt.title(

    "FULL THREE-FACTOR ALIGNMENT\n"
    "Normalized historical trade paths"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 31. MEAN RETURN CURVE THROUGH T240
# ================================================================

plt.figure(
    figsize=(16, 7)
)


plt.plot(

    minute_stats[
        "Minute"
    ],

    minute_stats[
        "Mean_Return"
    ],

    linewidth=3

)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":",
    label="T15"
)


plt.axvline(

    BEST_FULL_SAMPLE_MINUTE,

    linestyle="--",

    label=(
        f"Peak mean T"
        f"{BEST_FULL_SAMPLE_MINUTE}"
    )

)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "Average full-alignment return (pips)"
)


plt.title(
    "Full-Alignment Average Return Through T240"
)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 32. MEDIAN RETURN CURVE
# ================================================================

plt.figure(
    figsize=(16, 7)
)


plt.plot(

    minute_stats[
        "Minute"
    ],

    minute_stats[
        "Median_Return"
    ],

    linewidth=3

)


plt.axhline(
    0,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":"
)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "Median full-alignment return (pips)"
)


plt.title(
    "Full-Alignment Median Return Through T240"
)


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 33. WIN-RATE CURVE
# ================================================================

plt.figure(
    figsize=(16, 7)
)


plt.plot(

    minute_stats[
        "Minute"
    ],

    minute_stats[
        "Win_Rate"
    ]
    *
    100,

    linewidth=3

)


plt.axhline(
    50,
    linestyle="--"
)


plt.axvline(
    15,
    linestyle=":"
)


plt.xlabel(
    "Minutes after NFP"
)


plt.ylabel(
    "Directional win rate (%)"
)


plt.title(
    "Full-Alignment Win Rate Through T240"
)


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 34. SAVE ALL OUTPUTS
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "all_events_T240.csv",

    index=False

)


aligned.to_csv(

    f"{OUTPUT_DIR}/"
    "full_alignment_events_T240.csv",

    index=False

)


overall_horizon.to_csv(

    f"{OUTPUT_DIR}/"
    "overall_horizon_stats.csv",

    index=False

)


alignment_horizon.to_csv(

    f"{OUTPUT_DIR}/"
    "alignment_by_horizon.csv",

    index=False

)


minute_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "full_alignment_minute_curve.csv",

    index=False

)


aligned_horizon.to_csv(

    f"{OUTPUT_DIR}/"
    "full_alignment_key_horizons.csv",

    index=False

)


direction_horizon.to_csv(

    f"{OUTPUT_DIR}/"
    "full_alignment_by_direction.csv",

    index=False

)


incremental.to_csv(

    f"{OUTPUT_DIR}/"
    "incremental_after_T15.csv",

    index=False

)


excursion.to_csv(

    f"{OUTPUT_DIR}/"
    "mfe_mae_by_horizon.csv",

    index=False

)


reversal_table.to_csv(

    f"{OUTPUT_DIR}/"
    "reversal_continuation.csv",

    index=False

)


strength_persistence.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_strength_persistence.csv",

    index=False

)


model_horizon_results.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_direction_by_horizon.csv",

    index=False

)


train_results.to_csv(

    f"{OUTPUT_DIR}/"
    "training_horizon_selection.csv",

    index=False

)


walk_forward_exit.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_exit_selection.csv",

    index=False

)


# ================================================================
# 35. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("MASTER STUDY FINISHED")
print("=" * 90)


print(
    "All NFP events:",
    len(data)
)


print(
    "Fully aligned events:",
    len(aligned)
)


print(
    f"\nDescriptive full-sample best exit:"
    f" T{BEST_FULL_SAMPLE_MINUTE}"
)


print(
    f"Training-only selected exit:"
    f" T{TRAIN_SELECTED_MINUTE}"
)


print(
    f"Full USD bull descriptive best:"
    f" T{best_direction_minutes['FULL_USD_BULL']}"
)


print(
    f"Full USD bear descriptive best:"
    f" T{best_direction_minutes['FULL_USD_BEAR']}"
)


print(
    "\nResults saved to:"
)

print(
    OUTPUT_DIR
)


print(
"""

PLEASE SEND ME THESE OUTPUTS:

1. ALL NFP RELEASES — PRICE BEHAVIOUR BY HORIZON

2. ALIGNMENT SCORE — SELECTED HORIZONS

3. DESCRIPTIVE BEST FULL-ALIGNMENT EXIT

4. FULL ALIGNMENT — RETURN BY HORIZON

5. FULL ALIGNMENT — RESULTS BY DIRECTION

6. DESCRIPTIVE BEST EXIT BY DIRECTION

7. INCREMENTAL RETURN AFTER T15

8. FULL ALIGNMENT — MFE / MAE BY HORIZON

9. ALL RELEASES — REVERSAL / CONTINUATION

10. MACRO SURPRISE STRENGTH VS MOVE PERSISTENCE

11. WALK-FORWARD 3-FACTOR DIRECTION MODEL BY HORIZON

12. CHRONOLOGICAL 70/30 HORIZON TEST

13. EXPANDING WALK-FORWARD EXIT-SELECTION TEST

14. OVERLAY GRAPH TIMEFRAME


AND LOOK AT THE FOUR MOST IMPORTANT GRAPHS:

- FULL USD BULL (+3) overlay
- FULL USD BEAR (-3) overlay
- normalized full-alignment overlay
- average full-alignment return curve

"""
)

In [ ]:
# ================================================================
# EXECUTABLE EUR/USD NFP FULL-ALIGNMENT BACKTEST
# ================================================================
#
# SIGNAL
# ------
# FULL_USD_BULL (+3):
#     NFP surprise > 0
#     AHE surprise > 0
#     unemployment USD-surprise > 0
#     => SHORT EUR/USD
#
# FULL_USD_BEAR (-3):
#     all three < 0
#     => LONG EUR/USD
#
#
# IMPORTANT:
# ---------
# We DO NOT enter at the 08:29 pre-release price.
#
# We test executable approximations:
#
#     T0 close = 08:30 close
#     T1 close = 08:31 close
#     T2 close = 08:32 close
#     T5 close = 08:35 close
#
#
# FIXED EXITS:
#
#     T15
#     T30
#     T45
#     T60
#     T75
#     T90
#
#
# COST SENSITIVITY:
#
#     0, 1, 2, 3, 5 pips round trip
#
#
# ROBUSTNESS:
#
# 1. Full-sample matrix
# 2. 70/30 chronological test
# 3. Expanding walk-forward parameter selection
# 4. Bull / bear direction separately
# 5. Stop / target study
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_executable_backtest"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------
# NFP event timestamp confirmed from your data
# ------------------------------------------------

EVENT_HOUR = 8
EVENT_MINUTE = 30


# ------------------------------------------------
# Entry at CLOSE of these minutes after release
# ------------------------------------------------

ENTRY_MINUTES = [
    0,
    1,
    2,
    5
]


# ------------------------------------------------
# Exit at CLOSE of these minutes after release
# ------------------------------------------------

EXIT_MINUTES = [
    15,
    30,
    45,
    60,
    75,
    90
]


# ------------------------------------------------
# Round-trip cost sensitivity.
#
# This cost includes whatever combination you want
# to interpret as:
#
# spread + slippage + commissions
#
# ------------------------------------------------

COST_LEVELS = [
    0,
    1,
    2,
    3,
    5
]


# Main cost assumption used for optimization tests

BASE_COST_PIPS = 3.0


# ------------------------------------------------
# Stops / targets
# ------------------------------------------------

STOP_LEVELS = [
    10,
    15,
    20,
    30,
    40
]


TARGET_LEVELS = [
    20,
    30,
    40,
    50,
    60
]


MAX_STOP_TEST_HORIZON = 90


# Need at least this many historic fully aligned events
# before walk-forward parameter optimization

MIN_HISTORY = 10


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"

]


missing = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values("date")

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(drop=True)

)


# ================================================================
# 3. CREATE FULL ALIGNMENT SIGNAL
# ================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)

macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)

macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment"] = (

      macro["NFP_Sign"]
    + macro["AHE_Sign"]
    + macro["U3_Sign"]

)


macro["Regime"] = np.select(

    [

        macro["Alignment"] == 3,

        macro["Alignment"] == -3

    ],

    [

        "FULL_USD_BULL",

        "FULL_USD_BEAR"

    ],

    default="OTHER"

)


macro["Trade_Direction"] = np.select(

    [

        macro["Alignment"] == -3,

        macro["Alignment"] == 3

    ],

    [

        1,      # LONG EUR/USD

        -1      # SHORT EUR/USD

    ],

    default=np.nan

)


aligned_macro = macro[
    macro["Trade_Direction"].notna()
].copy()


print("=" * 90)
print("SIGNAL DATA")
print("=" * 90)

print(
    "Fully aligned releases:",
    len(aligned_macro)
)

print(
    aligned_macro[
        "Regime"
    ].value_counts()
)


# ================================================================
# 4. LOAD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(drop=True)

)


px = (
    prices
    .set_index("Datetime")
    .sort_index()
)


print("\nPRICE ROWS:", len(prices))

print(
    "PRICE RANGE:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 5. EXTRACT T0 -> T90 OHLC PATH
# ================================================================

def extract_event(
    release_date
):

    release_date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        release_date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    record = {
        "date":
            release_date
    }


    for minute in range(
        MAX_STOP_TEST_HORIZON + 1
    ):

        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            record[
                f"O{minute}"
            ] = np.nan

            record[
                f"H{minute}"
            ] = np.nan

            record[
                f"L{minute}"
            ] = np.nan

            record[
                f"C{minute}"
            ] = np.nan

            continue


        row = px.loc[
            dt
        ]


        record[
            f"O{minute}"
        ] = float(
            row["Open"]
        )


        record[
            f"H{minute}"
        ] = float(
            row["High"]
        )


        record[
            f"L{minute}"
        ] = float(
            row["Low"]
        )


        record[
            f"C{minute}"
        ] = float(
            row["Close"]
        )


    return record


event_rows = []


for d in aligned_macro[
    "date"
]:

    result = extract_event(
        d
    )

    if result is not None:

        event_rows.append(
            result
        )


event_prices = pd.DataFrame(
    event_rows
)


data = pd.merge(

    aligned_macro,

    event_prices,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values("date")

    .reset_index(drop=True)

)


print("\n" + "=" * 90)
print("EXECUTABLE SAMPLE")
print("=" * 90)

print(
    "Matched aligned releases:",
    len(data)
)


# ================================================================
# 6. FIXED ENTRY / EXIT PNL
# ================================================================

def calculate_fixed_pnl(
    row,
    entry_minute,
    exit_minute,
    cost_pips=0
):

    entry = row[
        f"C{entry_minute}"
    ]

    exit_price = row[
        f"C{exit_minute}"
    ]


    if (
        pd.isna(entry)
        or
        pd.isna(exit_price)
    ):

        return np.nan


    direction = row[
        "Trade_Direction"
    ]


    gross = (

        (
            exit_price
            -
            entry
        )

        *

        10000

        *

        direction

    )


    net = (
        gross
        -
        cost_pips
    )


    return net


# ================================================================
# 7. FULL ENTRY / EXIT MATRIX
# ================================================================

matrix_rows = []


for cost in COST_LEVELS:


    for entry in ENTRY_MINUTES:


        for exit_minute in EXIT_MINUTES:


            if exit_minute <= entry:

                continue


            pnl = data.apply(

                lambda row:
                    calculate_fixed_pnl(

                        row,

                        entry,

                        exit_minute,

                        cost

                    ),

                axis=1

            ).dropna()


            if pnl.empty:

                continue


            matrix_rows.append({

                "Cost_Pips":
                    cost,

                "Entry":
                    entry,

                "Exit":
                    exit_minute,

                "Holding_Minutes":
                    exit_minute
                    -
                    entry,

                "Trades":
                    len(pnl),

                "Win_Rate":
                    (
                        pnl > 0
                    ).mean(),

                "Mean_Net_Pips":
                    pnl.mean(),

                "Median_Net_Pips":
                    pnl.median(),

                "Total_Net_Pips":
                    pnl.sum(),

                "Mean_Winner":
                    pnl[
                        pnl > 0
                    ].mean(),

                "Mean_Loser":
                    pnl[
                        pnl < 0
                    ].mean()

            })


matrix = pd.DataFrame(
    matrix_rows
)


# ================================================================
# 8. RESULTS USING MAIN 3-PIP COST
# ================================================================

base_matrix = (

    matrix[
        matrix[
            "Cost_Pips"
        ]
        ==
        BASE_COST_PIPS
    ]

    .sort_values(
        "Mean_Net_Pips",
        ascending=False
    )

    .reset_index(drop=True)

)


print("\n" + "=" * 90)
print(
    f"ENTRY / EXIT MATRIX — "
    f"{BASE_COST_PIPS:.1f} PIP ROUND-TRIP COST"
)
print("=" * 90)


print(

    base_matrix[
        [
            "Entry",
            "Exit",
            "Holding_Minutes",
            "Trades",
            "Win_Rate",
            "Mean_Net_Pips",
            "Median_Net_Pips",
            "Total_Net_Pips"
        ]
    ]

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 9. BEST FULL-SAMPLE COMBINATION
# ================================================================

best_full = base_matrix.iloc[0]


BEST_ENTRY = int(
    best_full[
        "Entry"
    ]
)


BEST_EXIT = int(
    best_full[
        "Exit"
    ]
)


print("\n" + "=" * 90)
print("DESCRIPTIVE BEST EXECUTABLE SETUP")
print("=" * 90)


print(
    f"Entry:          "
    f"T{BEST_ENTRY} close"
)


print(
    f"Exit:           "
    f"T{BEST_EXIT} close"
)


print(
    f"Holding period: "
    f"{BEST_EXIT - BEST_ENTRY} minutes"
)


print(
    f"Mean net:       "
    f"{best_full['Mean_Net_Pips']:.2f} pips"
)


print(
    f"Median net:     "
    f"{best_full['Median_Net_Pips']:.2f} pips"
)


print(
    f"Win rate:       "
    f"{best_full['Win_Rate']:.1%}"
)


# ================================================================
# 10. COST SENSITIVITY
# ================================================================

cost_rows = []


for cost in COST_LEVELS:


    pnl = data.apply(

        lambda row:
            calculate_fixed_pnl(

                row,

                BEST_ENTRY,

                BEST_EXIT,

                cost

            ),

        axis=1

    ).dropna()


    cost_rows.append({

        "Cost_Pips":
            cost,

        "Trades":
            len(pnl),

        "Mean_Net_Pips":
            pnl.mean(),

        "Median_Net_Pips":
            pnl.median(),

        "Win_Rate":
            (
                pnl > 0
            ).mean(),

        "Total_Net_Pips":
            pnl.sum()

    })


cost_sensitivity = pd.DataFrame(
    cost_rows
)


print("\n" + "=" * 90)
print("COST SENSITIVITY — BEST FULL-SAMPLE SETUP")
print("=" * 90)


print(

    cost_sensitivity

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 11. DIRECTION-SPECIFIC ENTRY / EXIT STUDY
# ================================================================

direction_rows = []


for regime in [
    "FULL_USD_BULL",
    "FULL_USD_BEAR"
]:


    subset = data[
        data["Regime"]
        ==
        regime
    ]


    for entry in ENTRY_MINUTES:


        for exit_minute in EXIT_MINUTES:


            if exit_minute <= entry:

                continue


            pnl = subset.apply(

                lambda row:
                    calculate_fixed_pnl(

                        row,

                        entry,

                        exit_minute,

                        BASE_COST_PIPS

                    ),

                axis=1

            ).dropna()


            if pnl.empty:

                continue


            direction_rows.append({

                "Regime":
                    regime,

                "Entry":
                    entry,

                "Exit":
                    exit_minute,

                "Trades":
                    len(pnl),

                "Win_Rate":
                    (
                        pnl > 0
                    ).mean(),

                "Mean_Net_Pips":
                    pnl.mean(),

                "Median_Net_Pips":
                    pnl.median()

            })


direction_matrix = pd.DataFrame(
    direction_rows
)


print("\n" + "=" * 90)
print("BEST SETUP BY MACRO DIRECTION")
print("=" * 90)


for regime in [
    "FULL_USD_BULL",
    "FULL_USD_BEAR"
]:


    temp = (

        direction_matrix[
            direction_matrix[
                "Regime"
            ]
            ==
            regime
        ]

        .sort_values(
            "Mean_Net_Pips",
            ascending=False
        )

    )


    best = temp.iloc[0]


    print("\n", regime)


    print(
        f"Entry:      "
        f"T{int(best['Entry'])}"
    )


    print(
        f"Exit:       "
        f"T{int(best['Exit'])}"
    )


    print(
        f"Mean net:   "
        f"{best['Mean_Net_Pips']:.2f} pips"
    )


    print(
        f"Median net: "
        f"{best['Median_Net_Pips']:.2f} pips"
    )


    print(
        f"Win rate:   "
        f"{best['Win_Rate']:.1%}"
    )


# ================================================================
# 12. CHRONOLOGICAL 70 / 30 TEST
# ================================================================

split = int(
    len(data)
    *
    0.70
)


train = (
    data
    .iloc[:split]
    .copy()
)


test = (
    data
    .iloc[split:]
    .copy()
)


train_rows = []


for entry in ENTRY_MINUTES:


    for exit_minute in EXIT_MINUTES:


        if exit_minute <= entry:

            continue


        pnl = train.apply(

            lambda row:
                calculate_fixed_pnl(

                    row,

                    entry,

                    exit_minute,

                    BASE_COST_PIPS

                ),

            axis=1

        ).dropna()


        if pnl.empty:

            continue


        train_rows.append({

            "Entry":
                entry,

            "Exit":
                exit_minute,

            "Mean_Net":
                pnl.mean(),

            "Median_Net":
                pnl.median(),

            "Win_Rate":
                (
                    pnl > 0
                ).mean()

        })


train_selection = pd.DataFrame(
    train_rows
)


selected = (

    train_selection

    .sort_values(
        "Mean_Net",
        ascending=False
    )

    .iloc[0]

)


TRAIN_ENTRY = int(
    selected[
        "Entry"
    ]
)


TRAIN_EXIT = int(
    selected[
        "Exit"
    ]
)


test_pnl = test.apply(

    lambda row:
        calculate_fixed_pnl(

            row,

            TRAIN_ENTRY,

            TRAIN_EXIT,

            BASE_COST_PIPS

        ),

    axis=1

).dropna()


print("\n" + "=" * 90)
print("CHRONOLOGICAL 70/30 EXECUTION TEST")
print("=" * 90)


print(
    "Training trades:",
    len(train)
)

print(
    "Testing trades:",
    len(test)
)


print(
    f"\nSelected using first 70% only:"
)


print(
    f"Entry T{TRAIN_ENTRY} -> "
    f"Exit T{TRAIN_EXIT}"
)


print(
    f"\nLater 30% mean net:   "
    f"{test_pnl.mean():.2f} pips"
)


print(
    f"Later 30% median net: "
    f"{test_pnl.median():.2f} pips"
)


print(
    f"Later 30% win rate:   "
    f"{(test_pnl > 0).mean():.1%}"
)


print(
    f"Later 30% total net:  "
    f"{test_pnl.sum():.2f} pips"
)


# ================================================================
# 13. EXPANDING WALK-FORWARD PARAMETER SELECTION
# ================================================================
#
# For each new aligned event:
#
# 1. use only PRIOR aligned events
# 2. find best entry / exit pair
# 3. execute that pair on current event
#
# ================================================================

wf_rows = []


for i in range(
    MIN_HISTORY,
    len(data)
):


    history = data.iloc[
        :i
    ]


    current = data.iloc[
        i
    ]


    candidates = []


    for entry in ENTRY_MINUTES:


        for exit_minute in EXIT_MINUTES:


            if exit_minute <= entry:

                continue


            historic_pnl = history.apply(

                lambda row:
                    calculate_fixed_pnl(

                        row,

                        entry,

                        exit_minute,

                        BASE_COST_PIPS

                    ),

                axis=1

            ).dropna()


            if len(
                historic_pnl
            ) < MIN_HISTORY:

                continue


            candidates.append({

                "Entry":
                    entry,

                "Exit":
                    exit_minute,

                "Mean_Net":
                    historic_pnl.mean()

            })


    candidate_df = pd.DataFrame(
        candidates
    )


    if candidate_df.empty:

        continue


    chosen = candidate_df.loc[

        candidate_df[
            "Mean_Net"
        ].idxmax()

    ]


    entry = int(
        chosen[
            "Entry"
        ]
    )


    exit_minute = int(
        chosen[
            "Exit"
        ]
    )


    realized = calculate_fixed_pnl(

        current,

        entry,

        exit_minute,

        BASE_COST_PIPS

    )


    if pd.isna(realized):

        continue


    wf_rows.append({

        "date":
            current["date"],

        "Regime":
            current["Regime"],

        "Entry":
            entry,

        "Exit":
            exit_minute,

        "Net_Pips":
            realized

    })


walk_forward = pd.DataFrame(
    wf_rows
)


print("\n" + "=" * 90)
print("EXPANDING WALK-FORWARD EXECUTION TEST")
print("=" * 90)


if not walk_forward.empty:


    print(
        "OOS trades:",
        len(walk_forward)
    )


    print(
        f"Mean net:   "
        f"{walk_forward['Net_Pips'].mean():.2f} pips"
    )


    print(
        f"Median net: "
        f"{walk_forward['Net_Pips'].median():.2f} pips"
    )


    print(
        f"Win rate:   "
        f"{(walk_forward['Net_Pips'] > 0).mean():.1%}"
    )


    print(
        f"Total net:  "
        f"{walk_forward['Net_Pips'].sum():.2f} pips"
    )


    print(
        "\nSelected entry frequencies:"
    )


    print(
        walk_forward[
            "Entry"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


    print(
        "\nSelected exit frequencies:"
    )


    print(
        walk_forward[
            "Exit"
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )


# ================================================================
# 14. STOP / TARGET SIMULATOR
# ================================================================
#
# Entry is at close of entry minute.
#
# Starting from NEXT minute, examine High and Low.
#
# If both stop and target are touched inside the same one-minute
# candle, assume STOP FIRST.
#
# That is intentionally conservative because M1 data cannot tell
# us the intrabar sequence.
#
# If neither is hit, exit at MAX_STOP_TEST_HORIZON close.
#
# ================================================================

def simulate_stop_target(
    row,
    entry_minute,
    stop_pips,
    target_pips,
    final_exit=60,
    cost_pips=3.0
):


    entry_price = row[
        f"C{entry_minute}"
    ]


    if pd.isna(
        entry_price
    ):

        return np.nan


    direction = row[
        "Trade_Direction"
    ]


    # ------------------------------------------------------------
    # Build stop and target price
    # ------------------------------------------------------------

    if direction > 0:

        # LONG

        stop_price = (
            entry_price
            -
            stop_pips / 10000
        )

        target_price = (
            entry_price
            +
            target_pips / 10000
        )


    else:

        # SHORT

        stop_price = (
            entry_price
            +
            stop_pips / 10000
        )

        target_price = (
            entry_price
            -
            target_pips / 10000
        )


    # ------------------------------------------------------------
    # Start NEXT bar
    # ------------------------------------------------------------

    for minute in range(
        entry_minute + 1,
        final_exit + 1
    ):


        high = row[
            f"H{minute}"
        ]

        low = row[
            f"L{minute}"
        ]


        if pd.isna(high) or pd.isna(low):

            continue


        if direction > 0:

            stop_hit = (
                low <= stop_price
            )

            target_hit = (
                high >= target_price
            )


        else:

            stop_hit = (
                high >= stop_price
            )

            target_hit = (
                low <= target_price
            )


        # Conservative:
        # if both touched, stop wins

        if stop_hit:

            return (
                -stop_pips
                -
                cost_pips
            )


        if target_hit:

            return (
                target_pips
                -
                cost_pips
            )


    # ------------------------------------------------------------
    # Neither triggered -> time exit
    # ------------------------------------------------------------

    exit_price = row[
        f"C{final_exit}"
    ]


    if pd.isna(
        exit_price
    ):

        return np.nan


    gross = (

        (
            exit_price
            -
            entry_price
        )

        *
        10000

        *
        direction

    )


    return (
        gross
        -
        cost_pips
    )


# ================================================================
# 15. STOP / TARGET GRID
# ================================================================

stop_rows = []


# Use entry choices that are realistically interesting

STOP_ENTRY_MINUTES = [
    0,
    1,
    2
]


STOP_FINAL_EXITS = [
    30,
    60,
    90
]


for entry in STOP_ENTRY_MINUTES:


    for final_exit in STOP_FINAL_EXITS:


        for stop in STOP_LEVELS:


            for target in TARGET_LEVELS:


                pnl = data.apply(

                    lambda row:
                        simulate_stop_target(

                            row,

                            entry,

                            stop,

                            target,

                            final_exit,

                            BASE_COST_PIPS

                        ),

                    axis=1

                ).dropna()


                if pnl.empty:

                    continue


                stop_rows.append({

                    "Entry":
                        entry,

                    "Final_Exit":
                        final_exit,

                    "Stop":
                        stop,

                    "Target":
                        target,

                    "Trades":
                        len(pnl),

                    "Win_Rate":
                        (
                            pnl > 0
                        ).mean(),

                    "Mean_Net_Pips":
                        pnl.mean(),

                    "Median_Net_Pips":
                        pnl.median(),

                    "Total_Net_Pips":
                        pnl.sum()

                })


stop_target_results = pd.DataFrame(
    stop_rows
)


stop_target_ranked = (

    stop_target_results

    .sort_values(
        "Mean_Net_Pips",
        ascending=False
    )

    .reset_index(drop=True)

)


print("\n" + "=" * 90)
print("TOP 20 STOP / TARGET CONFIGURATIONS")
print("=" * 90)


print(

    stop_target_ranked

    .head(20)

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 16. SIMPLE EQUITY CURVE FOR WALK-FORWARD TEST
# ================================================================

if not walk_forward.empty:


    walk_forward[
        "Cumulative_Pips"
    ] = (

        walk_forward[
            "Net_Pips"
        ]
        .cumsum()

    )


    plt.figure(
        figsize=(13, 6)
    )


    plt.plot(

        walk_forward[
            "date"
        ],

        walk_forward[
            "Cumulative_Pips"
        ],

        marker="o"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "NFP date"
    )


    plt.ylabel(
        "Cumulative net pips"
    )


    plt.title(

        "Expanding Walk-Forward "
        "Full-Alignment NFP Strategy"

    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


# ================================================================
# 17. ENTRY-DELAY COMPARISON
# ================================================================

delay_rows = []


# Use 60-minute exit because previous research showed
# the aligned edge clustering around 41-63 minutes.

for entry in ENTRY_MINUTES:


    pnl = data.apply(

        lambda row:
            calculate_fixed_pnl(

                row,

                entry,

                60,

                BASE_COST_PIPS

            ),

        axis=1

    ).dropna()


    delay_rows.append({

        "Entry_Minute":
            entry,

        "Holding_Minutes":
            60 - entry,

        "Trades":
            len(pnl),

        "Win_Rate":
            (
                pnl > 0
            ).mean(),

        "Mean_Net_Pips":
            pnl.mean(),

        "Median_Net_Pips":
            pnl.median()

    })


entry_delay = pd.DataFrame(
    delay_rows
)


print("\n" + "=" * 90)
print("ENTRY DELAY COMPARISON — EXIT T60")
print("=" * 90)


print(

    entry_delay

    .assign(
        Win_Rate=lambda x:
            x["Win_Rate"]
            *
            100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 18. SAVE OUTPUTS
# ================================================================

matrix.to_csv(

    f"{OUTPUT_DIR}/"
    "entry_exit_matrix.csv",

    index=False

)


cost_sensitivity.to_csv(

    f"{OUTPUT_DIR}/"
    "cost_sensitivity.csv",

    index=False

)


direction_matrix.to_csv(

    f"{OUTPUT_DIR}/"
    "direction_specific_matrix.csv",

    index=False

)


train_selection.to_csv(

    f"{OUTPUT_DIR}/"
    "training_parameter_selection.csv",

    index=False

)


walk_forward.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_execution.csv",

    index=False

)


stop_target_results.to_csv(

    f"{OUTPUT_DIR}/"
    "stop_target_results.csv",

    index=False

)


entry_delay.to_csv(

    f"{OUTPUT_DIR}/"
    "entry_delay_comparison.csv",

    index=False

)


data.to_csv(

    f"{OUTPUT_DIR}/"
    "aligned_event_execution_data.csv",

    index=False

)


# ================================================================
# 19. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("EXECUTION STUDY FINISHED")
print("=" * 90)


print(
    "Aligned events:",
    len(data)
)


print(
    f"\nFull-sample best:"
    f" T{BEST_ENTRY} -> T{BEST_EXIT}"
)


print(
    f"70/30 training-selected:"
    f" T{TRAIN_ENTRY} -> T{TRAIN_EXIT}"
)


print(
    "\nSaved to:"
)

print(
    OUTPUT_DIR
)


print(
"""

SEND ME THESE OUTPUTS:

1. ENTRY / EXIT MATRIX — 3 PIP ROUND-TRIP COST
   (top ~15 rows is enough)

2. DESCRIPTIVE BEST EXECUTABLE SETUP

3. COST SENSITIVITY

4. BEST SETUP BY MACRO DIRECTION

5. CHRONOLOGICAL 70/30 EXECUTION TEST

6. EXPANDING WALK-FORWARD EXECUTION TEST

7. TOP 20 STOP / TARGET CONFIGURATIONS

8. ENTRY DELAY COMPARISON — EXIT T60


MOST IMPORTANT:

The 70/30 and expanding walk-forward results matter much more
than the full-sample best combination.

If a setup remains profitable when:

- entry occurs AFTER the NFP release,
- 3-5 pips of execution cost is deducted,
- parameters are selected using prior data only,

then we have moved much closer to evidence of a genuinely
tradable effect rather than merely an event-study correlation.

"""
)

In [ ]:
# ================================================================
# NFP POST-EVENT CONTINUATION / REVERSAL STUDY
# ================================================================
#
# PURPOSE
# -------
# Study what happens AFTER the 08:30 NFP candle has completed.
#
# Uses ALL releases, not just fully aligned +/-3 cases.
#
#
# KEY QUESTION:
#
# Given:
#
#   NFP surprise
#   AHE surprise
#   U3 surprise
#   alignment / macro strength
#   first 08:30 candle
#
# can we predict whether EUR/USD:
#
#   - continues
#   - reverses
#   - goes nowhere
#
# after the initial event candle?
#
#
# BASELINE:
#
# PRE = 08:29 close
#
# EVENT = 08:30 candle
#
# Post-event returns are measured FROM THE 08:30 CLOSE:
#
#   Post_T1  = 08:31 close - 08:30 close
#   Post_T2  = 08:32 close - 08:30 close
#   Post_T5  = 08:35 close - 08:30 close
#   Post_T15 = 08:45 close - 08:30 close
#   Post_T30 = 09:00 close - 08:30 close
#   Post_T60 = 09:30 close - 08:30 close
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_post_event_study"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30

MAX_MINUTE = 60

MIN_Z_HISTORY = 24
MIN_TRAIN = 60


HORIZONS = [
    1,
    2,
    5,
    10,
    15,
    30,
    60
]


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)

macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"
]


missing = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values("date")

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(drop=True)

)


# ================================================================
# 3. BUILD MACRO ALIGNMENT
# ================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)

macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)

macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment_Score"] = (

      macro["NFP_Sign"]
    + macro["AHE_Sign"]
    + macro["U3_Sign"]

)


macro["Macro_USD_Direction"] = np.sign(
    macro["Alignment_Score"]
)


macro["Expected_EURUSD_Direction"] = (
    -macro["Macro_USD_Direction"]
)


# ================================================================
# 4. HISTORICAL Z-SCORES
# ================================================================

def historical_zscore(
    series,
    minimum_history=24
):

    s = pd.to_numeric(
        series,
        errors="coerce"
    )


    historical_mean = (

        s
        .expanding(
            min_periods=minimum_history
        )
        .mean()
        .shift(1)

    )


    historical_std = (

        s
        .expanding(
            min_periods=minimum_history
        )
        .std()
        .shift(1)

    )


    z = (
        s - historical_mean
    ) / historical_std


    return z.replace(
        [np.inf, -np.inf],
        np.nan
    )


macro["NFP_Z"] = historical_zscore(
    macro["NFP_Surprise"],
    MIN_Z_HISTORY
)

macro["AHE_Z"] = historical_zscore(
    macro["AHE_Surprise"],
    MIN_Z_HISTORY
)

macro["U3_Z"] = historical_zscore(
    macro["U3_USD_Surprise"],
    MIN_Z_HISTORY
)


for col in [
    "NFP_Z",
    "AHE_Z",
    "U3_Z"
]:

    macro[col] = (
        macro[col]
        .clip(-3, 3)
    )


macro["Macro_Composite"] = (

      macro["NFP_Z"]
    + macro["AHE_Z"]
    + macro["U3_Z"]

) / 3


macro["Macro_Strength"] = np.sqrt(

      macro["NFP_Z"] ** 2
    + macro["AHE_Z"] ** 2
    + macro["U3_Z"] ** 2

)


# ================================================================
# 5. INTERACTION FEATURES
# ================================================================

macro["NFP_x_AHE"] = (
    macro["NFP_Z"]
    *
    macro["AHE_Z"]
)

macro["NFP_x_U3"] = (
    macro["NFP_Z"]
    *
    macro["U3_Z"]
)

macro["AHE_x_U3"] = (
    macro["AHE_Z"]
    *
    macro["U3_Z"]
)


macro["Abs_NFP"] = (
    macro["NFP_Z"].abs()
)

macro["Abs_AHE"] = (
    macro["AHE_Z"].abs()
)

macro["Abs_U3"] = (
    macro["U3_Z"].abs()
)


# ================================================================
# 6. LOAD EUR/USD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(drop=True)

)


px = (
    prices
    .set_index("Datetime")
    .sort_index()
)


print("=" * 90)
print("INPUT DATA")
print("=" * 90)

print(
    "Macro releases:",
    len(macro)
)

print(
    "Price rows:",
    len(prices)
)

print(
    "Price range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 7. EXTRACT EVENT + POST-EVENT PATH
# ================================================================

def extract_event(
    release_date
):

    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date
        +
        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    pre_dt = (
        event_dt
        -
        pd.Timedelta(minutes=1)
    )


    if (
        pre_dt not in px.index
        or
        event_dt not in px.index
    ):

        return None


    pre_close = float(
        px.loc[
            pre_dt,
            "Close"
        ]
    )


    event_row = px.loc[
        event_dt
    ]


    event_open = float(
        event_row["Open"]
    )

    event_high = float(
        event_row["High"]
    )

    event_low = float(
        event_row["Low"]
    )

    event_close = float(
        event_row["Close"]
    )


    record = {

        "date":
            date,

        "Pre_Close":
            pre_close,

        "Event_Open":
            event_open,

        "Event_High":
            event_high,

        "Event_Low":
            event_low,

        "Event_Close":
            event_close

    }


    # ------------------------------------------------------------
    # initial release candle
    # ------------------------------------------------------------

    record[
        "Initial_Move_Pips"
    ] = (

        event_close
        -
        pre_close

    ) * 10000


    record[
        "Event_Range_Pips"
    ] = (

        event_high
        -
        event_low

    ) * 10000


    record[
        "Event_Body_Pips"
    ] = (

        event_close
        -
        event_open

    ) * 10000


    # ------------------------------------------------------------
    # close location:
    #
    # 0 = close at candle low
    # 1 = close at candle high
    # ------------------------------------------------------------

    event_range = (
        event_high
        -
        event_low
    )


    if event_range > 0:

        record[
            "Close_Location"
        ] = (

            event_close
            -
            event_low

        ) / event_range

    else:

        record[
            "Close_Location"
        ] = np.nan


    # ------------------------------------------------------------
    # candle efficiency:
    #
    # |net displacement from pre|
    # /
    # high-low range
    # ------------------------------------------------------------

    if record[
        "Event_Range_Pips"
    ] > 0:

        record[
            "Event_Efficiency"
        ] = (

            abs(
                record[
                    "Initial_Move_Pips"
                ]
            )

            /

            record[
                "Event_Range_Pips"
            ]

        )

    else:

        record[
            "Event_Efficiency"
        ] = np.nan


    # ------------------------------------------------------------
    # Path through T60
    # ------------------------------------------------------------

    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        dt = (

            event_dt
            +
            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            record[
                f"Post_T{minute}"
            ] = np.nan

            record[
                f"Total_T{minute}"
            ] = np.nan

            continue


        close = float(
            px.loc[
                dt,
                "Close"
            ]
        )


        # from event close
        record[
            f"Post_T{minute}"
        ] = (

            close
            -
            event_close

        ) * 10000


        # from pre-release close
        record[
            f"Total_T{minute}"
        ] = (

            close
            -
            pre_close

        ) * 10000


    return record


event_rows = []


for d in macro["date"]:

    event = extract_event(
        d
    )

    if event is not None:

        event_rows.append(
            event
        )


event_df = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    event_df,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values("date")

    .reset_index(drop=True)

)


print("\nMatched releases:", len(data))


# ================================================================
# 8. INITIAL PRICE DIRECTION
# ================================================================

data["Initial_Price_Direction"] = np.sign(
    data["Initial_Move_Pips"]
)


# ================================================================
# 9. MACRO / PRICE AGREEMENT
# ================================================================
#
# Expected EURUSD direction:
#
# +1 = EURUSD should rise
# -1 = EURUSD should fall
#
# ================================================================

data["Macro_Price_Agree"] = (

    data[
        "Expected_EURUSD_Direction"
    ]

    ==

    data[
        "Initial_Price_Direction"
    ]

).astype(int)


# Zero alignment should not count as true agreement

data.loc[
    data[
        "Expected_EURUSD_Direction"
    ] == 0,
    "Macro_Price_Agree"
] = np.nan


# ================================================================
# 10. CONTINUATION / REVERSAL LABELS
# ================================================================

for horizon in HORIZONS:


    post = data[
        f"Post_T{horizon}"
    ]


    # ------------------------------------------------------------
    # continuation:
    #
    # post-event movement is in SAME direction as initial candle
    # ------------------------------------------------------------

    data[
        f"Continuation_T{horizon}"
    ] = (

        np.sign(
            post
        )

        ==

        data[
            "Initial_Price_Direction"
        ]

    ).astype(int)


    # ------------------------------------------------------------
    # reversal:
    #
    # post-event movement is opposite initial candle
    # ------------------------------------------------------------

    data[
        f"Reversal_T{horizon}"
    ] = (

        np.sign(
            post
        )

        ==

        -data[
            "Initial_Price_Direction"
        ]

    ).astype(int)


    # ------------------------------------------------------------
    # macro-direction continuation:
    #
    # after event close, price continues in macro-expected direction
    # ------------------------------------------------------------

    data[
        f"Macro_Continuation_T{horizon}"
    ] = (

        np.sign(
            post
        )

        ==

        data[
            "Expected_EURUSD_Direction"
        ]

    ).astype(int)


# ================================================================
# 11. INITIAL MOVE CAPTURE STUDY
# ================================================================
#
# How much of total absolute displacement already occurred
# by the 08:30 close?
#
# Only calculate where initial and later move are same direction.
#
# ================================================================

capture_rows = []


for horizon in HORIZONS:


    total = data[
        f"Total_T{horizon}"
    ]


    initial = data[
        "Initial_Move_Pips"
    ]


    same_direction = (

        np.sign(initial)
        ==
        np.sign(total)

    )


    valid = (

        same_direction

        &

        total.notna()

        &

        (
            total.abs() > 1
        )

    )


    ratio = (

        initial[
            valid
        ].abs()

        /

        total[
            valid
        ].abs()

    )


    capture_rows.append({

        "Horizon":
            horizon,

        "N":
            len(ratio),

        "Median_Fraction_Already_Captured":
            ratio.median(),

        "Mean_Fraction_Already_Captured":
            ratio.mean(),

        "Pct_Initial_Over_50pct":
            (
                ratio >= 0.50
            ).mean(),

        "Pct_Initial_Over_75pct":
            (
                ratio >= 0.75
            ).mean(),

        "Pct_Initial_Over_100pct":
            (
                ratio >= 1.00
            ).mean()

    })


capture_stats = pd.DataFrame(
    capture_rows
)


print("\n" + "=" * 90)
print("HOW MUCH OF THE MOVE IS ALREADY IN THE 08:30 CANDLE?")
print("=" * 90)


print(

    capture_stats

    .assign(

        Median_Fraction_Already_Captured=lambda x:
            x[
                "Median_Fraction_Already_Captured"
            ] * 100,

        Mean_Fraction_Already_Captured=lambda x:
            x[
                "Mean_Fraction_Already_Captured"
            ] * 100,

        Pct_Initial_Over_50pct=lambda x:
            x[
                "Pct_Initial_Over_50pct"
            ] * 100,

        Pct_Initial_Over_75pct=lambda x:
            x[
                "Pct_Initial_Over_75pct"
            ] * 100,

        Pct_Initial_Over_100pct=lambda x:
            x[
                "Pct_Initial_Over_100pct"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 12. POST-EVENT MOVE BY ALIGNMENT
# ================================================================

alignment_rows = []


for score in sorted(
    data[
        "Alignment_Score"
    ].dropna().unique()
):


    subset = data[
        data[
            "Alignment_Score"
        ]
        ==
        score
    ]


    for horizon in HORIZONS:


        move = subset[
            f"Post_T{horizon}"
        ].dropna()


        if move.empty:
            continue


        alignment_rows.append({

            "Alignment":
                score,

            "Horizon":
                horizon,

            "N":
                len(move),

            "Mean_Post_Move":
                move.mean(),

            "Median_Post_Move":
                move.median(),

            "Mean_Abs_Post_Move":
                move.abs().mean(),

            "Pct_Post_Up":
                (
                    move > 0
                ).mean()

        })


alignment_post = pd.DataFrame(
    alignment_rows
)


print("\n" + "=" * 90)
print("POST-08:30 MOVE BY ALIGNMENT")
print("=" * 90)


print(

    alignment_post[

        alignment_post[
            "Horizon"
        ].isin(
            [
                1,
                5,
                15,
                30,
                60
            ]
        )

    ]

    .assign(
        Pct_Post_Up=lambda x:
            x[
                "Pct_Post_Up"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 13. MACRO / PRICE AGREEMENT STUDY
# ================================================================

agreement_rows = []


for agree in [
    0,
    1
]:


    subset = data[
        data[
            "Macro_Price_Agree"
        ]
        ==
        agree
    ]


    label = (
        "AGREE"
        if agree == 1
        else
        "DISAGREE"
    )


    for horizon in HORIZONS:


        post = subset[
            f"Post_T{horizon}"
        ].dropna()


        if post.empty:
            continue


        expected = subset.loc[
            post.index,
            "Expected_EURUSD_Direction"
        ]


        macro_trade_return = (

            post
            *
            expected

        )


        agreement_rows.append({

            "Group":
                label,

            "Horizon":
                horizon,

            "N":
                len(post),

            "Mean_Post_Move":
                post.mean(),

            "Mean_Macro_Direction_Return":
                macro_trade_return.mean(),

            "Median_Macro_Direction_Return":
                macro_trade_return.median(),

            "Macro_Direction_Win_Rate":
                (
                    macro_trade_return > 0
                ).mean()

        })


agreement_stats = pd.DataFrame(
    agreement_rows
)


print("\n" + "=" * 90)
print("MACRO / INITIAL PRICE AGREEMENT")
print("=" * 90)


print(

    agreement_stats

    .assign(
        Macro_Direction_Win_Rate=lambda x:
            x[
                "Macro_Direction_Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 14. EVENT CANDLE SIZE BUCKETS
# ================================================================

valid_size = data.dropna(
    subset=[
        "Event_Range_Pips"
    ]
).copy()


valid_size[
    "Event_Range_Bucket"
] = pd.qcut(

    valid_size[
        "Event_Range_Pips"
    ],

    q=4,

    duplicates="drop"

)


size_rows = []


for bucket, subset in valid_size.groupby(
    "Event_Range_Bucket",
    observed=True
):


    for horizon in [
        5,
        15,
        30,
        60
    ]:


        post = subset[
            f"Post_T{horizon}"
        ].dropna()


        if post.empty:
            continue


        initial_direction = subset.loc[
            post.index,
            "Initial_Price_Direction"
        ]


        continuation_return = (

            post
            *
            initial_direction

        )


        size_rows.append({

            "Range_Bucket":
                str(bucket),

            "Horizon":
                horizon,

            "N":
                len(post),

            "Median_Event_Range":
                subset.loc[
                    post.index,
                    "Event_Range_Pips"
                ].median(),

            "Mean_Continuation_Return":
                continuation_return.mean(),

            "Median_Continuation_Return":
                continuation_return.median(),

            "Continuation_Win_Rate":
                (
                    continuation_return > 0
                ).mean()

        })


size_stats = pd.DataFrame(
    size_rows
)


print("\n" + "=" * 90)
print("INITIAL CANDLE SIZE VS CONTINUATION")
print("=" * 90)


print(

    size_stats

    .assign(
        Continuation_Win_Rate=lambda x:
            x[
                "Continuation_Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 15. CANDLE EFFICIENCY BUCKETS
# ================================================================

valid_eff = data.dropna(
    subset=[
        "Event_Efficiency"
    ]
).copy()


valid_eff[
    "Efficiency_Bucket"
] = pd.qcut(

    valid_eff[
        "Event_Efficiency"
    ],

    q=4,

    duplicates="drop"

)


eff_rows = []


for bucket, subset in valid_eff.groupby(
    "Efficiency_Bucket",
    observed=True
):


    for horizon in [
        5,
        15,
        30,
        60
    ]:


        post = subset[
            f"Post_T{horizon}"
        ].dropna()


        if post.empty:
            continue


        initial_direction = subset.loc[
            post.index,
            "Initial_Price_Direction"
        ]


        continuation_return = (

            post
            *
            initial_direction

        )


        eff_rows.append({

            "Efficiency_Bucket":
                str(bucket),

            "Horizon":
                horizon,

            "N":
                len(post),

            "Median_Efficiency":
                subset.loc[
                    post.index,
                    "Event_Efficiency"
                ].median(),

            "Mean_Continuation_Return":
                continuation_return.mean(),

            "Median_Continuation_Return":
                continuation_return.median(),

            "Continuation_Win_Rate":
                (
                    continuation_return > 0
                ).mean()

        })


efficiency_stats = pd.DataFrame(
    eff_rows
)


print("\n" + "=" * 90)
print("EVENT-CANDLE EFFICIENCY VS CONTINUATION")
print("=" * 90)


print(

    efficiency_stats

    .assign(
        Continuation_Win_Rate=lambda x:
            x[
                "Continuation_Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 16. FEATURES FOR WALK-FORWARD MODEL
# ================================================================

FEATURES = [

    "NFP_Z",
    "AHE_Z",
    "U3_Z",

    "Macro_Composite",
    "Macro_Strength",

    "Alignment_Score",

    "Initial_Move_Pips",
    "Event_Range_Pips",
    "Event_Efficiency",
    "Close_Location"

]


# ================================================================
# 17. WALK-FORWARD CLASSIFIER
# ================================================================

def walk_forward_classifier(
    df,
    features,
    target,
    min_train=60
):

    predictions = []


    for i in range(
        min_train,
        len(df)
    ):


        train = (

            df.iloc[:i]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        test = (

            df.iloc[[i]]

            .dropna(
                subset=
                    features
                    +
                    [target]
            )

        )


        if len(test) == 0:
            continue


        if len(train) < min_train:
            continue


        if train[
            target
        ].nunique() < 2:
            continue


        model = Pipeline([

            (
                "scale",
                StandardScaler()
            ),

            (
                "logit",

                LogisticRegression(

                    C=0.25,

                    penalty="l2",

                    solver="liblinear",

                    class_weight="balanced",

                    max_iter=5000

                )

            )

        ])


        model.fit(

            train[
                features
            ],

            train[
                target
            ]

        )


        probability = (

            model

            .predict_proba(

                test[
                    features
                ]

            )[0, 1]

        )


        predictions.append({

            "date":
                test[
                    "date"
                ].iloc[0],

            "Actual":
                int(
                    test[
                        target
                    ].iloc[0]
                ),

            "Probability":
                probability

        })


    return pd.DataFrame(
        predictions
    )


# ================================================================
# 18. MODEL CONTINUATION AT T15 / T30 / T60
# ================================================================

model_rows = []


for horizon in [
    5,
    15,
    30,
    60
]:


    target = (
        f"Continuation_T{horizon}"
    )


    preds = walk_forward_classifier(

        data,

        FEATURES,

        target,

        MIN_TRAIN

    )


    if preds.empty:
        continue


    y = preds[
        "Actual"
    ]

    p = preds[
        "Probability"
    ]


    predicted = (
        p >= 0.5
    ).astype(int)


    result = {

        "Target":
            f"Continuation_T{horizon}",

        "N":
            len(preds),

        "Accuracy":
            accuracy_score(
                y,
                predicted
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y,
                predicted
            )

    }


    if y.nunique() > 1:

        result[
            "ROC_AUC"
        ] = roc_auc_score(
            y,
            p
        )

    else:

        result[
            "ROC_AUC"
        ] = np.nan


    model_rows.append(
        result
    )


continuation_models = pd.DataFrame(
    model_rows
)


print("\n" + "=" * 90)
print("WALK-FORWARD CONTINUATION MODEL")
print("=" * 90)


print(

    continuation_models

    .assign(

        Accuracy=lambda x:
            x["Accuracy"]
            *
            100,

        Balanced_Accuracy=lambda x:
            x[
                "Balanced_Accuracy"
            ]
            *
            100

    )

    .round(3)

    .to_string(
        index=False
    )

)


# ================================================================
# 19. SIMPLE TRADE STUDY:
# CONTINUE VS FADE AFTER EVENT CANDLE
# ================================================================
#
# For each release:
#
# CONTINUE = trade in direction of 08:30 candle
#
# FADE = trade opposite direction of 08:30 candle
#
# measured from 08:30 close
#
# ================================================================

trade_rows = []


for horizon in [
    5,
    15,
    30,
    60
]:


    post = data[
        f"Post_T{horizon}"
    ]


    initial_dir = data[
        "Initial_Price_Direction"
    ]


    continuation_trade = (

        post
        *
        initial_dir

    )


    fade_trade = (

        post
        *
        -initial_dir

    )


    trade_rows.append({

        "Horizon":
            horizon,

        "Continue_Mean":
            continuation_trade.mean(),

        "Continue_Median":
            continuation_trade.median(),

        "Continue_Win":
            (
                continuation_trade > 0
            ).mean(),

        "Fade_Mean":
            fade_trade.mean(),

        "Fade_Median":
            fade_trade.median(),

        "Fade_Win":
            (
                fade_trade > 0
            ).mean()

    })


trade_comparison = pd.DataFrame(
    trade_rows
)


print("\n" + "=" * 90)
print("NAIVE CONTINUE VS FADE FROM 08:30 CLOSE")
print("=" * 90)


print(

    trade_comparison

    .assign(

        Continue_Win=lambda x:
            x["Continue_Win"]
            *
            100,

        Fade_Win=lambda x:
            x["Fade_Win"]
            *
            100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 20. OVERLAY BY ALIGNMENT SCORE
# ================================================================

plot_horizon = 30


for score in [
    -3,
    -2,
    -1,
    0,
    1,
    2,
    3
]:


    subset = data[
        data[
            "Alignment_Score"
        ]
        ==
        score
    ]


    if len(subset) < 3:
        continue


    plt.figure(
        figsize=(12, 6)
    )


    for _, row in subset.iterrows():


        path = [

            0

        ] + [

            row[
                f"Post_T{i}"
            ]

            for i in range(
                1,
                plot_horizon + 1
            )

        ]


        plt.plot(

            range(
                plot_horizon + 1
            ),

            path,

            alpha=0.18,

            linewidth=1

        )


    mean_path = [

        0

    ] + [

        subset[
            f"Post_T{i}"
        ].mean()

        for i in range(
            1,
            plot_horizon + 1
        )

    ]


    plt.plot(

        range(
            plot_horizon + 1
        ),

        mean_path,

        linewidth=4,

        label="Mean post-event path"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes after 08:30 candle close"
    )


    plt.ylabel(
        "EUR/USD pips from 08:30 close"
    )


    plt.title(

        f"Alignment {score:+.0f} "
        f"— Post-NFP Paths [n={len(subset)}]"

    )


    plt.legend()

    plt.grid(
        True,
        alpha=0.25
    )

    plt.show()


# ================================================================
# 21. SAVE RESULTS
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "post_event_master.csv",

    index=False

)


capture_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "initial_move_capture.csv",

    index=False

)


alignment_post.to_csv(

    f"{OUTPUT_DIR}/"
    "post_move_by_alignment.csv",

    index=False

)


agreement_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_price_agreement.csv",

    index=False

)


size_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "event_size_continuation.csv",

    index=False

)


efficiency_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "event_efficiency_continuation.csv",

    index=False

)


continuation_models.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_continuation_models.csv",

    index=False

)


trade_comparison.to_csv(

    f"{OUTPUT_DIR}/"
    "naive_continue_vs_fade.csv",

    index=False

)


# ================================================================
# 22. FINISH
# ================================================================

print("\n" + "=" * 90)
print("POST-EVENT STUDY FINISHED")
print("=" * 90)


print(
    "Events:",
    len(data)
)


print(
    "Saved to:",
    OUTPUT_DIR
)


print(
"""

SEND ME THESE SECTIONS:

1. HOW MUCH OF THE MOVE IS ALREADY IN THE 08:30 CANDLE?

2. POST-08:30 MOVE BY ALIGNMENT

3. MACRO / INITIAL PRICE AGREEMENT

4. INITIAL CANDLE SIZE VS CONTINUATION

5. EVENT-CANDLE EFFICIENCY VS CONTINUATION

6. WALK-FORWARD CONTINUATION MODEL

7. NAIVE CONTINUE VS FADE FROM 08:30 CLOSE


WHAT WE ARE LOOKING FOR:

A) Does a strong initial candle tend to exhaust the move?

B) Does a clean efficient candle tend to continue?

C) When macro and the first candle disagree, is fading or
   following the macro more profitable?

D) Are +/-2 signals potentially more tradable than +/-3?

E) Is there a reliable post-event continuation/reversal model
   across all ~194 releases?

"""
)

In [ ]:
# ================================================================
# NFP POST-08:30 MFE / MAE + TAKE-PROFIT STUDY
# ================================================================
#
# ENTRY REFERENCE:
#
#     08:30 CLOSE
#
# We study everything that happens AFTER that close.
#
#
# FOR EVERY NFP RELEASE:
#
# - raw upside excursion
# - raw downside excursion
# - MFE / MAE in macro direction
# - MFE / MAE in initial-candle direction
# - time to MFE
# - time to MAE
# - TP hit probabilities
# - time to TP
# - adverse excursion BEFORE TP
# - TP-or-time-exit expectancy
#
#
# Macro information remains attached:
#
# - NFP actual / forecast / surprise
# - AHE actual / forecast / surprise
# - U3 actual / forecast / surprise
# - alignment score
# - macro direction
# - initial 08:30 candle direction
# - macro / price agreement
# - event candle range
# - event efficiency
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_mfe_mae_study"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30


# Study up to four hours after NFP

MAX_MINUTE = 240


REPORT_HORIZONS = [
    5,
    10,
    15,
    30,
    45,
    60,
    90,
    120,
    180,
    240
]


# Candidate take profits

TP_LEVELS = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
    60,
    75,
    100
]


# Cost used for TP/time-exit expectancy.
#
# You can change this later.

ROUND_TRIP_COST_PIPS = 3.0


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"
]


missing = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values("date")

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(drop=True)

)


# ================================================================
# 3. MACRO ALIGNMENT
# ================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)


macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)


macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment_Score"] = (

      macro["NFP_Sign"]
    + macro["AHE_Sign"]
    + macro["U3_Sign"]

)


# Positive alignment = USD bullish
#
# Therefore EUR/USD expected DOWN.

macro[
    "Expected_EURUSD_Direction"
] = (

    -np.sign(
        macro[
            "Alignment_Score"
        ]
    )

)


# ================================================================
# 4. LOAD PRICES
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(drop=True)

)


px = (
    prices
    .set_index("Datetime")
    .sort_index()
)


print("=" * 90)
print("INPUT")
print("=" * 90)

print(
    "Macro releases:",
    len(macro)
)

print(
    "Price rows:",
    len(prices)
)

print(
    "Price range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 5. EXTRACT EVENT PATH
# ================================================================

def extract_event(
    release_date
):

    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    pre_dt = (
        event_dt
        -
        pd.Timedelta(minutes=1)
    )


    if (
        pre_dt not in px.index
        or
        event_dt not in px.index
    ):

        return None


    pre_close = float(
        px.loc[
            pre_dt,
            "Close"
        ]
    )


    event = px.loc[
        event_dt
    ]


    event_open = float(
        event["Open"]
    )

    event_high = float(
        event["High"]
    )

    event_low = float(
        event["Low"]
    )

    event_close = float(
        event["Close"]
    )


    record = {

        "date":
            date,

        "Pre_Close":
            pre_close,

        "Entry_Price":
            event_close,

        "Event_Open":
            event_open,

        "Event_High":
            event_high,

        "Event_Low":
            event_low,

        "Event_Close":
            event_close
    }


    # ------------------------------------------------------------
    # FIRST 08:30 MOVE
    # ------------------------------------------------------------

    record[
        "Initial_Move_Pips"
    ] = (

        event_close
        -
        pre_close

    ) * 10000


    record[
        "Event_Range_Pips"
    ] = (

        event_high
        -
        event_low

    ) * 10000


    record[
        "Initial_Price_Direction"
    ] = np.sign(

        record[
            "Initial_Move_Pips"
        ]

    )


    # ------------------------------------------------------------
    # CLOSE LOCATION
    # ------------------------------------------------------------

    event_range = (
        event_high
        -
        event_low
    )


    if event_range > 0:

        record[
            "Close_Location"
        ] = (

            event_close
            -
            event_low

        ) / event_range

    else:

        record[
            "Close_Location"
        ] = np.nan


    # ------------------------------------------------------------
    # EFFICIENCY
    # ------------------------------------------------------------

    if record[
        "Event_Range_Pips"
    ] > 0:

        record[
            "Event_Efficiency"
        ] = (

            abs(
                record[
                    "Initial_Move_Pips"
                ]
            )

            /

            record[
                "Event_Range_Pips"
            ]

        )

    else:

        record[
            "Event_Efficiency"
        ] = np.nan


    # ------------------------------------------------------------
    # RAW PATH AFTER 08:30 CLOSE
    # ------------------------------------------------------------

    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            record[
                f"O{minute}"
            ] = np.nan

            record[
                f"H{minute}"
            ] = np.nan

            record[
                f"L{minute}"
            ] = np.nan

            record[
                f"C{minute}"
            ] = np.nan

            continue


        row = px.loc[
            dt
        ]


        record[
            f"O{minute}"
        ] = float(
            row["Open"]
        )


        record[
            f"H{minute}"
        ] = float(
            row["High"]
        )


        record[
            f"L{minute}"
        ] = float(
            row["Low"]
        )


        record[
            f"C{minute}"
        ] = float(
            row["Close"]
        )


    return record


# ================================================================
# 6. BUILD EVENT DATASET
# ================================================================

event_rows = []


for date in macro[
    "date"
]:

    row = extract_event(
        date
    )


    if row is not None:

        event_rows.append(
            row
        )


events = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    events,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values("date")

    .reset_index(drop=True)

)


# ================================================================
# 7. MACRO / PRICE AGREEMENT
# ================================================================

data[
    "Macro_Price_Agree"
] = (

    data[
        "Expected_EURUSD_Direction"
    ]

    ==

    data[
        "Initial_Price_Direction"
    ]

)


data.loc[

    data[
        "Expected_EURUSD_Direction"
    ] == 0,

    "Macro_Price_Agree"

] = np.nan


print("\nMatched events:", len(data))


# ================================================================
# 8. RAW EXCURSION FUNCTION
# ================================================================

def calculate_excursion(
    row,
    horizon
):

    entry = row[
        "Entry_Price"
    ]


    upside = []
    downside = []


    for minute in range(
        1,
        horizon + 1
    ):


        high = row[
            f"H{minute}"
        ]

        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        upside.append(
            (
                high
                -
                entry
            ) * 10000
        )


        downside.append(
            (
                low
                -
                entry
            ) * 10000
        )


    if len(
        upside
    ) == 0:

        return pd.Series({

            "Max_Up":
                np.nan,

            "Max_Down":
                np.nan,

            "Time_Max_Up":
                np.nan,

            "Time_Max_Down":
                np.nan
        })


    max_up = max(
        upside
    )


    max_down = min(
        downside
    )


    time_max_up = (
        upside.index(
            max_up
        )
        +
        1
    )


    time_max_down = (
        downside.index(
            max_down
        )
        +
        1
    )


    return pd.Series({

        "Max_Up":
            max_up,

        "Max_Down":
            max_down,

        "Time_Max_Up":
            time_max_up,

        "Time_Max_Down":
            time_max_down
    })


# ================================================================
# 9. CALCULATE RAW EXCURSIONS
# ================================================================

for horizon in REPORT_HORIZONS:


    temp = data.apply(

        lambda row:
            calculate_excursion(
                row,
                horizon
            ),

        axis=1

    )


    data[
        f"Max_Up_T{horizon}"
    ] = temp[
        "Max_Up"
    ]


    data[
        f"Max_Down_T{horizon}"
    ] = temp[
        "Max_Down"
    ]


    data[
        f"Time_Max_Up_T{horizon}"
    ] = temp[
        "Time_Max_Up"
    ]


    data[
        f"Time_Max_Down_T{horizon}"
    ] = temp[
        "Time_Max_Down"
    ]


# ================================================================
# 10. NORMALIZED MFE / MAE FUNCTION
# ================================================================

def directional_mfe_mae(
    row,
    direction,
    horizon
):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return pd.Series({

            "MFE":
                np.nan,

            "MAE":
                np.nan,

            "Time_MFE":
                np.nan,

            "Time_MAE":
                np.nan
        })


    max_up = row[
        f"Max_Up_T{horizon}"
    ]


    max_down = row[
        f"Max_Down_T{horizon}"
    ]


    time_up = row[
        f"Time_Max_Up_T{horizon}"
    ]


    time_down = row[
        f"Time_Max_Down_T{horizon}"
    ]


    if direction > 0:

        # LONG EURUSD

        mfe = max_up

        mae = (
            -max_down
        )

        time_mfe = time_up

        time_mae = time_down


    else:

        # SHORT EURUSD

        mfe = (
            -max_down
        )

        mae = max_up

        time_mfe = time_down

        time_mae = time_up


    return pd.Series({

        "MFE":
            mfe,

        "MAE":
            mae,

        "Time_MFE":
            time_mfe,

        "Time_MAE":
            time_mae
    })


# ================================================================
# 11. MFE / MAE FOR BOTH TRADE IDEAS
# ================================================================

for horizon in REPORT_HORIZONS:


    # ------------------------------------------------------------
    # MACRO-DIRECTION TRADE
    # ------------------------------------------------------------

    macro_temp = data.apply(

        lambda row:
            directional_mfe_mae(

                row,

                row[
                    "Expected_EURUSD_Direction"
                ],

                horizon

            ),

        axis=1

    )


    data[
        f"Macro_MFE_T{horizon}"
    ] = macro_temp[
        "MFE"
    ]


    data[
        f"Macro_MAE_T{horizon}"
    ] = macro_temp[
        "MAE"
    ]


    data[
        f"Macro_Time_MFE_T{horizon}"
    ] = macro_temp[
        "Time_MFE"
    ]


    data[
        f"Macro_Time_MAE_T{horizon}"
    ] = macro_temp[
        "Time_MAE"
    ]


    # ------------------------------------------------------------
    # INITIAL-CANDLE CONTINUATION TRADE
    # ------------------------------------------------------------

    initial_temp = data.apply(

        lambda row:
            directional_mfe_mae(

                row,

                row[
                    "Initial_Price_Direction"
                ],

                horizon

            ),

        axis=1

    )


    data[
        f"Initial_MFE_T{horizon}"
    ] = initial_temp[
        "MFE"
    ]


    data[
        f"Initial_MAE_T{horizon}"
    ] = initial_temp[
        "MAE"
    ]


    data[
        f"Initial_Time_MFE_T{horizon}"
    ] = initial_temp[
        "Time_MFE"
    ]


    data[
        f"Initial_Time_MAE_T{horizon}"
    ] = initial_temp[
        "Time_MAE"
    ]


# ================================================================
# 12. SUMMARY MFE / MAE BY HORIZON
# ================================================================

summary_rows = []


for strategy in [
    "Macro",
    "Initial"
]:


    for horizon in REPORT_HORIZONS:


        mfe = data[
            f"{strategy}_MFE_T{horizon}"
        ].dropna()


        mae = data.loc[
            mfe.index,
            f"{strategy}_MAE_T{horizon}"
        ]


        time_mfe = data.loc[
            mfe.index,
            f"{strategy}_Time_MFE_T{horizon}"
        ]


        summary_rows.append({

            "Strategy":
                strategy,

            "Horizon":
                horizon,

            "N":
                len(mfe),

            "Mean_MFE":
                mfe.mean(),

            "Median_MFE":
                mfe.median(),

            "P25_MFE":
                mfe.quantile(0.25),

            "P75_MFE":
                mfe.quantile(0.75),

            "Mean_MAE":
                mae.mean(),

            "Median_MAE":
                mae.median(),

            "P75_MAE":
                mae.quantile(0.75),

            "Median_Time_MFE":
                time_mfe.median(),

            "MFE_MAE_Ratio":
                (
                    mfe.mean()
                    /
                    mae.mean()
                )

                if mae.mean() != 0

                else np.nan

        })


mfe_mae_summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 90)
print("MFE / MAE SUMMARY")
print("=" * 90)


print(

    mfe_mae_summary

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 13. MACRO-DIRECTION MFE / MAE BY ALIGNMENT
# ================================================================

alignment_rows = []


for alignment in sorted(
    data[
        "Alignment_Score"
    ].dropna().unique()
):


    subset = data[
        data[
            "Alignment_Score"
        ]
        ==
        alignment
    ]


    for horizon in [
        15,
        30,
        60,
        120
    ]:


        mfe = subset[
            f"Macro_MFE_T{horizon}"
        ].dropna()


        mae = subset.loc[
            mfe.index,
            f"Macro_MAE_T{horizon}"
        ]


        if mfe.empty:

            continue


        alignment_rows.append({

            "Alignment":
                alignment,

            "Horizon":
                horizon,

            "N":
                len(mfe),

            "Mean_MFE":
                mfe.mean(),

            "Median_MFE":
                mfe.median(),

            "Mean_MAE":
                mae.mean(),

            "Median_MAE":
                mae.median(),

            "Pct_MFE_20plus":
                (
                    mfe >= 20
                ).mean(),

            "Pct_MFE_30plus":
                (
                    mfe >= 30
                ).mean(),

            "Pct_MAE_Under10":
                (
                    mae <= 10
                ).mean()

        })


alignment_mfe = pd.DataFrame(
    alignment_rows
)


print("\n" + "=" * 90)
print("MACRO-DIRECTION MFE / MAE BY ALIGNMENT")
print("=" * 90)


print(

    alignment_mfe

    .assign(

        Pct_MFE_20plus=lambda x:
            x[
                "Pct_MFE_20plus"
            ] * 100,

        Pct_MFE_30plus=lambda x:
            x[
                "Pct_MFE_30plus"
            ] * 100,

        Pct_MAE_Under10=lambda x:
            x[
                "Pct_MAE_Under10"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 14. MACRO / PRICE AGREEMENT MFE / MAE
# ================================================================

agreement_rows = []


for agree in [
    True,
    False
]:


    subset = data[
        data[
            "Macro_Price_Agree"
        ]
        ==
        agree
    ]


    label = (
        "AGREE"
        if agree
        else
        "DISAGREE"
    )


    for horizon in [
        15,
        30,
        60,
        120
    ]:


        mfe = subset[
            f"Macro_MFE_T{horizon}"
        ].dropna()


        mae = subset.loc[
            mfe.index,
            f"Macro_MAE_T{horizon}"
        ]


        if mfe.empty:

            continue


        agreement_rows.append({

            "Group":
                label,

            "Horizon":
                horizon,

            "N":
                len(mfe),

            "Mean_MFE":
                mfe.mean(),

            "Median_MFE":
                mfe.median(),

            "Mean_MAE":
                mae.mean(),

            "Median_MAE":
                mae.median(),

            "MFE_MAE_Ratio":
                (
                    mfe.mean()
                    /
                    mae.mean()
                )

                if mae.mean() != 0

                else np.nan

        })


agreement_mfe = pd.DataFrame(
    agreement_rows
)


print("\n" + "=" * 90)
print("MACRO / PRICE AGREEMENT — MFE / MAE")
print("=" * 90)


print(

    agreement_mfe

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 15. TP FIRST-TOUCH FUNCTION
# ================================================================
#
# Determine:
#
# - whether TP was reached
# - first minute TP was touched
# - maximum adverse excursion before TP
#
# ================================================================

def tp_first_touch(
    row,
    direction,
    tp_pips,
    horizon
):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return pd.Series({

            "TP_Hit":
                0,

            "Time_To_TP":
                np.nan,

            "MAE_Before_TP":
                np.nan,

            "Final_Return":
                np.nan

        })


    entry = row[
        "Entry_Price"
    ]


    worst_adverse = 0


    for minute in range(
        1,
        horizon + 1
    ):


        high = row[
            f"H{minute}"
        ]

        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        if direction > 0:

            favorable = (

                high
                -
                entry

            ) * 10000


            adverse = max(

                0,

                (
                    entry
                    -
                    low
                ) * 10000

            )


        else:

            favorable = (

                entry
                -
                low

            ) * 10000


            adverse = max(

                0,

                (
                    high
                    -
                    entry
                ) * 10000

            )


        worst_adverse = max(
            worst_adverse,
            adverse
        )


        if favorable >= tp_pips:

            return pd.Series({

                "TP_Hit":
                    1,

                "Time_To_TP":
                    minute,

                "MAE_Before_TP":
                    worst_adverse,

                "Final_Return":
                    tp_pips

            })


    # ------------------------------------------------------------
    # TP NOT HIT -> exit at horizon close
    # ------------------------------------------------------------

    final_close = row[
        f"C{horizon}"
    ]


    if pd.isna(
        final_close
    ):

        final_return = np.nan


    else:

        final_return = (

            (
                final_close
                -
                entry
            )

            *
            10000

            *
            direction

        )


    return pd.Series({

        "TP_Hit":
            0,

        "Time_To_TP":
            np.nan,

        "MAE_Before_TP":
            worst_adverse,

        "Final_Return":
            final_return

    })


# ================================================================
# 16. TP STUDY
# ================================================================

tp_rows = []


for strategy in [
    "Macro",
    "Initial"
]:


    if strategy == "Macro":

        direction_column = (
            "Expected_EURUSD_Direction"
        )

    else:

        direction_column = (
            "Initial_Price_Direction"
        )


    for horizon in [
        15,
        30,
        60,
        120
    ]:


        for tp in TP_LEVELS:


            results = data.apply(

                lambda row:
                    tp_first_touch(

                        row,

                        row[
                            direction_column
                        ],

                        tp,

                        horizon

                    ),

                axis=1

            )


            valid = results[
                "Final_Return"
            ].notna()


            temp = results[
                valid
            ]


            if temp.empty:

                continue


            net_return = (

                temp[
                    "Final_Return"
                ]

                -

                ROUND_TRIP_COST_PIPS

            )


            hits = temp[
                temp[
                    "TP_Hit"
                ]
                ==
                1
            ]


            tp_rows.append({

                "Strategy":
                    strategy,

                "Horizon":
                    horizon,

                "TP":
                    tp,

                "N":
                    len(temp),

                "TP_Hit_Rate":
                    temp[
                        "TP_Hit"
                    ].mean(),

                "Mean_Time_To_TP":
                    hits[
                        "Time_To_TP"
                    ].mean(),

                "Median_Time_To_TP":
                    hits[
                        "Time_To_TP"
                    ].median(),

                "Mean_MAE_Before_TP":
                    hits[
                        "MAE_Before_TP"
                    ].mean(),

                "Median_MAE_Before_TP":
                    hits[
                        "MAE_Before_TP"
                    ].median(),

                "Mean_Net_Return":
                    net_return.mean(),

                "Median_Net_Return":
                    net_return.median(),

                "Win_Rate_Net":
                    (
                        net_return > 0
                    ).mean()

            })


tp_stats = pd.DataFrame(
    tp_rows
)


# ================================================================
# 17. BEST TP BY STRATEGY / HORIZON
# ================================================================

best_tp_rows = []


for strategy in [
    "Macro",
    "Initial"
]:


    for horizon in [
        15,
        30,
        60,
        120
    ]:


        subset = tp_stats[

            (
                tp_stats[
                    "Strategy"
                ]
                ==
                strategy
            )

            &

            (
                tp_stats[
                    "Horizon"
                ]
                ==
                horizon
            )

        ]


        if subset.empty:

            continue


        best = subset.loc[

            subset[
                "Mean_Net_Return"
            ].idxmax()

        ]


        best_tp_rows.append(
            best
        )


best_tp = pd.DataFrame(
    best_tp_rows
)


print("\n" + "=" * 90)
print("BEST TAKE PROFIT BY STRATEGY / HORIZON")
print("=" * 90)


print(

    best_tp

    .assign(

        TP_Hit_Rate=lambda x:
            x[
                "TP_Hit_Rate"
            ] * 100,

        Win_Rate_Net=lambda x:
            x[
                "Win_Rate_Net"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 18. FULL TP TABLE — MACRO DIRECTION
# ================================================================

print("\n" + "=" * 90)
print("MACRO-DIRECTION TAKE-PROFIT TABLE")
print("=" * 90)


print(

    tp_stats[
        tp_stats[
            "Strategy"
        ]
        ==
        "Macro"
    ]

    .assign(

        TP_Hit_Rate=lambda x:
            x[
                "TP_Hit_Rate"
            ] * 100,

        Win_Rate_Net=lambda x:
            x[
                "Win_Rate_Net"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 19. EVENT-LEVEL TABLE
# ================================================================
#
# This is the table I expect will be especially useful.
#
# It lets us inspect EVERY release manually.
#
# ================================================================

event_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_USD_Surprise",

    "Alignment_Score",

    "Expected_EURUSD_Direction",

    "Initial_Move_Pips",

    "Initial_Price_Direction",

    "Macro_Price_Agree",

    "Event_Range_Pips",

    "Event_Efficiency",

    "Close_Location",

    "Macro_MFE_T15",
    "Macro_MAE_T15",

    "Macro_MFE_T30",
    "Macro_MAE_T30",

    "Macro_MFE_T60",
    "Macro_MAE_T60",

    "Macro_MFE_T120",
    "Macro_MAE_T120",

    "Macro_Time_MFE_T60",
    "Macro_Time_MAE_T60",

    "Initial_MFE_T15",
    "Initial_MAE_T15",

    "Initial_MFE_T30",
    "Initial_MAE_T30",

    "Initial_MFE_T60",
    "Initial_MAE_T60",

    "Initial_MFE_T120",
    "Initial_MAE_T120"
]


event_level = data[
    event_columns
].copy()


print("\n" + "=" * 90)
print("LAST 25 EVENTS — MACRO + MFE / MAE")
print("=" * 90)


print(

    event_level

    .tail(25)

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 20. BEST MACRO-DIRECTION MFE EVENTS
# ================================================================

top_mfe = (

    event_level

    .sort_values(
        "Macro_MFE_T60",
        ascending=False
    )

    .head(25)

)


print("\n" + "=" * 90)
print("TOP 25 MACRO-DIRECTION MFE EVENTS — T60")
print("=" * 90)


print(

    top_mfe

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 21. WORST MACRO-DIRECTION MAE EVENTS
# ================================================================

worst_mae = (

    event_level

    .sort_values(
        "Macro_MAE_T60",
        ascending=False
    )

    .head(25)

)


print("\n" + "=" * 90)
print("TOP 25 WORST MACRO-DIRECTION MAE EVENTS — T60")
print("=" * 90)


print(

    worst_mae

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 22. MFE VS MAE SCATTER — MACRO DIRECTION
# ================================================================

plot_data = data.dropna(
    subset=[
        "Macro_MFE_T60",
        "Macro_MAE_T60"
    ]
)


plt.figure(
    figsize=(11, 7)
)


plt.scatter(

    plot_data[
        "Macro_MAE_T60"
    ],

    plot_data[
        "Macro_MFE_T60"
    ],

    alpha=0.65

)


plt.xlabel(
    "MAE after 08:30 close (pips)"
)


plt.ylabel(
    "MFE after 08:30 close (pips)"
)


plt.title(
    "Macro-Direction Trade: T60 MFE vs MAE"
)


plt.axline(
    (0, 0),
    slope=1,
    linestyle="--"
)


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 23. MEDIAN MFE / MAE CURVE
# ================================================================

macro_summary = mfe_mae_summary[
    mfe_mae_summary[
        "Strategy"
    ]
    ==
    "Macro"
]


plt.figure(
    figsize=(12, 7)
)


plt.plot(

    macro_summary[
        "Horizon"
    ],

    macro_summary[
        "Median_MFE"
    ],

    marker="o",

    label="Median MFE"

)


plt.plot(

    macro_summary[
        "Horizon"
    ],

    macro_summary[
        "Median_MAE"
    ],

    marker="o",

    label="Median MAE"

)


plt.xlabel(
    "Minutes after 08:30 close"
)


plt.ylabel(
    "Pips"
)


plt.title(
    "Macro-Direction MFE / MAE Development"
)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 24. SAVE RESULTS
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "mfe_mae_master.csv",

    index=False

)


event_level.to_csv(

    f"{OUTPUT_DIR}/"
    "event_level_macro_mfe_mae.csv",

    index=False

)


mfe_mae_summary.to_csv(

    f"{OUTPUT_DIR}/"
    "mfe_mae_summary.csv",

    index=False

)


alignment_mfe.to_csv(

    f"{OUTPUT_DIR}/"
    "mfe_mae_by_alignment.csv",

    index=False

)


agreement_mfe.to_csv(

    f"{OUTPUT_DIR}/"
    "mfe_mae_macro_price_agreement.csv",

    index=False

)


tp_stats.to_csv(

    f"{OUTPUT_DIR}/"
    "take_profit_study.csv",

    index=False

)


best_tp.to_csv(

    f"{OUTPUT_DIR}/"
    "best_take_profit.csv",

    index=False

)


top_mfe.to_csv(

    f"{OUTPUT_DIR}/"
    "top_mfe_events.csv",

    index=False

)


worst_mae.to_csv(

    f"{OUTPUT_DIR}/"
    "worst_mae_events.csv",

    index=False

)


# ================================================================
# 25. FINISH
# ================================================================

print("\n" + "=" * 90)
print("MFE / MAE STUDY FINISHED")
print("=" * 90)


print(
    "Events:",
    len(data)
)


print(
    "Saved to:",
    OUTPUT_DIR
)


print(
"""

SEND ME THESE OUTPUTS:

1. MFE / MAE SUMMARY

2. MACRO-DIRECTION MFE / MAE BY ALIGNMENT

3. MACRO / PRICE AGREEMENT — MFE / MAE

4. BEST TAKE PROFIT BY STRATEGY / HORIZON

5. MACRO-DIRECTION TAKE-PROFIT TABLE

6. TOP 25 MACRO-DIRECTION MFE EVENTS — T60

7. TOP 25 WORST MACRO-DIRECTION MAE EVENTS — T60


MOST IMPORTANT QUESTIONS:

A) What is the typical MFE after entering at the 08:30 close?

B) How much MAE must we tolerate to capture that MFE?

C) Does a 15 / 20 / 30 / 40 pip TP capture most of the
   opportunity without requiring huge drawdown?

D) How long does it normally take to hit the profitable extreme?

E) Do macro-price disagreement events have better
   MFE/MAE characteristics?

F) Which NFP/AHE/U3 surprise combinations create the cleanest
   post-event trades?

"""
)

In [ ]:
# ================================================================
# MASTER NFP POST-08:30 TRADING RESEARCH
# ================================================================
#
# GOALS
# -----
#
# 1. Correct MFE / MAE after the 08:30 candle close.
#
# 2. Focus strongly on short horizons:
#
#       T5
#       T10
#       T15
#       T30
#       T60
#
# 3. Compare:
#
#       A) trade MACRO direction
#       B) trade macro direction ONLY when first candle disagrees
#       C) trade macro direction when first candle agrees
#       D) FOLLOW first candle
#       E) FADE first candle
#
# 4. Study "clean trades":
#
#       high MFE
#       low MAE
#
#    and inspect what the macro release looked like.
#
# 5. Study a PULLBACK strategy:
#
#       first candle moves strongly
#       price extends further after 08:30
#       enter against that extension
#       target a retracement
#
# 6. Study DYNAMIC / TRAILING STOPS.
#
# 7. Run chronological 70/30 tests so we do not simply optimize
#    the entire history and fool ourselves.
#
#
# ENTRY REFERENCE
# ---------------
#
# 08:30 close.
#
# The macro data is known at the release.
#
# The 08:30 candle itself is allowed to finish before entry.
#
#
# COST
# ----
#
# All strategy P&L tables below use:
#
#       3 pip round-trip trading cost
#
# Change COST_PIPS below if desired.
#
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_master_postevent_v2"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30


MAX_MINUTE = 60


HORIZONS = [
    5,
    10,
    15,
    30,
    60
]


COST_PIPS = 3.0


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


macro_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"

]


missing = [
    c
    for c in macro_columns
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_columns
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values("date")

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(drop=True)

)


# ================================================================
# 3. BUILD MACRO SIGNAL
# ================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)


macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)


macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment"] = (

      macro["NFP_Sign"]

    + macro["AHE_Sign"]

    + macro["U3_Sign"]

)


# ------------------------------------------------
# Positive alignment = USD bullish
# therefore EURUSD bearish.
# ------------------------------------------------

macro["Macro_Direction"] = (

    -np.sign(
        macro["Alignment"]
    )

)


macro["Abs_Alignment"] = (
    macro["Alignment"].abs()
)


# ================================================================
# 4. MACRO MAGNITUDE
# ================================================================

macro["Abs_NFP_Surprise"] = (
    macro["NFP_Surprise"].abs()
)


macro["Abs_AHE_Surprise"] = (
    macro["AHE_Surprise"].abs()
)


macro["Abs_U3_Surprise"] = (
    macro["U3_USD_Surprise"].abs()
)


# ================================================================
# 5. LOAD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(drop=True)

)


px = (

    prices

    .set_index("Datetime")

    .sort_index()

)


print("=" * 95)
print("INPUT")
print("=" * 95)

print(
    "Macro releases:",
    len(macro)
)

print(
    "Price rows:",
    len(prices)
)

print(
    "Range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# ================================================================
# 6. EXTRACT EVENT
# ================================================================

def extract_event(
    release_date
):

    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    pre_dt = (

        event_dt

        -

        pd.Timedelta(minutes=1)

    )


    if (
        pre_dt not in px.index
        or
        event_dt not in px.index
    ):

        return None


    pre_close = float(
        px.loc[
            pre_dt,
            "Close"
        ]
    )


    event = px.loc[
        event_dt
    ]


    event_open = float(
        event["Open"]
    )

    event_high = float(
        event["High"]
    )

    event_low = float(
        event["Low"]
    )

    event_close = float(
        event["Close"]
    )


    record = {

        "date":
            date,

        "Pre_Close":
            pre_close,

        "Entry_Price":
            event_close,

        "Event_Open":
            event_open,

        "Event_High":
            event_high,

        "Event_Low":
            event_low,

        "Event_Close":
            event_close
    }


    # ============================================================
    # 08:30 CANDLE FEATURES
    # ============================================================

    record[
        "Initial_Move_Pips"
    ] = (

        event_close

        -

        pre_close

    ) * 10000


    record[
        "Initial_Direction"
    ] = np.sign(

        record[
            "Initial_Move_Pips"
        ]

    )


    record[
        "Event_Range_Pips"
    ] = (

        event_high
        -
        event_low

    ) * 10000


    record[
        "Event_Body_Pips"
    ] = (

        event_close
        -
        event_open

    ) * 10000


    record[
        "Abs_Initial_Move"
    ] = abs(

        record[
            "Initial_Move_Pips"
        ]

    )


    # ------------------------------------------------------------
    # close location
    #
    # 0 = candle low
    # 1 = candle high
    # ------------------------------------------------------------

    rng = (
        event_high
        -
        event_low
    )


    if rng > 0:

        record[
            "Close_Location"
        ] = (

            event_close
            -
            event_low

        ) / rng

    else:

        record[
            "Close_Location"
        ] = np.nan


    # ------------------------------------------------------------
    # efficiency
    # ------------------------------------------------------------

    if record[
        "Event_Range_Pips"
    ] > 0:

        record[
            "Event_Efficiency"
        ] = (

            record[
                "Abs_Initial_Move"
            ]

            /

            record[
                "Event_Range_Pips"
            ]

        )

    else:

        record[
            "Event_Efficiency"
        ] = np.nan


    # ============================================================
    # POST-EVENT OHLC
    # ============================================================

    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            for prefix in [
                "O",
                "H",
                "L",
                "C"
            ]:

                record[
                    f"{prefix}{minute}"
                ] = np.nan

            continue


        bar = px.loc[
            dt
        ]


        record[
            f"O{minute}"
        ] = float(
            bar["Open"]
        )


        record[
            f"H{minute}"
        ] = float(
            bar["High"]
        )


        record[
            f"L{minute}"
        ] = float(
            bar["Low"]
        )


        record[
            f"C{minute}"
        ] = float(
            bar["Close"]
        )


    return record


# ================================================================
# 7. BUILD EVENT DATA
# ================================================================

event_rows = []


for date in macro[
    "date"
]:

    row = extract_event(
        date
    )


    if row is not None:

        event_rows.append(
            row
        )


events = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    events,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values("date")

    .reset_index(drop=True)

)


print(
    "\nMatched releases:",
    len(data)
)


# ================================================================
# 8. MACRO / INITIAL PRICE AGREEMENT
# ================================================================

data[
    "Macro_Price_Agree"
] = (

    data[
        "Macro_Direction"
    ]

    ==

    data[
        "Initial_Direction"
    ]

)


data.loc[

    data[
        "Macro_Direction"
    ] == 0,

    "Macro_Price_Agree"

] = np.nan


data[
    "Macro_Price_Group"
] = np.select(

    [

        data[
            "Macro_Price_Agree"
        ] == True,

        data[
            "Macro_Price_Agree"
        ] == False

    ],

    [
        "AGREE",
        "DISAGREE"
    ],

    default="NEUTRAL_MACRO"

)


# ================================================================
# 9. CORRECT MFE / MAE
# ================================================================
#
# MFE and MAE are always >= 0.
#
# ================================================================

def mfe_mae(
    row,
    direction,
    horizon
):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return pd.Series({

            "MFE":
                np.nan,

            "MAE":
                np.nan,

            "Time_MFE":
                np.nan,

            "Time_MAE":
                np.nan

        })


    entry = row[
        "Entry_Price"
    ]


    max_favorable = 0.0
    max_adverse = 0.0

    time_mfe = 0
    time_mae = 0


    for minute in range(
        1,
        horizon + 1
    ):


        high = row[
            f"H{minute}"
        ]


        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        if direction > 0:

            favorable = max(

                0,

                (
                    high
                    -
                    entry
                )
                *
                10000

            )


            adverse = max(

                0,

                (
                    entry
                    -
                    low
                )
                *
                10000

            )


        else:

            favorable = max(

                0,

                (
                    entry
                    -
                    low
                )
                *
                10000

            )


            adverse = max(

                0,

                (
                    high
                    -
                    entry
                )
                *
                10000

            )


        if favorable > max_favorable:

            max_favorable = favorable
            time_mfe = minute


        if adverse > max_adverse:

            max_adverse = adverse
            time_mae = minute


    return pd.Series({

        "MFE":
            max_favorable,

        "MAE":
            max_adverse,

        "Time_MFE":
            time_mfe,

        "Time_MAE":
            time_mae

    })


# ================================================================
# 10. CREATE MFE / MAE FOR 3 IDEAS
# ================================================================
#
# A) MACRO direction
#
# B) FOLLOW initial candle
#
# C) FADE initial candle
#
# ================================================================

for horizon in HORIZONS:


    for name, direction_col, multiplier in [

        (
            "Macro",
            "Macro_Direction",
            1
        ),

        (
            "Follow",
            "Initial_Direction",
            1
        ),

        (
            "Fade",
            "Initial_Direction",
            -1
        )

    ]:


        temp = data.apply(

            lambda row:

                mfe_mae(

                    row,

                    (
                        row[
                            direction_col
                        ]

                        *
                        multiplier

                    ),

                    horizon

                ),

            axis=1

        )


        data[
            f"{name}_MFE_T{horizon}"
        ] = temp[
            "MFE"
        ]


        data[
            f"{name}_MAE_T{horizon}"
        ] = temp[
            "MAE"
        ]


        data[
            f"{name}_Time_MFE_T{horizon}"
        ] = temp[
            "Time_MFE"
        ]


# ================================================================
# 11. FIXED-HORIZON RETURN
# ================================================================

def fixed_return(
    row,
    direction,
    horizon,
    cost=COST_PIPS
):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return np.nan


    final = row[
        f"C{horizon}"
    ]


    if pd.isna(
        final
    ):

        return np.nan


    gross = (

        (
            final
            -
            row[
                "Entry_Price"
            ]
        )

        *
        10000

        *
        direction

    )


    return (
        gross
        -
        cost
    )


# ================================================================
# 12. SHORT-HORIZON STRATEGY COMPARISON
# ================================================================

strategy_rows = []


for horizon in HORIZONS:


    strategies = {


        "MACRO_ALL":

            (
                data[
                    "Macro_Direction"
                ],

                data[
                    "Macro_Direction"
                ] != 0

            ),


        "MACRO_DISAGREE":

            (
                data[
                    "Macro_Direction"
                ],

                data[
                    "Macro_Price_Group"
                ]
                ==
                "DISAGREE"

            ),


        "MACRO_AGREE":

            (
                data[
                    "Macro_Direction"
                ],

                data[
                    "Macro_Price_Group"
                ]
                ==
                "AGREE"

            ),


        "FOLLOW_INITIAL":

            (
                data[
                    "Initial_Direction"
                ],

                data[
                    "Initial_Direction"
                ] != 0

            ),


        "FADE_INITIAL":

            (
                -data[
                    "Initial_Direction"
                ],

                data[
                    "Initial_Direction"
                ] != 0

            )

    }


    for strategy, (
        directions,
        mask
    ) in strategies.items():


        pnl = []


        for idx in data[
            mask
        ].index:


            r = fixed_return(

                data.loc[
                    idx
                ],

                directions.loc[
                    idx
                ],

                horizon

            )


            if pd.notna(
                r
            ):

                pnl.append(
                    r
                )


        pnl = pd.Series(
            pnl
        )


        if pnl.empty:

            continue


        strategy_rows.append({

            "Strategy":
                strategy,

            "Horizon":
                horizon,

            "Trades":
                len(pnl),

            "Mean_Net":
                pnl.mean(),

            "Median_Net":
                pnl.median(),

            "Win_Rate":
                (
                    pnl > 0
                ).mean(),

            "Total_Net":
                pnl.sum()

        })


short_strategy = pd.DataFrame(
    strategy_rows
)


print("\n" + "=" * 95)
print("SHORT-HORIZON STRATEGY COMPARISON")
print("=" * 95)


print(

    short_strategy

    .assign(
        Win_Rate=lambda x:
            x[
                "Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 13. CLEAN-WINNER CLASSIFICATION
# ================================================================
#
# This is designed to answer:
#
# what did the releases look like when the trade was actually
# clean enough to exploit?
#
# ================================================================

data[
    "Clean_Macro_T10"
] = (

    (
        data[
            "Macro_MFE_T10"
        ]
        >=
        10
    )

    &

    (
        data[
            "Macro_MAE_T10"
        ]
        <=
        8
    )

)


data[
    "Clean_Macro_T15"
] = (

    (
        data[
            "Macro_MFE_T15"
        ]
        >=
        15
    )

    &

    (
        data[
            "Macro_MAE_T15"
        ]
        <=
        10
    )

)


data[
    "Excellent_Asymmetry_T15"
] = (

    data[
        "Macro_MFE_T15"
    ]

    >=

    (
        2

        *

        data[
            "Macro_MAE_T15"
        ]

    )

)


# ================================================================
# 14. CLEAN-TRADE PROFILE BY MACRO/PRICE GROUP
# ================================================================

profile_rows = []


for group in [

    "AGREE",
    "DISAGREE",
    "NEUTRAL_MACRO"

]:


    subset = data[
        data[
            "Macro_Price_Group"
        ]
        ==
        group
    ]


    if subset.empty:

        continue


    profile_rows.append({

        "Group":
            group,

        "N":
            len(subset),

        "Clean_T10_Pct":
            subset[
                "Clean_Macro_T10"
            ].mean(),

        "Clean_T15_Pct":
            subset[
                "Clean_Macro_T15"
            ].mean(),

        "Excellent_Asymmetry_Pct":
            subset[
                "Excellent_Asymmetry_T15"
            ].mean(),

        "Median_Initial_Move":
            subset[
                "Abs_Initial_Move"
            ].median(),

        "Median_Range":
            subset[
                "Event_Range_Pips"
            ].median(),

        "Median_Efficiency":
            subset[
                "Event_Efficiency"
            ].median()

    })


clean_profile = pd.DataFrame(
    profile_rows
)


print("\n" + "=" * 95)
print("CLEAN TRADE PROFILE")
print("=" * 95)


print(

    clean_profile

    .assign(

        Clean_T10_Pct=lambda x:
            x[
                "Clean_T10_Pct"
            ] * 100,

        Clean_T15_Pct=lambda x:
            x[
                "Clean_T15_Pct"
            ] * 100,

        Excellent_Asymmetry_Pct=lambda x:
            x[
                "Excellent_Asymmetry_Pct"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 15. CLEAN TRADE PROFILE BY ALIGNMENT
# ================================================================

alignment_profile = (

    data[
        data[
            "Macro_Direction"
        ]
        !=
        0
    ]

    .groupby(
        "Alignment"
    )

    .agg(

        N=(
            "date",
            "count"
        ),

        Clean_T10=(
            "Clean_Macro_T10",
            "mean"
        ),

        Clean_T15=(
            "Clean_Macro_T15",
            "mean"
        ),

        Excellent_T15=(
            "Excellent_Asymmetry_T15",
            "mean"
        ),

        Median_MFE_T10=(
            "Macro_MFE_T10",
            "median"
        ),

        Median_MAE_T10=(
            "Macro_MAE_T10",
            "median"
        ),

        Median_MFE_T15=(
            "Macro_MFE_T15",
            "median"
        ),

        Median_MAE_T15=(
            "Macro_MAE_T15",
            "median"
        )

    )

    .reset_index()

)


print("\n" + "=" * 95)
print("CLEAN TRADES BY ALIGNMENT")
print("=" * 95)


print(

    alignment_profile

    .assign(

        Clean_T10=lambda x:
            x[
                "Clean_T10"
            ] * 100,

        Clean_T15=lambda x:
            x[
                "Clean_T15"
            ] * 100,

        Excellent_T15=lambda x:
            x[
                "Excellent_T15"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 16. EVENT-LEVEL CLEAN WINNERS / BAD TRADES
# ================================================================

inspect_cols = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_USD_Surprise",

    "Alignment",

    "Macro_Price_Group",

    "Initial_Move_Pips",

    "Event_Range_Pips",

    "Event_Efficiency",

    "Close_Location",

    "Macro_MFE_T5",
    "Macro_MAE_T5",

    "Macro_MFE_T10",
    "Macro_MAE_T10",

    "Macro_MFE_T15",
    "Macro_MAE_T15",

    "Macro_MFE_T30",
    "Macro_MAE_T30"

]


clean_events = (

    data[
        data[
            "Clean_Macro_T15"
        ]
        ==
        True
    ]

    [inspect_cols]

    .copy()

)


bad_events = (

    data[
        (
            data[
                "Macro_Direction"
            ]
            !=
            0
        )

        &

        (
            data[
                "Macro_MAE_T15"
            ]
            >=
            20
        )

        &

        (
            data[
                "Macro_MFE_T15"
            ]
            <
            10
        )
    ]

    [inspect_cols]

    .copy()

)


print("\n" + "=" * 95)
print("CLEAN T15 MACRO TRADES")
print("=" * 95)


print(

    clean_events

    .round(2)

    .to_string(
        index=False
    )

)


print("\n" + "=" * 95)
print("BAD T15 MACRO TRADES")
print("=" * 95)


print(

    bad_events

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 17. IMMEDIATE PULLBACK / FADE STUDY
# ================================================================
#
# Bet that the initial 08:30 move retraces after the candle closes.
#
# Direction:
#
#       opposite Initial_Direction
#
# ================================================================

pullback_rows = []


# ------------------------------------------------
# Initial-move buckets
# ------------------------------------------------

move_bins = [

    0,
    10,
    20,
    30,
    40,
    60,
    np.inf

]


move_labels = [

    "0-10",
    "10-20",
    "20-30",
    "30-40",
    "40-60",
    "60+"

]


data[
    "Initial_Move_Bucket"
] = pd.cut(

    data[
        "Abs_Initial_Move"
    ],

    bins=move_bins,

    labels=move_labels,

    right=False

)


for bucket in move_labels:


    subset = data[
        data[
            "Initial_Move_Bucket"
        ]
        ==
        bucket
    ]


    for horizon in [
        5,
        10,
        15,
        30
    ]:


        pnl = []


        for idx, row in subset.iterrows():


            direction = (
                -row[
                    "Initial_Direction"
                ]
            )


            r = fixed_return(

                row,

                direction,

                horizon

            )


            if pd.notna(
                r
            ):

                pnl.append(
                    r
                )


        pnl = pd.Series(
            pnl
        )


        if pnl.empty:

            continue


        pullback_rows.append({

            "Initial_Move_Bucket":
                bucket,

            "Horizon":
                horizon,

            "Trades":
                len(pnl),

            "Mean_Net":
                pnl.mean(),

            "Median_Net":
                pnl.median(),

            "Win_Rate":
                (
                    pnl > 0
                ).mean()

        })


immediate_pullback = pd.DataFrame(
    pullback_rows
)


print("\n" + "=" * 95)
print("IMMEDIATE FADE BY INITIAL MOVE SIZE")
print("=" * 95)


print(

    immediate_pullback

    .assign(
        Win_Rate=lambda x:
            x[
                "Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 18. EXTENSION-THEN-PULLBACK SIMULATOR
# ================================================================
#
# Example:
#
# 08:30 candle goes UP.
#
# After 08:30 close, EURUSD extends another 10 pips upward.
#
# We then SHORT that extension, betting on a pullback.
#
#
# Conservative rules:
#
# - entry happens at threshold price
# - activation bar itself cannot immediately hit TP
# - TP / stop checked starting next minute
# - if stop and TP both occur in same later M1 bar:
#       STOP FIRST
#
# ================================================================

def extension_pullback_trade(

    row,

    extension_pips,

    trigger_window,

    target_pips,

    stop_pips,

    final_exit

):


    initial_direction = row[
        "Initial_Direction"
    ]


    if (
        pd.isna(initial_direction)
        or
        initial_direction == 0
    ):

        return np.nan


    event_close = row[
        "Entry_Price"
    ]


    # ------------------------------------------------------------
    # FIND EXTENSION TRIGGER
    # ------------------------------------------------------------

    trigger_minute = None
    entry_price = None


    for minute in range(
        1,
        trigger_window + 1
    ):


        high = row[
            f"H{minute}"
        ]


        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        if initial_direction > 0:

            trigger_price = (

                event_close

                +

                extension_pips
                /
                10000

            )


            if high >= trigger_price:

                trigger_minute = minute
                entry_price = trigger_price
                trade_direction = -1
                break


        else:

            trigger_price = (

                event_close

                -

                extension_pips
                /
                10000

            )


            if low <= trigger_price:

                trigger_minute = minute
                entry_price = trigger_price
                trade_direction = 1
                break


    if trigger_minute is None:

        return np.nan


    # ------------------------------------------------------------
    # STOP / TARGET
    # ------------------------------------------------------------

    if trade_direction > 0:

        stop_price = (

            entry_price

            -

            stop_pips
            /
            10000

        )


        target_price = (

            entry_price

            +

            target_pips
            /
            10000

        )


    else:

        stop_price = (

            entry_price

            +

            stop_pips
            /
            10000

        )


        target_price = (

            entry_price

            -

            target_pips
            /
            10000

        )


    # ------------------------------------------------------------
    # Start checking NEXT minute
    # ------------------------------------------------------------

    for minute in range(
        trigger_minute + 1,
        final_exit + 1
    ):


        high = row[
            f"H{minute}"
        ]


        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        if trade_direction > 0:

            stop_hit = (
                low <= stop_price
            )

            target_hit = (
                high >= target_price
            )


        else:

            stop_hit = (
                high >= stop_price
            )

            target_hit = (
                low <= target_price
            )


        # conservative intrabar assumption

        if stop_hit:

            return (
                -stop_pips
                -
                COST_PIPS
            )


        if target_hit:

            return (
                target_pips
                -
                COST_PIPS
            )


    # ------------------------------------------------------------
    # TIME EXIT
    # ------------------------------------------------------------

    final_close = row[
        f"C{final_exit}"
    ]


    if pd.isna(
        final_close
    ):

        return np.nan


    gross = (

        (
            final_close
            -
            entry_price
        )

        *
        10000

        *
        trade_direction

    )


    return (
        gross
        -
        COST_PIPS
    )


# ================================================================
# 19. PULLBACK PARAMETER GRID
# ================================================================

EXTENSIONS = [
    5,
    10,
    15,
    20
]


TRIGGER_WINDOWS = [
    3,
    5,
    10
]


PULLBACK_TARGETS = [
    5,
    10,
    15,
    20
]


PULLBACK_STOPS = [
    10,
    15,
    20,
    30
]


PULLBACK_EXITS = [
    10,
    15,
    30
]


pullback_grid_rows = []


for extension in EXTENSIONS:

    for window in TRIGGER_WINDOWS:

        for target in PULLBACK_TARGETS:

            for stop in PULLBACK_STOPS:

                for final_exit in PULLBACK_EXITS:


                    if final_exit <= window:

                        continue


                    pnl = data.apply(

                        lambda row:

                            extension_pullback_trade(

                                row,

                                extension,

                                window,

                                target,

                                stop,

                                final_exit

                            ),

                        axis=1

                    ).dropna()


                    # Require enough actual triggered trades

                    if len(
                        pnl
                    ) < 20:

                        continue


                    pullback_grid_rows.append({

                        "Extension":
                            extension,

                        "Trigger_Window":
                            window,

                        "Target":
                            target,

                        "Stop":
                            stop,

                        "Final_Exit":
                            final_exit,

                        "Trades":
                            len(pnl),

                        "Mean_Net":
                            pnl.mean(),

                        "Median_Net":
                            pnl.median(),

                        "Win_Rate":
                            (
                                pnl > 0
                            ).mean(),

                        "Total_Net":
                            pnl.sum()

                    })


pullback_grid = pd.DataFrame(
    pullback_grid_rows
)


pullback_ranked = (

    pullback_grid

    .sort_values(
        "Mean_Net",
        ascending=False
    )

    .reset_index(drop=True)

)


print("\n" + "=" * 95)
print("TOP 25 EXTENSION -> PULLBACK SETUPS")
print("=" * 95)


print(

    pullback_ranked

    .head(25)

    .assign(
        Win_Rate=lambda x:
            x[
                "Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 20. 70/30 PULLBACK ROBUSTNESS
# ================================================================

split = int(
    len(data)
    *
    0.70
)


train = (
    data
    .iloc[:split]
    .copy()
)


test = (
    data
    .iloc[split:]
    .copy()
)


train_grid_rows = []


for extension in EXTENSIONS:

    for window in TRIGGER_WINDOWS:

        for target in PULLBACK_TARGETS:

            for stop in PULLBACK_STOPS:

                for final_exit in PULLBACK_EXITS:


                    if final_exit <= window:

                        continue


                    pnl = train.apply(

                        lambda row:

                            extension_pullback_trade(

                                row,

                                extension,

                                window,

                                target,

                                stop,

                                final_exit

                            ),

                        axis=1

                    ).dropna()


                    if len(
                        pnl
                    ) < 15:

                        continue


                    train_grid_rows.append({

                        "Extension":
                            extension,

                        "Window":
                            window,

                        "Target":
                            target,

                        "Stop":
                            stop,

                        "Exit":
                            final_exit,

                        "Mean":
                            pnl.mean()

                    })


train_pullback_grid = pd.DataFrame(
    train_grid_rows
)


if not train_pullback_grid.empty:


    pb_best = train_pullback_grid.loc[

        train_pullback_grid[
            "Mean"
        ].idxmax()

    ]


    test_pb = test.apply(

        lambda row:

            extension_pullback_trade(

                row,

                int(
                    pb_best[
                        "Extension"
                    ]
                ),

                int(
                    pb_best[
                        "Window"
                    ]
                ),

                int(
                    pb_best[
                        "Target"
                    ]
                ),

                int(
                    pb_best[
                        "Stop"
                    ]
                ),

                int(
                    pb_best[
                        "Exit"
                    ]
                )

            ),

        axis=1

    ).dropna()


    print("\n" + "=" * 95)
    print("70/30 PULLBACK TEST")
    print("=" * 95)


    print(
        "TRAIN selected:"
    )


    print(
        pb_best.to_string()
    )


    print(
        "\nTEST triggered trades:",
        len(test_pb)
    )


    print(
        f"Test mean:   "
        f"{test_pb.mean():.2f}"
    )


    print(
        f"Test median: "
        f"{test_pb.median():.2f}"
    )


    print(
        f"Test win:    "
        f"{(test_pb > 0).mean():.1%}"
    )


# ================================================================
# 21. CONSERVATIVE TRAILING STOP SIMULATOR
# ================================================================
#
# Designed primarily for MACRO-DISAGREEMENT setup.
#
#
# Rules:
#
# - enter at 08:30 close
# - direction = MACRO direction
# - fixed catastrophe stop always exists
# - trailing stop activates after X pips favorable
# - trailing stop uses only PREVIOUS COMPLETED information
#
# This avoids assuming we know the intrabar high/low sequence.
#
# ================================================================

def trailing_trade(

    row,

    direction,

    activation,

    trail_distance,

    hard_stop,

    final_exit

):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return np.nan


    entry = row[
        "Entry_Price"
    ]


    if direction > 0:

        hard_stop_price = (

            entry

            -

            hard_stop
            /
            10000

        )

    else:

        hard_stop_price = (

            entry

            +

            hard_stop
            /
            10000

        )


    best_favorable = 0.0

    trailing_active = False
    trailing_stop_price = None


    for minute in range(
        1,
        final_exit + 1
    ):


        high = row[
            f"H{minute}"
        ]


        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        # ========================================================
        # 1. CHECK HARD STOP FIRST
        # ========================================================

        if direction > 0:

            if low <= hard_stop_price:

                return (
                    -hard_stop
                    -
                    COST_PIPS
                )


        else:

            if high >= hard_stop_price:

                return (
                    -hard_stop
                    -
                    COST_PIPS
                )


        # ========================================================
        # 2. CHECK EXISTING TRAILING STOP
        #
        # It was calculated from PRIOR completed bars.
        # ========================================================

        if trailing_active:


            if direction > 0:

                if low <= trailing_stop_price:

                    gross = (

                        trailing_stop_price
                        -
                        entry

                    ) * 10000


                    return (
                        gross
                        -
                        COST_PIPS
                    )


            else:

                if high >= trailing_stop_price:

                    gross = (

                        entry
                        -
                        trailing_stop_price

                    ) * 10000


                    return (
                        gross
                        -
                        COST_PIPS
                    )


        # ========================================================
        # 3. UPDATE FAVORABLE EXCURSION USING THIS COMPLETED BAR
        # ========================================================

        if direction > 0:

            favorable = (

                high
                -
                entry

            ) * 10000


        else:

            favorable = (

                entry
                -
                low

            ) * 10000


        best_favorable = max(

            best_favorable,

            favorable

        )


        # ========================================================
        # 4. ACTIVATE / RATChet TRAILING STOP
        #
        # New stop becomes active NEXT minute.
        # ========================================================

        if best_favorable >= activation:


            trailing_active = True


            locked_pips = (

                best_favorable

                -

                trail_distance

            )


            if direction > 0:

                candidate = (

                    entry

                    +

                    locked_pips
                    /
                    10000

                )


                if trailing_stop_price is None:

                    trailing_stop_price = candidate

                else:

                    trailing_stop_price = max(

                        trailing_stop_price,

                        candidate

                    )


            else:

                candidate = (

                    entry

                    -

                    locked_pips
                    /
                    10000

                )


                if trailing_stop_price is None:

                    trailing_stop_price = candidate

                else:

                    trailing_stop_price = min(

                        trailing_stop_price,

                        candidate

                    )


    # ============================================================
    # TIME EXIT
    # ============================================================

    final_close = row[
        f"C{final_exit}"
    ]


    if pd.isna(
        final_close
    ):

        return np.nan


    gross = (

        (
            final_close
            -
            entry
        )

        *
        10000

        *
        direction

    )


    return (
        gross
        -
        COST_PIPS
    )


# ================================================================
# 22. TRAILING-STOP GRID
# ================================================================
#
# Main focus:
#
# MACRO direction + INITIAL PRICE DISAGREEMENT
#
# ================================================================

disagree = data[

    data[
        "Macro_Price_Group"
    ]
    ==
    "DISAGREE"

].copy()


TRAIL_ACTIVATIONS = [
    5,
    10,
    15
]


TRAIL_DISTANCES = [
    5,
    10,
    15,
    20
]


HARD_STOPS = [
    15,
    20,
    30,
    40
]


TRAIL_EXITS = [
    5,
    10,
    15,
    30,
    60
]


trail_rows = []


for activation in TRAIL_ACTIVATIONS:

    for trail in TRAIL_DISTANCES:

        for stop in HARD_STOPS:

            for final_exit in TRAIL_EXITS:


                pnl = disagree.apply(

                    lambda row:

                        trailing_trade(

                            row,

                            row[
                                "Macro_Direction"
                            ],

                            activation,

                            trail,

                            stop,

                            final_exit

                        ),

                    axis=1

                ).dropna()


                if len(
                    pnl
                ) < 20:

                    continue


                trail_rows.append({

                    "Activation":
                        activation,

                    "Trail":
                        trail,

                    "Hard_Stop":
                        stop,

                    "Exit":
                        final_exit,

                    "Trades":
                        len(pnl),

                    "Mean_Net":
                        pnl.mean(),

                    "Median_Net":
                        pnl.median(),

                    "Win_Rate":
                        (
                            pnl > 0
                        ).mean(),

                    "Total_Net":
                        pnl.sum()

                })


trail_grid = pd.DataFrame(
    trail_rows
)


trail_ranked = (

    trail_grid

    .sort_values(
        "Mean_Net",
        ascending=False
    )

    .reset_index(drop=True)

)


print("\n" + "=" * 95)
print("TOP 25 MACRO-DISAGREEMENT TRAILING-STOP SETUPS")
print("=" * 95)


print(

    trail_ranked

    .head(25)

    .assign(
        Win_Rate=lambda x:
            x[
                "Win_Rate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 23. 70/30 TRAILING STOP ROBUSTNESS
# ================================================================

disagree = (

    disagree

    .sort_values("date")

    .reset_index(drop=True)

)


split_dis = int(
    len(disagree)
    *
    0.70
)


trail_train = disagree.iloc[
    :split_dis
]


trail_test = disagree.iloc[
    split_dis:
]


trail_train_rows = []


for activation in TRAIL_ACTIVATIONS:

    for trail in TRAIL_DISTANCES:

        for stop in HARD_STOPS:

            for final_exit in TRAIL_EXITS:


                pnl = trail_train.apply(

                    lambda row:

                        trailing_trade(

                            row,

                            row[
                                "Macro_Direction"
                            ],

                            activation,

                            trail,

                            stop,

                            final_exit

                        ),

                    axis=1

                ).dropna()


                if len(
                    pnl
                ) < 15:

                    continue


                trail_train_rows.append({

                    "Activation":
                        activation,

                    "Trail":
                        trail,

                    "Stop":
                        stop,

                    "Exit":
                        final_exit,

                    "Mean":
                        pnl.mean()

                })


trail_train_grid = pd.DataFrame(
    trail_train_rows
)


if not trail_train_grid.empty:


    trail_best = trail_train_grid.loc[

        trail_train_grid[
            "Mean"
        ].idxmax()

    ]


    trail_test_pnl = trail_test.apply(

        lambda row:

            trailing_trade(

                row,

                row[
                    "Macro_Direction"
                ],

                int(
                    trail_best[
                        "Activation"
                    ]
                ),

                int(
                    trail_best[
                        "Trail"
                    ]
                ),

                int(
                    trail_best[
                        "Stop"
                    ]
                ),

                int(
                    trail_best[
                        "Exit"
                    ]
                )

            ),

        axis=1

    ).dropna()


    print("\n" + "=" * 95)
    print("70/30 MACRO-DISAGREEMENT TRAILING TEST")
    print("=" * 95)


    print(
        "TRAIN selected:"
    )


    print(
        trail_best.to_string()
    )


    print(
        "\nTEST trades:",
        len(
            trail_test_pnl
        )
    )


    print(
        f"TEST mean:   "
        f"{trail_test_pnl.mean():.2f}"
    )


    print(
        f"TEST median: "
        f"{trail_test_pnl.median():.2f}"
    )


    print(
        f"TEST win:    "
        f"{(trail_test_pnl > 0).mean():.1%}"
    )


    print(
        f"TEST total:  "
        f"{trail_test_pnl.sum():.2f}"
    )


# ================================================================
# 24. DISAGREEMENT — SHORT HORIZON MFE / MAE
# ================================================================

disagreement_rows = []


for horizon in [
    5,
    10,
    15,
    30
]:


    mfe = disagree[
        f"Macro_MFE_T{horizon}"
    ]


    mae = disagree[
        f"Macro_MAE_T{horizon}"
    ]


    disagreement_rows.append({

        "Horizon":
            horizon,

        "N":
            len(disagree),

        "Mean_MFE":
            mfe.mean(),

        "Median_MFE":
            mfe.median(),

        "Mean_MAE":
            mae.mean(),

        "Median_MAE":
            mae.median(),

        "Pct_MFE_5":
            (
                mfe >= 5
            ).mean(),

        "Pct_MFE_10":
            (
                mfe >= 10
            ).mean(),

        "Pct_MFE_15":
            (
                mfe >= 15
            ).mean(),

        "Pct_MAE_Under5":
            (
                mae <= 5
            ).mean(),

        "Pct_MAE_Under10":
            (
                mae <= 10
            ).mean()

    })


disagreement_short = pd.DataFrame(
    disagreement_rows
)


print("\n" + "=" * 95)
print("MACRO-DISAGREEMENT SHORT-HORIZON EXCURSION")
print("=" * 95)


print(

    disagreement_short

    .assign(

        Pct_MFE_5=lambda x:
            x[
                "Pct_MFE_5"
            ] * 100,

        Pct_MFE_10=lambda x:
            x[
                "Pct_MFE_10"
            ] * 100,

        Pct_MFE_15=lambda x:
            x[
                "Pct_MFE_15"
            ] * 100,

        Pct_MAE_Under5=lambda x:
            x[
                "Pct_MAE_Under5"
            ] * 100,

        Pct_MAE_Under10=lambda x:
            x[
                "Pct_MAE_Under10"
            ] * 100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 25. MFE / MAE SCATTER — DISAGREEMENT
# ================================================================

plt.figure(
    figsize=(11, 7)
)


plt.scatter(

    disagree[
        "Macro_MAE_T15"
    ],

    disagree[
        "Macro_MFE_T15"
    ],

    alpha=0.7

)


plt.axline(

    (0, 0),

    slope=1,

    linestyle="--"

)


plt.axline(

    (0, 0),

    slope=2,

    linestyle=":"

)


plt.xlabel(
    "T15 MAE (pips)"
)


plt.ylabel(
    "T15 MFE (pips)"
)


plt.title(

    "Macro / Initial Price Disagreement\n"
    "T15 MFE versus MAE"

)


plt.grid(
    True,
    alpha=0.3
)


plt.show()


# ================================================================
# 26. SHORT-HORIZON MFE CURVES
# ================================================================

summary_plot = []


for group in [

    "AGREE",
    "DISAGREE"

]:


    subset = data[
        data[
            "Macro_Price_Group"
        ]
        ==
        group
    ]


    for horizon in HORIZONS:


        summary_plot.append({

            "Group":
                group,

            "Horizon":
                horizon,

            "Median_MFE":
                subset[
                    f"Macro_MFE_T{horizon}"
                ].median(),

            "Median_MAE":
                subset[
                    f"Macro_MAE_T{horizon}"
                ].median()

        })


summary_plot = pd.DataFrame(
    summary_plot
)


for group in [

    "AGREE",
    "DISAGREE"

]:


    subset = summary_plot[
        summary_plot[
            "Group"
        ]
        ==
        group
    ]


    plt.figure(
        figsize=(11, 6)
    )


    plt.plot(

        subset[
            "Horizon"
        ],

        subset[
            "Median_MFE"
        ],

        marker="o",

        label="Median MFE"

    )


    plt.plot(

        subset[
            "Horizon"
        ],

        subset[
            "Median_MAE"
        ],

        marker="o",

        label="Median MAE"

    )


    plt.xlabel(
        "Minutes after 08:30 close"
    )


    plt.ylabel(
        "Pips"
    )


    plt.title(
        f"{group}: MFE / MAE Development"
    )


    plt.legend()


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


# ================================================================
# 27. SAVE EVERYTHING
# ================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "master_event_data.csv",

    index=False

)


short_strategy.to_csv(

    f"{OUTPUT_DIR}/"
    "short_horizon_strategy_comparison.csv",

    index=False

)


clean_profile.to_csv(

    f"{OUTPUT_DIR}/"
    "clean_trade_profile.csv",

    index=False

)


alignment_profile.to_csv(

    f"{OUTPUT_DIR}/"
    "clean_trade_by_alignment.csv",

    index=False

)


clean_events.to_csv(

    f"{OUTPUT_DIR}/"
    "clean_t15_events.csv",

    index=False

)


bad_events.to_csv(

    f"{OUTPUT_DIR}/"
    "bad_t15_events.csv",

    index=False

)


immediate_pullback.to_csv(

    f"{OUTPUT_DIR}/"
    "immediate_fade_by_initial_move.csv",

    index=False

)


pullback_grid.to_csv(

    f"{OUTPUT_DIR}/"
    "extension_pullback_grid.csv",

    index=False

)


trail_grid.to_csv(

    f"{OUTPUT_DIR}/"
    "macro_disagreement_trailing_grid.csv",

    index=False

)


disagreement_short.to_csv(

    f"{OUTPUT_DIR}/"
    "disagreement_short_horizon_mfe_mae.csv",

    index=False

)


# ================================================================
# 28. FINAL
# ================================================================

print("\n" + "=" * 95)
print("MASTER POST-EVENT STUDY FINISHED")
print("=" * 95)


print(
    "Total events:",
    len(data)
)


print(
    "Macro disagreement events:",
    len(disagree)
)


print(
    "Clean T10 macro events:",
    data[
        "Clean_Macro_T10"
    ].sum()
)


print(
    "Clean T15 macro events:",
    data[
        "Clean_Macro_T15"
    ].sum()
)


print(
    "\nSaved to:"
)

print(
    OUTPUT_DIR
)


print(
"""

PLEASE SEND ME THESE OUTPUTS:

1. SHORT-HORIZON STRATEGY COMPARISON

2. CLEAN TRADE PROFILE

3. CLEAN TRADES BY ALIGNMENT

4. CLEAN T15 MACRO TRADES
   (if extremely long, send the first ~30)

5. BAD T15 MACRO TRADES

6. IMMEDIATE FADE BY INITIAL MOVE SIZE

7. TOP 25 EXTENSION -> PULLBACK SETUPS

8. 70/30 PULLBACK TEST

9. TOP 25 MACRO-DISAGREEMENT TRAILING-STOP SETUPS

10. 70/30 MACRO-DISAGREEMENT TRAILING TEST

11. MACRO-DISAGREEMENT SHORT-HORIZON EXCURSION


WHAT I CARE ABOUT MOST:

A) Does macro disagreement remain useful specifically at T5/T10?

B) Is T5 or T10 more robust than T15?

C) Do large 08:30 moves reliably retrace?

D) Is an "extend another X pips then fade" strategy better than
   immediately fading at the 08:30 close?

E) Can trailing stops improve the disagreement setup without
   destroying the win rate?

F) Do the 70/30 tests remain positive?

G) What macro characteristics distinguish the CLEAN trades
   from the BAD trades?

"""
)

In [ ]:
# =====================================================================
# NFP EUR/USD — ROBUST STRATEGY SEARCH
# =====================================================================
#
# PRIMARY HYPOTHESIS
#
#   Macro direction != first 08:30 candle direction
#                      ↓
#        trade back toward MACRO direction
#
#
# MAIN RESEARCH QUESTIONS
#
# 1. Is disagreement still useful after conditioning on alignment?
# 2. Are -2 and +1 especially useful?
# 3. Is T5 / T10 / T15 best?
# 4. Can +5/+10 favorable excursion be converted into realized P&L?
# 5. Does breakeven / partial profit / trailing help?
# 6. Does any rule survive TRUE expanding walk-forward testing?
#
#
# IMPORTANT
#
# All entries are referenced to the 08:30 close.
#
# Trading cost is deducted from every completed trade.
#
# Parameter selection for the final walk-forward test uses
# ONLY information available BEFORE each new NFP event.
#
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# =====================================================================
# 1. SETTINGS
# =====================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_robust_strategy_search"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30

MAX_MINUTE = 30

COST_PIPS = 3.0


# Need enough previous releases before we select rules.

MIN_HISTORY = 70


# Candidate horizons intentionally kept small.

HORIZONS = [
    5,
    10,
    15
]


# =====================================================================
# 2. LOAD MACRO
# =====================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


macro_cols = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_Surprise",
    "U3_USD_Surprise"

]


missing = [
    c
    for c in macro_cols
    if c not in macro_raw.columns
]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        macro_cols
    ]

    .dropna(
        subset=["date"]
    )

    .sort_values("date")

    .drop_duplicates(
        "date"
    )

    .reset_index(drop=True)

)


# =====================================================================
# 3. MACRO FEATURES
# =====================================================================

macro["NFP_Sign"] = np.sign(
    macro["NFP_Surprise"]
)


macro["AHE_Sign"] = np.sign(
    macro["AHE_Surprise"]
)


macro["U3_Sign"] = np.sign(
    macro["U3_USD_Surprise"]
)


macro["Alignment"] = (

      macro["NFP_Sign"]
    + macro["AHE_Sign"]
    + macro["U3_Sign"]

)


# Positive alignment = bullish USD
# Therefore EUR/USD direction should be negative.

macro["Macro_Direction"] = (

    -np.sign(
        macro["Alignment"]
    )

)


macro["Abs_Alignment"] = (
    macro["Alignment"].abs()
)


# =====================================================================
# 4. LOAD EUR/USD
# =====================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        "Datetime"
    )

    .reset_index(drop=True)

)


px = (

    prices

    .set_index("Datetime")

    .sort_index()

)


print("=" * 90)
print("INPUT")
print("=" * 90)

print(
    "Macro events:",
    len(macro)
)

print(
    "Price rows:",
    len(prices)
)


# =====================================================================
# 5. EVENT EXTRACTION
# =====================================================================

def extract_event(
    release_date
):


    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    pre_dt = (

        event_dt

        -

        pd.Timedelta(
            minutes=1
        )

    )


    if (
        event_dt not in px.index
        or
        pre_dt not in px.index
    ):

        return None


    pre_close = float(
        px.loc[
            pre_dt,
            "Close"
        ]
    )


    event = px.loc[
        event_dt
    ]


    event_open = float(
        event["Open"]
    )

    event_high = float(
        event["High"]
    )

    event_low = float(
        event["Low"]
    )

    event_close = float(
        event["Close"]
    )


    result = {

        "date":
            date,

        "Entry":
            event_close,

        "Pre_Close":
            pre_close

    }


    initial_move = (

        event_close
        -
        pre_close

    ) * 10000


    event_range = (

        event_high
        -
        event_low

    ) * 10000


    result[
        "Initial_Move"
    ] = initial_move


    result[
        "Abs_Initial_Move"
    ] = abs(
        initial_move
    )


    result[
        "Initial_Direction"
    ] = np.sign(
        initial_move
    )


    result[
        "Event_Range"
    ] = event_range


    if event_range > 0:

        result[
            "Efficiency"
        ] = (

            abs(
                initial_move
            )

            /

            event_range

        )


        result[
            "Close_Location"
        ] = (

            event_close
            -
            event_low

        ) / (

            event_high
            -
            event_low

        )


    else:

        result[
            "Efficiency"
        ] = np.nan

        result[
            "Close_Location"
        ] = np.nan


    for minute in range(
        1,
        MAX_MINUTE + 1
    ):


        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            for prefix in [
                "O",
                "H",
                "L",
                "C"
            ]:

                result[
                    f"{prefix}{minute}"
                ] = np.nan

            continue


        bar = px.loc[
            dt
        ]


        result[
            f"O{minute}"
        ] = float(
            bar["Open"]
        )


        result[
            f"H{minute}"
        ] = float(
            bar["High"]
        )


        result[
            f"L{minute}"
        ] = float(
            bar["Low"]
        )


        result[
            f"C{minute}"
        ] = float(
            bar["Close"]
        )


    return result


event_rows = []


for date in macro[
    "date"
]:

    r = extract_event(
        date
    )

    if r is not None:

        event_rows.append(
            r
        )


event_df = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    event_df,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values("date")

    .reset_index(drop=True)

)


# =====================================================================
# 6. MACRO / PRICE RELATIONSHIP
# =====================================================================

data[
    "Disagreement"
] = (

    (
        data[
            "Macro_Direction"
        ]
        !=
        0
    )

    &

    (
        data[
            "Initial_Direction"
        ]

        !=

        data[
            "Macro_Direction"
        ]
    )

)


data[
    "Agreement"
] = (

    (
        data[
            "Macro_Direction"
        ]
        !=
        0
    )

    &

    (
        data[
            "Initial_Direction"
        ]

        ==

        data[
            "Macro_Direction"
        ]
    )

)


print(
    "\nMatched:",
    len(data)
)


print(
    "Disagreement:",
    data[
        "Disagreement"
    ].sum()
)


# =====================================================================
# 7. SIMPLE TRADE RETURN
# =====================================================================

def close_return(
    row,
    direction,
    horizon,
    cost=COST_PIPS
):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return np.nan


    final = row[
        f"C{horizon}"
    ]


    if pd.isna(
        final
    ):

        return np.nan


    gross = (

        final
        -
        row[
            "Entry"
        ]

    ) * 10000 * direction


    return (
        gross
        -
        cost
    )


# =====================================================================
# 8. ALIGNMENT × DISAGREEMENT CROSS-TAB
# =====================================================================

cross_rows = []


for alignment in [
    -3,
    -2,
    -1,
    1,
    2,
    3
]:


    for disagree in [
        False,
        True
    ]:


        subset = data[

            (
                data[
                    "Alignment"
                ]
                ==
                alignment
            )

            &

            (
                data[
                    "Disagreement"
                ]
                ==
                disagree
            )

        ]


        for horizon in HORIZONS:


            pnl = subset.apply(

                lambda row:

                    close_return(

                        row,

                        row[
                            "Macro_Direction"
                        ],

                        horizon

                    ),

                axis=1

            ).dropna()


            if pnl.empty:

                continue


            cross_rows.append({

                "Alignment":
                    alignment,

                "Disagreement":
                    disagree,

                "Horizon":
                    horizon,

                "N":
                    len(pnl),

                "Mean":
                    pnl.mean(),

                "Median":
                    pnl.median(),

                "WinRate":
                    (
                        pnl > 0
                    ).mean()

            })


cross_tab = pd.DataFrame(
    cross_rows
)


print("\n" + "=" * 90)
print("ALIGNMENT x DISAGREEMENT")
print("=" * 90)


print(

    cross_tab

    .assign(
        WinRate=lambda x:
            x[
                "WinRate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# =====================================================================
# 9. TRADE MANAGEMENT SIMULATOR
# =====================================================================
#
# IMPORTANT:
#
# Uses conservative M1 assumptions.
#
# Existing stop is tested BEFORE new trailing information
# from the same candle is used.
#
#
# Modes:
#
# FIXED
# BE5
# BE10
# PARTIAL5
# PARTIAL10
# TRAIL10
#
# =====================================================================

def managed_trade(

    row,

    direction,

    horizon,

    mode,

    stop_pips,

    target_pips=None,

    partial_fraction=0.5,

    trail_distance=5

):


    if (
        pd.isna(direction)
        or
        direction == 0
    ):

        return np.nan


    entry = row[
        "Entry"
    ]


    remaining = 1.0

    realized = 0.0


    # -------------------------------------------------------------
    # Initial hard stop
    # -------------------------------------------------------------

    stop_level_pips = (
        -float(
            stop_pips
        )
    )


    best_favorable = 0.0


    partial_done = False

    be_active = False

    trail_active = False


    for minute in range(
        1,
        horizon + 1
    ):


        high = row[
            f"H{minute}"
        ]


        low = row[
            f"L{minute}"
        ]


        if (
            pd.isna(high)
            or
            pd.isna(low)
        ):

            continue


        # ---------------------------------------------------------
        # Express entire bar in trade-direction pips.
        #
        # bar_best  = most favorable price
        # bar_worst = most adverse price
        # ---------------------------------------------------------

        if direction > 0:

            bar_best = (

                high
                -
                entry

            ) * 10000


            bar_worst = (

                low
                -
                entry

            ) * 10000


        else:

            bar_best = (

                entry
                -
                low

            ) * 10000


            bar_worst = (

                entry
                -
                high

            ) * 10000


        # ---------------------------------------------------------
        # CHECK CURRENT STOP FIRST
        # ---------------------------------------------------------

        if bar_worst <= stop_level_pips:


            realized += (

                remaining

                *

                stop_level_pips

            )


            remaining = 0

            break


        # ---------------------------------------------------------
        # OPTIONAL FIXED TARGET
        # ---------------------------------------------------------

        if (
            target_pips is not None
            and
            bar_best >= target_pips
        ):


            realized += (

                remaining

                *

                target_pips

            )


            remaining = 0

            break


        # ---------------------------------------------------------
        # UPDATE FAVORABLE MOVE
        # ---------------------------------------------------------

        best_favorable = max(

            best_favorable,

            bar_best

        )


        # ---------------------------------------------------------
        # BREAKEVEN
        # ---------------------------------------------------------

        if mode == "BE5":

            if best_favorable >= 5:

                stop_level_pips = max(

                    stop_level_pips,

                    0

                )

                be_active = True


        elif mode == "BE10":

            if best_favorable >= 10:

                stop_level_pips = max(

                    stop_level_pips,

                    0

                )

                be_active = True


        # ---------------------------------------------------------
        # PARTIAL + BREAKEVEN
        # ---------------------------------------------------------

        elif mode == "PARTIAL5":

            if (
                not partial_done
                and
                best_favorable >= 5
            ):


                realized += (

                    partial_fraction
                    *
                    5

                )


                remaining -= (
                    partial_fraction
                )


                partial_done = True


                stop_level_pips = max(

                    stop_level_pips,

                    0

                )


        elif mode == "PARTIAL10":

            if (
                not partial_done
                and
                best_favorable >= 10
            ):


                realized += (

                    partial_fraction
                    *
                    10

                )


                remaining -= (
                    partial_fraction
                )


                partial_done = True


                stop_level_pips = max(

                    stop_level_pips,

                    0

                )


        # ---------------------------------------------------------
        # TRAIL AFTER +10
        # ---------------------------------------------------------

        elif mode == "TRAIL10":

            if best_favorable >= 10:


                trail_active = True


                new_stop = (

                    best_favorable

                    -

                    trail_distance

                )


                stop_level_pips = max(

                    stop_level_pips,

                    new_stop

                )


    # -------------------------------------------------------------
    # TIME EXIT
    # -------------------------------------------------------------

    if remaining > 0:


        final = row[
            f"C{horizon}"
        ]


        if pd.isna(
            final
        ):

            return np.nan


        final_pips = (

            final
            -
            entry

        ) * 10000 * direction


        realized += (

            remaining

            *

            final_pips

        )


    # Deduct cost once per original trade.

    return (

        realized

        -
        COST_PIPS

    )


# =====================================================================
# 10. SMALL, INTENTIONALLY RESTRICTED STRATEGY FAMILY
# =====================================================================
#
# We do NOT test thousands of combinations.
#
# =====================================================================

strategy_specs = []


FILTERS = [

    "ALL_DISAGREE",

    "DISAGREE_MINUS2_PLUS1",

    "DISAGREE_ABS1_2",

    "DISAGREE_EFF_LT_070",

    "DISAGREE_MOVE_LT20",

    "DISAGREE_MOVE_10_30"

]


MODES = [

    "FIXED",
    "BE5",
    "BE10",
    "PARTIAL5",
    "PARTIAL10",
    "TRAIL10"

]


STOPS = [
    15,
    20,
    30
]


TARGETS = [
    None,
    10,
    15,
    20
]


for filter_name in FILTERS:

    for horizon in HORIZONS:

        for mode in MODES:

            for stop in STOPS:


                # trailing / partial / BE generally use time exit
                # rather than fixed target.

                if mode == "FIXED":

                    candidate_targets = TARGETS

                else:

                    candidate_targets = [
                        None
                    ]


                for target in candidate_targets:


                    strategy_specs.append({

                        "Filter":
                            filter_name,

                        "Horizon":
                            horizon,

                        "Mode":
                            mode,

                        "Stop":
                            stop,

                        "Target":
                            target

                    })


strategy_specs = pd.DataFrame(
    strategy_specs
)


# =====================================================================
# 11. FILTER FUNCTION
# =====================================================================

def strategy_mask(
    df,
    filter_name
):


    base = (
        df[
            "Disagreement"
        ]
        ==
        True
    )


    if filter_name == "ALL_DISAGREE":

        return base


    if filter_name == "DISAGREE_MINUS2_PLUS1":

        return (

            base

            &

            df[
                "Alignment"
            ].isin(
                [
                    -2,
                    1
                ]
            )

        )


    if filter_name == "DISAGREE_ABS1_2":

        return (

            base

            &

            df[
                "Abs_Alignment"
            ].isin(
                [
                    1,
                    2
                ]
            )

        )


    if filter_name == "DISAGREE_EFF_LT_070":

        return (

            base

            &

            (
                df[
                    "Efficiency"
                ]
                <
                0.70
            )

        )


    if filter_name == "DISAGREE_MOVE_LT20":

        return (

            base

            &

            (
                df[
                    "Abs_Initial_Move"
                ]
                <
                20
            )

        )


    if filter_name == "DISAGREE_MOVE_10_30":

        return (

            base

            &

            (
                df[
                    "Abs_Initial_Move"
                ]
                >=
                10
            )

            &

            (
                df[
                    "Abs_Initial_Move"
                ]
                <
                30
            )

        )


    raise ValueError(
        filter_name
    )


# =====================================================================
# 12. EVALUATE ONE STRATEGY
# =====================================================================

def evaluate_spec(
    df,
    spec
):


    mask = strategy_mask(

        df,

        spec[
            "Filter"
        ]

    )


    subset = df[
        mask
    ]


    pnl = []


    for _, row in subset.iterrows():


        result = managed_trade(

            row=row,

            direction=row[
                "Macro_Direction"
            ],

            horizon=int(
                spec[
                    "Horizon"
                ]
            ),

            mode=spec[
                "Mode"
            ],

            stop_pips=float(
                spec[
                    "Stop"
                ]
            ),

            target_pips=(

                None

                if pd.isna(
                    spec[
                        "Target"
                    ]
                )

                else float(
                    spec[
                        "Target"
                    ]
                )

            )

        )


        if pd.notna(
            result
        ):

            pnl.append(
                result
            )


    pnl = np.array(
        pnl,
        dtype=float
    )


    if len(pnl) == 0:

        return None


    return pnl


# =====================================================================
# 13. FULL SAMPLE — DESCRIPTIVE ONLY
# =====================================================================

descriptive_rows = []


for _, spec in strategy_specs.iterrows():


    pnl = evaluate_spec(
        data,
        spec
    )


    if (
        pnl is None
        or
        len(pnl) < 15
    ):

        continue


    descriptive_rows.append({

        **spec.to_dict(),

        "Trades":
            len(pnl),

        "Mean":
            np.mean(pnl),

        "Median":
            np.median(pnl),

        "WinRate":
            np.mean(
                pnl > 0
            ),

        "Std":
            np.std(
                pnl,
                ddof=1
            ),

        "Total":
            np.sum(pnl)

    })


descriptive = pd.DataFrame(
    descriptive_rows
)


descriptive = descriptive.sort_values(

    "Mean",

    ascending=False

)


print("\n" + "=" * 90)
print("TOP DESCRIPTIVE STRATEGIES — NOT OOS")
print("=" * 90)


print(

    descriptive

    .head(25)

    .assign(
        WinRate=lambda x:
            x[
                "WinRate"
            ] * 100
    )

    .round(2)

    .to_string(
        index=False
    )

)


# =====================================================================
# 14. ROBUST TRAINING SCORE
# =====================================================================
#
# Instead of selecting highest raw mean:
#
# score =
#
#   mean return
#   -
#   0.75 * standard error
#
#
# This penalizes:
#
# - small samples
# - high variance
# - one giant winner dominating the mean
#
# =====================================================================

def robust_score(
    pnl
):


    pnl = np.asarray(
        pnl
    )


    if len(pnl) < 15:

        return -np.inf


    mean = np.mean(
        pnl
    )


    std = np.std(
        pnl,
        ddof=1
    )


    se = (

        std

        /

        np.sqrt(
            len(pnl)
        )

    )


    return (

        mean

        -

        0.75
        *
        se

    )


# =====================================================================
# 15. EXPANDING WALK-FORWARD STRATEGY SELECTION
# =====================================================================
#
# For each event after MIN_HISTORY:
#
# 1. Consider ONLY older releases.
# 2. Evaluate every candidate strategy.
# 3. Select strategy with best robust score.
# 4. Check whether CURRENT event qualifies.
# 5. If yes, execute the selected rule.
#
# =====================================================================

wf_rows = []


for i in range(
    MIN_HISTORY,
    len(data)
):


    history = data.iloc[
        :i
    ].copy()


    current = data.iloc[
        [i]
    ].copy()


    candidates = []


    for _, spec in strategy_specs.iterrows():


        pnl = evaluate_spec(

            history,

            spec

        )


        if (
            pnl is None
            or
            len(pnl) < 15
        ):

            continue


        score = robust_score(
            pnl
        )


        candidates.append({

            **spec.to_dict(),

            "Historical_Trades":
                len(pnl),

            "Historical_Mean":
                np.mean(
                    pnl
                ),

            "Historical_Median":
                np.median(
                    pnl
                ),

            "Historical_WinRate":
                np.mean(
                    pnl > 0
                ),

            "Score":
                score

        })


    candidate_df = pd.DataFrame(
        candidates
    )


    if candidate_df.empty:

        continue


    chosen = candidate_df.loc[

        candidate_df[
            "Score"
        ].idxmax()

    ]


    current_mask = strategy_mask(

        current,

        chosen[
            "Filter"
        ]

    )


    # Selected strategy says:
    #
    # no trade today.

    if not current_mask.iloc[0]:

        wf_rows.append({

            "date":
                current[
                    "date"
                ].iloc[0],

            "Trade":
                False,

            "PnL":
                0.0,

            "Filter":
                chosen[
                    "Filter"
                ],

            "Horizon":
                chosen[
                    "Horizon"
                ],

            "Mode":
                chosen[
                    "Mode"
                ],

            "Stop":
                chosen[
                    "Stop"
                ],

            "Target":
                chosen[
                    "Target"
                ],

            "Historical_Score":
                chosen[
                    "Score"
                ]

        })


        continue


    row = current.iloc[0]


    result = managed_trade(

        row=row,

        direction=row[
            "Macro_Direction"
        ],

        horizon=int(
            chosen[
                "Horizon"
            ]
        ),

        mode=chosen[
            "Mode"
        ],

        stop_pips=float(
            chosen[
                "Stop"
            ]
        ),

        target_pips=(

            None

            if pd.isna(
                chosen[
                    "Target"
                ]
            )

            else float(
                chosen[
                    "Target"
                ]
            )

        )

    )


    wf_rows.append({

        "date":
            row[
                "date"
            ],

        "Trade":
            True,

        "PnL":
            result,

        "Alignment":
            row[
                "Alignment"
            ],

        "Initial_Move":
            row[
                "Initial_Move"
            ],

        "Efficiency":
            row[
                "Efficiency"
            ],

        "Filter":
            chosen[
                "Filter"
            ],

        "Horizon":
            chosen[
                "Horizon"
            ],

        "Mode":
            chosen[
                "Mode"
            ],

        "Stop":
            chosen[
                "Stop"
            ],

        "Target":
            chosen[
                "Target"
            ],

        "Historical_Score":
            chosen[
                "Score"
            ]

    })


walkforward = pd.DataFrame(
    wf_rows
)


trades = walkforward[

    walkforward[
        "Trade"
    ]
    ==
    True

].copy()


# =====================================================================
# 16. FINAL WALK-FORWARD RESULTS
# =====================================================================

print("\n" + "=" * 90)
print("FINAL WALK-FORWARD OOS RESULT")
print("=" * 90)


print(
    "Evaluation NFP releases:",
    len(
        walkforward
    )
)


print(
    "Trades actually taken:",
    len(
        trades
    )
)


if len(trades) > 0:


    print(
        f"Trade frequency: "
        f"{len(trades) / len(walkforward):.1%}"
    )


    print(
        f"Mean net:        "
        f"{trades['PnL'].mean():.2f} pips"
    )


    print(
        f"Median net:      "
        f"{trades['PnL'].median():.2f} pips"
    )


    print(
        f"Win rate:        "
        f"{(trades['PnL'] > 0).mean():.1%}"
    )


    print(
        f"Total net:       "
        f"{trades['PnL'].sum():.2f} pips"
    )


    print(
        f"Average winner:  "
        f"{trades.loc[trades['PnL'] > 0, 'PnL'].mean():.2f}"
    )


    print(
        f"Average loser:   "
        f"{trades.loc[trades['PnL'] < 0, 'PnL'].mean():.2f}"
    )


# =====================================================================
# 17. OOS MAX DRAWDOWN
# =====================================================================

if len(trades) > 0:


    trades[
        "Equity"
    ] = (

        trades[
            "PnL"
        ]

        .cumsum()

    )


    trades[
        "Peak"
    ] = (

        trades[
            "Equity"
        ]

        .cummax()

    )


    trades[
        "Drawdown"
    ] = (

        trades[
            "Equity"
        ]

        -

        trades[
            "Peak"
        ]

    )


    max_dd = trades[
        "Drawdown"
    ].min()


    print(
        f"Maximum drawdown:"
        f" {max_dd:.2f} pips"
    )


# =====================================================================
# 18. BOOTSTRAP CONFIDENCE INTERVAL
# =====================================================================
#
# This is NOT a replacement for more data.
#
# It gives us a sense of uncertainty around OOS mean expectancy.
#
# =====================================================================

if len(trades) >= 10:


    rng = np.random.default_rng(
        12345
    )


    values = trades[
        "PnL"
    ].values


    boot_means = []


    for _ in range(
        10000
    ):


        sample = rng.choice(

            values,

            size=len(values),

            replace=True

        )


        boot_means.append(

            np.mean(
                sample
            )

        )


    low = np.percentile(
        boot_means,
        2.5
    )


    high = np.percentile(
        boot_means,
        97.5
    )


    probability_positive = np.mean(

        np.array(
            boot_means
        )
        >
        0

    )


    print(
        f"Bootstrap 95% CI:"
        f" [{low:.2f}, {high:.2f}] pips/trade"
    )


    print(
        f"Bootstrap P(mean > 0):"
        f" {probability_positive:.1%}"
    )


# =====================================================================
# 19. WHICH RULES DID WALK-FORWARD ACTUALLY CHOOSE?
# =====================================================================

print("\n" + "=" * 90)
print("WALK-FORWARD RULE SELECTION")
print("=" * 90)


if not walkforward.empty:


    print(
        "\nFilters:"
    )


    print(

        walkforward[
            "Filter"
        ]

        .value_counts()

        .to_string()

    )


    print(
        "\nHorizons:"
    )


    print(

        walkforward[
            "Horizon"
        ]

        .value_counts()

        .sort_index()

        .to_string()

    )


    print(
        "\nManagement modes:"
    )


    print(

        walkforward[
            "Mode"
        ]

        .value_counts()

        .to_string()

    )


# =====================================================================
# 20. OOS RESULTS BY ALIGNMENT
# =====================================================================

if len(trades) > 0:


    alignment_oos = (

        trades

        .groupby(
            "Alignment"
        )

        .agg(

            Trades=(
                "PnL",
                "count"
            ),

            Mean=(
                "PnL",
                "mean"
            ),

            Median=(
                "PnL",
                "median"
            ),

            WinRate=(

                "PnL",

                lambda x:
                    (
                        x > 0
                    ).mean()

            ),

            Total=(
                "PnL",
                "sum"
            )

        )

        .reset_index()

    )


    print("\n" + "=" * 90)
    print("OOS RESULTS BY ALIGNMENT")
    print("=" * 90)


    print(

        alignment_oos

        .assign(
            WinRate=lambda x:
                x[
                    "WinRate"
                ]
                *
                100
        )

        .round(2)

        .to_string(
            index=False
        )

    )


# =====================================================================
# 21. RECENT OOS TRADES
# =====================================================================

if len(trades) > 0:


    print("\n" + "=" * 90)
    print("LAST 25 OOS TRADES")
    print("=" * 90)


    columns = [

        "date",
        "PnL",
        "Alignment",
        "Initial_Move",
        "Efficiency",
        "Filter",
        "Horizon",
        "Mode",
        "Stop",
        "Target"

    ]


    print(

        trades[
            columns
        ]

        .tail(25)

        .round(2)

        .to_string(
            index=False
        )

    )


# =====================================================================
# 22. EQUITY CURVE
# =====================================================================

if len(trades) > 0:


    plt.figure(
        figsize=(14, 7)
    )


    plt.plot(

        trades[
            "date"
        ],

        trades[
            "Equity"
        ],

        marker="o"

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "NFP release date"
    )


    plt.ylabel(
        "Cumulative OOS net pips"
    )


    plt.title(

        "NFP Strategy — "
        "Expanding Walk-Forward OOS Equity"

    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.show()


# =====================================================================
# 23. SAVE OUTPUTS
# =====================================================================

data.to_csv(

    f"{OUTPUT_DIR}/"
    "event_master.csv",

    index=False

)


cross_tab.to_csv(

    f"{OUTPUT_DIR}/"
    "alignment_disagreement_cross_tab.csv",

    index=False

)


descriptive.to_csv(

    f"{OUTPUT_DIR}/"
    "descriptive_strategy_search.csv",

    index=False

)


walkforward.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_all_releases.csv",

    index=False

)


trades.to_csv(

    f"{OUTPUT_DIR}/"
    "walkforward_trades.csv",

    index=False

)


# =====================================================================
# 24. FINAL MESSAGE
# =====================================================================

print("\n" + "=" * 90)
print("RESEARCH FINISHED")
print("=" * 90)


print(
    "Saved to:"
)

print(
    OUTPUT_DIR
)


print(
"""

SEND ME THESE OUTPUTS:

1. ALIGNMENT x DISAGREEMENT

2. TOP DESCRIPTIVE STRATEGIES — NOT OOS
   (top 15 is enough)

3. FINAL WALK-FORWARD OOS RESULT

4. WALK-FORWARD RULE SELECTION

5. OOS RESULTS BY ALIGNMENT

6. LAST 25 OOS TRADES


HOW I WILL JUDGE IT:

VERY PROMISING:
    OOS mean > +3 pips
    OOS median > 0
    positive total P&L
    tolerable drawdown
    bootstrap interval mostly above zero

INTERESTING BUT UNPROVEN:
    OOS mean roughly +1 to +3
    median positive
    CI crosses zero

NOT ROBUST:
    OOS mean <= 0
    or profits depend on one/two huge winners

Do NOT judge it from the descriptive top strategies.

The FINAL WALK-FORWARD OOS RESULT is the important part.

"""
)

In [ ]:
# ================================================================
# EUR/USD NFP OVERLAY — FIXED VERSION
#
# NFP BEAT vs NFP NO-BEAT
# 08:30 OPEN -> T15
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# ================================================================
# 1. SETTINGS
# ================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


EVENT_HOUR = 8
EVENT_MINUTE = 30

MAX_MINUTE = 15


# ================================================================
# 2. LOAD MACRO DATA
# ================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


# IMPORTANT:
# Keep ONLY the macro columns needed here.
# This prevents collisions with old T0/T1/T15 columns.

macro = macro_raw[
    [
        "date",
        "NFP_Actual",
        "NFP_Forecast",
        "NFP_Surprise"
    ]
].copy()


macro = (

    macro

    .dropna(
        subset=[
            "date",
            "NFP_Actual",
            "NFP_Forecast"
        ]
    )

    .sort_values("date")

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(drop=True)

)


# ================================================================
# 3. DEFINE NFP GROUP
# ================================================================

macro["NFP_Group"] = np.where(

    macro["NFP_Actual"]
    >
    macro["NFP_Forecast"],

    "NFP_BEAT",

    "NFP_NO_BEAT"

)


print("=" * 80)
print("NFP GROUP COUNTS")
print("=" * 80)


print(
    macro[
        "NFP_Group"
    ]
    .value_counts()
    .to_string()
)


# ================================================================
# 4. LOAD PRICE DATA
# ================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values("Datetime")

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(drop=True)

)


px = (

    prices

    .set_index("Datetime")

    .sort_index()

)


# ================================================================
# 5. EXTRACT PATH
# ================================================================
#
# IMPORTANT:
#
# Baseline = 08:30 OPEN
#
# Overlay_T0 = 08:30 CLOSE - 08:30 OPEN
#
# Overlay_T1 = 08:31 CLOSE - 08:30 OPEN
#
# ...
#
# Overlay_T15 = 08:45 CLOSE - 08:30 OPEN
#
# ================================================================

def extract_path(
    release_date
):

    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    if event_dt not in px.index:

        return None


    event_open = float(

        px.loc[
            event_dt,
            "Open"
        ]

    )


    record = {

        "date":
            date,

        "Overlay_Event_Open":
            event_open

    }


    for minute in range(
        0,
        MAX_MINUTE + 1
    ):


        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            record[
                f"Overlay_T{minute}"
            ] = np.nan

            continue


        close = float(

            px.loc[
                dt,
                "Close"
            ]

        )


        record[
            f"Overlay_T{minute}"
        ] = (

            close

            -

            event_open

        ) * 10000


    return record


# ================================================================
# 6. BUILD PATH DATA
# ================================================================

rows = []


for d in macro[
    "date"
]:

    r = extract_path(
        d
    )


    if r is not None:

        rows.append(
            r
        )


paths = pd.DataFrame(
    rows
)


data = pd.merge(

    macro,

    paths,

    on="date",

    how="inner",

    validate="one_to_one"

)


print(
    "\nMatched releases:",
    len(data)
)


print(
    "\nColumns created:"
)


print(
    [
        c
        for c in data.columns
        if c.startswith(
            "Overlay_T"
        )
    ]
)


# ================================================================
# 7. SUMMARY TABLE
# ================================================================

summary_rows = []


for group in [
    "NFP_BEAT",
    "NFP_NO_BEAT"
]:


    subset = data[

        data[
            "NFP_Group"
        ]
        ==
        group

    ]


    for minute in [
        0,
        1,
        5,
        10,
        15
    ]:


        vals = subset[
            f"Overlay_T{minute}"
        ].dropna()


        summary_rows.append({

            "Group":
                group,

            "Minute":
                minute,

            "N":
                len(vals),

            "Mean_Pips":
                vals.mean(),

            "Median_Pips":
                vals.median(),

            "Pct_EURUSD_Up":
                (
                    vals > 0
                ).mean()

        })


summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 80)
print("NFP BEAT vs NO-BEAT SUMMARY")
print("=" * 80)


print(

    summary

    .assign(

        Pct_EURUSD_Up=lambda x:
            x[
                "Pct_EURUSD_Up"
            ]
            *
            100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# ================================================================
# 8. PREPARE GROUPS
# ================================================================

beat = data[

    data[
        "NFP_Group"
    ]
    ==
    "NFP_BEAT"

].copy()


no_beat = data[

    data[
        "NFP_Group"
    ]
    ==
    "NFP_NO_BEAT"

].copy()


minutes = np.arange(
    0,
    MAX_MINUTE + 1
)


# ================================================================
# 9. NFP BEAT OVERLAY
# ================================================================

plt.figure(
    figsize=(14, 8)
)


for _, row in beat.iterrows():


    path = np.array([

        row[
            f"Overlay_T{i}"
        ]

        for i in minutes

    ])


    plt.plot(

        minutes,

        path,

        alpha=0.13,

        linewidth=1

    )


beat_mean = np.array([

    beat[
        f"Overlay_T{i}"
    ].mean()

    for i in minutes

])


beat_median = np.array([

    beat[
        f"Overlay_T{i}"
    ].median()

    for i in minutes

])


plt.plot(

    minutes,

    beat_mean,

    linewidth=4,

    label="Mean"

)


plt.plot(

    minutes,

    beat_median,

    linewidth=3,

    linestyle="--",

    label="Median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Minutes after 08:30"
)


plt.ylabel(
    "EUR/USD displacement from 08:30 open (pips)"
)


plt.title(

    f"NFP BEAT — EUR/USD Overlay\n"
    f"08:30 Open → T15 | N = {len(beat)}"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 10. NFP NO-BEAT OVERLAY
# ================================================================

plt.figure(
    figsize=(14, 8)
)


for _, row in no_beat.iterrows():


    path = np.array([

        row[
            f"Overlay_T{i}"
        ]

        for i in minutes

    ])


    plt.plot(

        minutes,

        path,

        alpha=0.13,

        linewidth=1

    )


no_beat_mean = np.array([

    no_beat[
        f"Overlay_T{i}"
    ].mean()

    for i in minutes

])


no_beat_median = np.array([

    no_beat[
        f"Overlay_T{i}"
    ].median()

    for i in minutes

])


plt.plot(

    minutes,

    no_beat_mean,

    linewidth=4,

    label="Mean"

)


plt.plot(

    minutes,

    no_beat_median,

    linewidth=3,

    linestyle="--",

    label="Median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Minutes after 08:30"
)


plt.ylabel(
    "EUR/USD displacement from 08:30 open (pips)"
)


plt.title(

    f"NFP NO-BEAT — EUR/USD Overlay\n"
    f"08:30 Open → T15 | N = {len(no_beat)}"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


# ================================================================
# 11. DIRECT COMPARISON
# ================================================================

plt.figure(
    figsize=(14, 8)
)


plt.plot(

    minutes,

    beat_mean,

    linewidth=4,

    label="NFP beat — mean"

)


plt.plot(

    minutes,

    no_beat_mean,

    linewidth=4,

    label="NFP no-beat — mean"

)


plt.plot(

    minutes,

    beat_median,

    linewidth=2,

    linestyle="--",

    label="NFP beat — median"

)


plt.plot(

    minutes,

    no_beat_median,

    linewidth=2,

    linestyle="--",

    label="NFP no-beat — median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Minutes after 08:30"
)


plt.ylabel(
    "EUR/USD displacement from 08:30 open (pips)"
)


plt.title(

    "EUR/USD NFP Reaction\n"
    "NFP Beat vs NFP No-Beat"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.show()


print(
"""
DONE.

The important visual is the THIRD chart.

It directly compares:

NFP BEAT mean/median
vs
NFP NO-BEAT mean/median

from the actual 08:30 OPEN through T15.

Paste the summary table back to me if you want me to interpret
what the chart is showing quantitatively.
"""
)

In [ ]:
# =====================================================================
# EUR/USD NFP OVERLAY + PIP RETURN DISTRIBUTION STUDY
# =====================================================================
#
# FOUR MACRO GROUPS
# -----------------
#
# 1. NFP BEAT
#
#       NFP Actual > Forecast
#
#
# 2. NFP BEAT + ALL THREE USD BULLISH
#
#       NFP Actual > Forecast
#       AHE Actual > Forecast
#       U3 Actual  < Forecast
#
#
# 3. NFP MISS
#
#       NFP Actual < Forecast
#
#
# 4. NFP MISS + ALL THREE USD BEARISH
#
#       NFP Actual < Forecast
#       AHE Actual < Forecast
#       U3 Actual  > Forecast
#
#
# ================================================================
# PRICE BASELINES
# ================================================================
#
# FIRST MINUTE:
#
#       08:30 OPEN = 0
#       endpoint = 08:30 CLOSE
#
#
# ALL OTHER HORIZONS:
#
#       08:30 CLOSE = 0
#
#       POST T5  = 08:35 close - 08:30 close
#       POST T10 = 08:40 close - 08:30 close
#       POST T15 = 08:45 close - 08:30 close
#       POST T30 = 09:00 close - 08:30 close
#       POST T45 = 09:15 close - 08:30 close
#       POST T60 = 09:30 close - 08:30 close
#
#
# ================================================================
# OUTPUTS
# ================================================================
#
# For the FIRST MINUTE:
#
#   A. 4-panel price overlay
#   B. mean/median comparison
#   C. 4-panel pip-return distributions
#
#
# For T5 / T10 / T15 / T30 / T45 / T60:
#
#   A. 4-panel price overlays from 08:30 CLOSE
#   B. mean path comparison
#   C. median path comparison
#   D. 4-panel distributions of terminal pip returns
#
#
# Also prints numerical summary statistics.
#
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive


# =====================================================================
# 1. SETTINGS
# =====================================================================

drive.mount(
    "/content/drive",
    force_remount=True
)


PRICE_PATH = (
    "/content/drive/MyDrive/"
    "filtered_eurusd_nfp_events.csv"
)


MACRO_PATH = (
    "/content/drive/MyDrive/"
    "nfp_three_factor_results/"
    "nfp_three_factor_master.csv"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "nfp_overlay_distribution_study"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


EVENT_HOUR = 8
EVENT_MINUTE = 30


POST_HORIZONS = [
    5,
    10,
    15,
    30,
    45,
    60
]


MAX_POST_HORIZON = max(
    POST_HORIZONS
)


# =====================================================================
# 2. LOAD MACRO DATA
# =====================================================================

macro_raw = pd.read_csv(
    MACRO_PATH
)


macro_raw["date"] = pd.to_datetime(
    macro_raw["date"],
    errors="coerce"
)


required_columns = [

    "date",

    "NFP_Actual",
    "NFP_Forecast",
    "NFP_Surprise",

    "AHE_Actual",
    "AHE_Forecast",
    "AHE_Surprise",

    "U3_Actual",
    "U3_Forecast",
    "U3_USD_Surprise"

]


missing = [

    col
    for col in required_columns
    if col not in macro_raw.columns

]


if missing:

    raise ValueError(
        f"Missing macro columns: {missing}"
    )


macro = (

    macro_raw[
        required_columns
    ]

    .copy()

    .dropna(
        subset=[
            "date",
            "NFP_Actual",
            "NFP_Forecast",
            "AHE_Actual",
            "AHE_Forecast",
            "U3_Actual",
            "U3_Forecast"
        ]
    )

    .sort_values(
        "date"
    )

    .drop_duplicates(
        subset="date",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


print("=" * 100)
print("MACRO INPUT")
print("=" * 100)

print(
    "Usable releases:",
    len(macro)
)


# =====================================================================
# 3. DEFINE MACRO GROUPS
# =====================================================================

# ---------------------------------------------------------------------
# NFP
# ---------------------------------------------------------------------

macro["NFP_Beat"] = (

    macro["NFP_Actual"]
    >
    macro["NFP_Forecast"]

)


macro["NFP_Miss"] = (

    macro["NFP_Actual"]
    <
    macro["NFP_Forecast"]

)


macro["NFP_Inline"] = (

    macro["NFP_Actual"]
    ==
    macro["NFP_Forecast"]

)


# ---------------------------------------------------------------------
# AHE
# ---------------------------------------------------------------------

macro["AHE_Beat"] = (

    macro["AHE_Actual"]
    >
    macro["AHE_Forecast"]

)


macro["AHE_Miss"] = (

    macro["AHE_Actual"]
    <
    macro["AHE_Forecast"]

)


# ---------------------------------------------------------------------
# UNEMPLOYMENT
#
# Lower than expected unemployment = USD bullish
#
# Higher than expected unemployment = USD bearish
# ---------------------------------------------------------------------

macro["U3_USD_Bullish"] = (

    macro["U3_Actual"]
    <
    macro["U3_Forecast"]

)


macro["U3_USD_Bearish"] = (

    macro["U3_Actual"]
    >
    macro["U3_Forecast"]

)


# ---------------------------------------------------------------------
# FULL THREE-FACTOR ALIGNMENT
# ---------------------------------------------------------------------

macro["All_3_USD_Bullish"] = (

    macro["NFP_Beat"]

    &

    macro["AHE_Beat"]

    &

    macro["U3_USD_Bullish"]

)


macro["All_3_USD_Bearish"] = (

    macro["NFP_Miss"]

    &

    macro["AHE_Miss"]

    &

    macro["U3_USD_Bearish"]

)


print("\n" + "=" * 100)
print("MACRO GROUP COUNTS")
print("=" * 100)


print(
    "NFP Beat:",
    macro["NFP_Beat"].sum()
)


print(
    "NFP Beat + all 3 USD bullish:",
    macro["All_3_USD_Bullish"].sum()
)


print(
    "NFP Miss:",
    macro["NFP_Miss"].sum()
)


print(
    "NFP Miss + all 3 USD bearish:",
    macro["All_3_USD_Bearish"].sum()
)


print(
    "NFP exactly inline:",
    macro["NFP_Inline"].sum()
)


# =====================================================================
# 4. LOAD EUR/USD PRICE DATA
# =====================================================================

prices = pd.read_csv(
    PRICE_PATH
)


prices["Datetime"] = pd.to_datetime(
    prices["Datetime"],
    errors="coerce"
)


for col in [
    "Open",
    "High",
    "Low",
    "Close"
]:

    prices[col] = pd.to_numeric(
        prices[col],
        errors="coerce"
    )


prices = (

    prices

    .dropna(
        subset=[
            "Datetime",
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    .sort_values(
        "Datetime"
    )

    .drop_duplicates(
        subset="Datetime",
        keep="first"
    )

    .reset_index(
        drop=True
    )

)


px = (

    prices

    .set_index(
        "Datetime"
    )

    .sort_index()

)


print("\n" + "=" * 100)
print("PRICE INPUT")
print("=" * 100)


print(
    "Rows:",
    len(prices)
)


print(
    "Range:",
    prices["Datetime"].min(),
    "->",
    prices["Datetime"].max()
)


# =====================================================================
# 5. EXTRACT EVENT PRICE PATH
# =====================================================================
#
# FIRST-MINUTE PATH:
#
#   FIRST_0 = 08:30 open = 0
#   FIRST_1 = 08:30 close relative to 08:30 open
#
#
# POST-FIRST-MINUTE PATH:
#
#   POST_0 = 08:30 close = 0
#
#   POST_1 = 08:31 close relative to 08:30 close
#   POST_2 = 08:32 close relative to 08:30 close
#   ...
#   POST_60 = 09:30 close relative to 08:30 close
#
# =====================================================================

def extract_event_path(
    release_date
):


    date = pd.Timestamp(
        release_date
    ).normalize()


    event_dt = (

        date

        +

        pd.Timedelta(
            hours=EVENT_HOUR,
            minutes=EVENT_MINUTE
        )

    )


    if event_dt not in px.index:

        return None


    event_bar = px.loc[
        event_dt
    ]


    event_open = float(
        event_bar["Open"]
    )


    event_high = float(
        event_bar["High"]
    )


    event_low = float(
        event_bar["Low"]
    )


    event_close = float(
        event_bar["Close"]
    )


    record = {

        "date":
            date,

        "Event_Open":
            event_open,

        "Event_High":
            event_high,

        "Event_Low":
            event_low,

        "Event_Close":
            event_close,

        # ---------------------------------------------------------
        # FIRST MINUTE
        # ---------------------------------------------------------

        "FIRST_0":
            0.0,

        "FIRST_1":

            (
                event_close
                -
                event_open
            )
            *
            10000,

        # ---------------------------------------------------------
        # POST-FIRST-MINUTE
        # ---------------------------------------------------------

        "POST_0":
            0.0

    }


    # -----------------------------------------------------------------
    # Additional event-candle statistics
    # -----------------------------------------------------------------

    record[
        "First_Minute_Range_Pips"
    ] = (

        event_high
        -
        event_low

    ) * 10000


    record[
        "First_Minute_Body_Pips"
    ] = (

        event_close
        -
        event_open

    ) * 10000


    # -----------------------------------------------------------------
    # SUBSEQUENT MINUTES
    # -----------------------------------------------------------------

    for minute in range(
        1,
        MAX_POST_HORIZON + 1
    ):


        dt = (

            event_dt

            +

            pd.Timedelta(
                minutes=minute
            )

        )


        if dt not in px.index:

            record[
                f"POST_{minute}"
            ] = np.nan

            continue


        close = float(

            px.loc[
                dt,
                "Close"
            ]

        )


        record[
            f"POST_{minute}"
        ] = (

            close
            -
            event_close

        ) * 10000


    return record


# =====================================================================
# 6. EXTRACT ALL RELEASES
# =====================================================================

event_rows = []


for date in macro[
    "date"
]:


    event = extract_event_path(
        date
    )


    if event is not None:

        event_rows.append(
            event
        )


event_paths = pd.DataFrame(
    event_rows
)


data = pd.merge(

    macro,

    event_paths,

    on="date",

    how="inner",

    validate="one_to_one"

)


data = (

    data

    .sort_values(
        "date"
    )

    .reset_index(
        drop=True
    )

)


print(
    "\nMatched macro + price releases:",
    len(data)
)


# =====================================================================
# 7. DEFINE FOUR GROUPS
# =====================================================================

GROUPS = {

    "NFP BEAT":

        data[
            "NFP_Beat"
        ],


    "NFP BEAT + ALL 3 USD BULLISH":

        data[
            "All_3_USD_Bullish"
        ],


    "NFP MISS":

        data[
            "NFP_Miss"
        ],


    "NFP MISS + ALL 3 USD BEARISH":

        data[
            "All_3_USD_Bearish"
        ]

}


def get_group(
    group_name
):

    return data[
        GROUPS[
            group_name
        ]
    ].copy()


# =====================================================================
# 8. CONFIRM FINAL COUNTS AFTER PRICE MATCHING
# =====================================================================

print("\n" + "=" * 100)
print("FINAL MATCHED GROUP COUNTS")
print("=" * 100)


for group_name in GROUPS.keys():

    subset = get_group(
        group_name
    )

    print(
        f"{group_name}: {len(subset)}"
    )


# =====================================================================
# 9. FIRST-MINUTE NUMERICAL SUMMARY
# =====================================================================

first_summary_rows = []


for group_name in GROUPS.keys():


    subset = get_group(
        group_name
    )


    values = subset[
        "FIRST_1"
    ].dropna()


    first_summary_rows.append({

        "Group":
            group_name,

        "N":
            len(values),

        "Mean_Pips":
            values.mean(),

        "Median_Pips":
            values.median(),

        "Std_Pips":
            values.std(),

        "P25":
            values.quantile(0.25),

        "P75":
            values.quantile(0.75),

        "Min":
            values.min(),

        "Max":
            values.max(),

        "Mean_Abs_Pips":
            values.abs().mean(),

        "Median_Abs_Pips":
            values.abs().median(),

        "Pct_EURUSD_Up":
            (
                values > 0
            ).mean(),

        "Pct_EURUSD_Down":
            (
                values < 0
            ).mean()

    })


first_summary = pd.DataFrame(
    first_summary_rows
)


print("\n" + "=" * 100)
print("FIRST-MINUTE PIP RETURN SUMMARY")
print("=" * 100)


print(

    first_summary

    .assign(

        Pct_EURUSD_Up=lambda x:
            x[
                "Pct_EURUSD_Up"
            ]
            *
            100,

        Pct_EURUSD_Down=lambda x:
            x[
                "Pct_EURUSD_Down"
            ]
            *
            100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# =====================================================================
# 10. FIRST-MINUTE PRICE OVERLAY
# =====================================================================

fig, axes = plt.subplots(

    2,
    2,

    figsize=(18, 12),

    sharey=True

)


axes = axes.flatten()


# Common y-axis

all_first_moves = data.loc[

    (
        data["NFP_Beat"]
        |
        data["NFP_Miss"]
    ),

    "FIRST_1"

].dropna()


if len(
    all_first_moves
) > 0:


    low = np.percentile(
        all_first_moves,
        1
    )


    high = np.percentile(
        all_first_moves,
        99
    )


    pad = max(

        5,

        (
            high - low
        )
        *
        0.10

    )


    first_y_min = low - pad
    first_y_max = high + pad


else:

    first_y_min = None
    first_y_max = None


for ax, group_name in zip(
    axes,
    GROUPS.keys()
):


    subset = get_group(
        group_name
    )


    # Individual moves

    for _, row in subset.iterrows():


        ax.plot(

            [0, 1],

            [
                0,
                row["FIRST_1"]
            ],

            alpha=0.14,

            linewidth=1

        )


    # Mean

    mean_move = subset[
        "FIRST_1"
    ].mean()


    median_move = subset[
        "FIRST_1"
    ].median()


    ax.plot(

        [0, 1],

        [
            0,
            mean_move
        ],

        linewidth=4,

        label=(
            f"Mean = "
            f"{mean_move:.1f} pips"
        )

    )


    ax.plot(

        [0, 1],

        [
            0,
            median_move
        ],

        linewidth=3,

        linestyle="--",

        label=(
            f"Median = "
            f"{median_move:.1f} pips"
        )

    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_title(

        f"{group_name}\n"
        f"N = {len(subset)}"

    )


    ax.set_xticks(
        [
            0,
            1
        ]
    )


    ax.set_xticklabels(
        [
            "08:30 Open",
            "08:30 Close"
        ]
    )


    ax.set_ylabel(
        "EUR/USD displacement (pips)"
    )


    if (
        first_y_min is not None
        and
        first_y_max is not None
    ):

        ax.set_ylim(
            first_y_min,
            first_y_max
        )


    ax.legend()


    ax.grid(
        True,
        alpha=0.25
    )


fig.suptitle(

    "EUR/USD RESPONSE DURING THE FIRST NFP MINUTE\n"
    "Baseline = 08:30 Open",

    fontsize=16

)


plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.94
    ]
)


plt.savefig(

    f"{OUTPUT_DIR}/"
    "01_first_minute_overlay.png",

    dpi=220,

    bbox_inches="tight"

)


plt.show()


# =====================================================================
# 11. FIRST-MINUTE MEAN + MEDIAN COMPARISON
# =====================================================================

group_labels = []
group_means = []
group_medians = []


for group_name in GROUPS.keys():


    subset = get_group(
        group_name
    )


    group_labels.append(
        group_name
    )


    group_means.append(

        subset[
            "FIRST_1"
        ].mean()

    )


    group_medians.append(

        subset[
            "FIRST_1"
        ].median()

    )


x = np.arange(
    len(group_labels)
)


width = 0.36


plt.figure(
    figsize=(15, 8)
)


plt.bar(

    x - width / 2,

    group_means,

    width,

    label="Mean"

)


plt.bar(

    x + width / 2,

    group_medians,

    width,

    label="Median"

)


plt.axhline(
    0,
    linestyle="--"
)


plt.xticks(

    x,

    group_labels,

    rotation=15,

    ha="right"

)


plt.ylabel(
    "EUR/USD first-minute return (pips)"
)


plt.title(

    "FIRST-MINUTE EUR/USD RETURN\n"
    "Mean and Median by NFP Release Group"

)


plt.legend()


plt.grid(
    True,
    axis="y",
    alpha=0.25
)


plt.tight_layout()


plt.savefig(

    f"{OUTPUT_DIR}/"
    "02_first_minute_mean_median.png",

    dpi=220,

    bbox_inches="tight"

)


plt.show()


# =====================================================================
# 12. FIRST-MINUTE RETURN DISTRIBUTIONS
# =====================================================================

fig, axes = plt.subplots(

    2,
    2,

    figsize=(18, 12),

    sharex=True,
    sharey=True

)


axes = axes.flatten()


# Common distribution range

first_dist_values = []


for group_name in GROUPS.keys():


    vals = get_group(
        group_name
    )[
        "FIRST_1"
    ].dropna().values


    first_dist_values.extend(
        vals
    )


first_dist_values = np.array(
    first_dist_values
)


dist_low = np.percentile(
    first_dist_values,
    1
)


dist_high = np.percentile(
    first_dist_values,
    99
)


bins = np.linspace(

    dist_low,

    dist_high,

    25

)


for ax, group_name in zip(
    axes,
    GROUPS.keys()
):


    subset = get_group(
        group_name
    )


    values = subset[
        "FIRST_1"
    ].dropna()


    mean_val = values.mean()

    median_val = values.median()


    ax.hist(

        values,

        bins=bins,

        alpha=0.75,

        edgecolor="black"

    )


    ax.axvline(

        0,

        linestyle="--",

        linewidth=1.5,

        label="0 pips"

    )


    ax.axvline(

        mean_val,

        linewidth=2.5,

        label=(
            f"Mean {mean_val:.1f}"
        )

    )


    ax.axvline(

        median_val,

        linewidth=2.5,

        linestyle=":",

        label=(
            f"Median {median_val:.1f}"
        )

    )


    ax.set_title(

        f"{group_name}\n"
        f"N = {len(values)}"

    )


    ax.set_xlabel(
        "First-minute EUR/USD return (pips)"
    )


    ax.set_ylabel(
        "Number of releases"
    )


    ax.legend()


    ax.grid(
        True,
        alpha=0.20
    )


fig.suptitle(

    "DISTRIBUTION OF EUR/USD RETURNS DURING THE FIRST NFP MINUTE\n"
    "08:30 Open → 08:30 Close",

    fontsize=16

)


plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.94
    ]
)


plt.savefig(

    f"{OUTPUT_DIR}/"
    "03_first_minute_return_distributions.png",

    dpi=220,

    bbox_inches="tight"

)


plt.show()


# =====================================================================
# 13. FUNCTION: POST-FIRST-MINUTE OVERLAY
# =====================================================================

def plot_post_overlay(
    ax,
    subset,
    horizon,
    title
):


    minutes = np.arange(
        0,
        horizon + 1
    )


    # Individual historical paths

    for _, row in subset.iterrows():


        path = np.array([

            row[
                f"POST_{minute}"
            ]

            for minute in minutes

        ])


        ax.plot(

            minutes,

            path,

            alpha=0.12,

            linewidth=0.9

        )


    # Mean path

    mean_path = np.array([

        subset[
            f"POST_{minute}"
        ].mean()

        for minute in minutes

    ])


    # Median path

    median_path = np.array([

        subset[
            f"POST_{minute}"
        ].median()

        for minute in minutes

    ])


    ax.plot(

        minutes,

        mean_path,

        linewidth=3.5,

        label="Mean"

    )


    ax.plot(

        minutes,

        median_path,

        linewidth=2.5,

        linestyle="--",

        label="Median"

    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_title(

        f"{title}\n"
        f"N = {len(subset)}"

    )


    ax.set_xlabel(
        "Minutes after 08:30 close"
    )


    ax.set_ylabel(
        "EUR/USD displacement (pips)"
    )


    ax.legend()


    ax.grid(
        True,
        alpha=0.25
    )


# =====================================================================
# 14. POST-FIRST-MINUTE NUMERICAL SUMMARY
# =====================================================================

post_summary_rows = []


for group_name in GROUPS.keys():


    subset = get_group(
        group_name
    )


    for horizon in POST_HORIZONS:


        values = subset[
            f"POST_{horizon}"
        ].dropna()


        post_summary_rows.append({

            "Group":
                group_name,

            "Horizon_Min":
                horizon,

            "N":
                len(values),

            "Mean_Pips":
                values.mean(),

            "Median_Pips":
                values.median(),

            "Std_Pips":
                values.std(),

            "P25":
                values.quantile(0.25),

            "P75":
                values.quantile(0.75),

            "Min":
                values.min(),

            "Max":
                values.max(),

            "Mean_Abs_Pips":
                values.abs().mean(),

            "Median_Abs_Pips":
                values.abs().median(),

            "Pct_EURUSD_Up":
                (
                    values > 0
                ).mean(),

            "Pct_EURUSD_Down":
                (
                    values < 0
                ).mean()

        })


post_summary = pd.DataFrame(
    post_summary_rows
)


print("\n" + "=" * 100)
print("POST-FIRST-MINUTE PIP RETURN SUMMARY")
print("=" * 100)


print(

    post_summary

    .assign(

        Pct_EURUSD_Up=lambda x:
            x[
                "Pct_EURUSD_Up"
            ]
            *
            100,

        Pct_EURUSD_Down=lambda x:
            x[
                "Pct_EURUSD_Down"
            ]
            *
            100

    )

    .round(2)

    .to_string(
        index=False
    )

)


# =====================================================================
# 15. CREATE POST-FIRST-MINUTE OVERLAYS
# =====================================================================

for horizon in POST_HORIZONS:


    fig, axes = plt.subplots(

        2,
        2,

        figsize=(18, 12),

        sharex=True,
        sharey=True

    )


    axes = axes.flatten()


    group_names = list(
        GROUPS.keys()
    )


    # -------------------------------------------------------------
    # COMMON Y SCALE
    # -------------------------------------------------------------

    all_values = []


    for group_name in group_names:


        subset = get_group(
            group_name
        )


        for minute in range(
            0,
            horizon + 1
        ):


            values = subset[
                f"POST_{minute}"
            ].dropna().values


            all_values.extend(
                values
            )


    all_values = np.array(
        all_values
    )


    if len(
        all_values
    ) > 0:


        low = np.percentile(
            all_values,
            1
        )


        high = np.percentile(
            all_values,
            99
        )


        padding = max(

            5,

            (
                high
                -
                low
            )
            *
            0.10

        )


        y_min = low - padding
        y_max = high + padding


    else:

        y_min = None
        y_max = None


    # -------------------------------------------------------------
    # PLOT GROUPS
    # -------------------------------------------------------------

    for ax, group_name in zip(
        axes,
        group_names
    ):


        subset = get_group(
            group_name
        )


        plot_post_overlay(

            ax,

            subset,

            horizon,

            group_name

        )


        if (
            y_min is not None
            and
            y_max is not None
        ):

            ax.set_ylim(
                y_min,
                y_max
            )


    fig.suptitle(

        f"EUR/USD PRICE ACTION DURING THE {horizon} MINUTES "
        f"AFTER THE FIRST NFP CANDLE\n"
        f"Baseline = 08:30 Close = 0 pips",

        fontsize=16

    )


    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.94
        ]
    )


    plt.savefig(

        f"{OUTPUT_DIR}/"
        f"overlay_post_T{horizon}.png",

        dpi=220,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================
# 16. MEAN PATH COMPARISON
# =====================================================================

for horizon in POST_HORIZONS:


    minutes = np.arange(
        0,
        horizon + 1
    )


    plt.figure(
        figsize=(15, 8)
    )


    for group_name in GROUPS.keys():


        subset = get_group(
            group_name
        )


        mean_path = [

            subset[
                f"POST_{minute}"
            ].mean()

            for minute in minutes

        ]


        plt.plot(

            minutes,

            mean_path,

            linewidth=3,

            label=(
                f"{group_name} "
                f"(N={len(subset)})"
            )

        )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes after 08:30 close"
    )


    plt.ylabel(
        "Mean EUR/USD displacement (pips)"
    )


    plt.title(

        f"MEAN EUR/USD PRICE PATH\n"
        f"First {horizon} Minutes After the 08:30 Candle Close"

    )


    plt.legend()


    plt.grid(
        True,
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        f"{OUTPUT_DIR}/"
        f"mean_path_post_T{horizon}.png",

        dpi=220,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================
# 17. MEDIAN PATH COMPARISON
# =====================================================================

for horizon in POST_HORIZONS:


    minutes = np.arange(
        0,
        horizon + 1
    )


    plt.figure(
        figsize=(15, 8)
    )


    for group_name in GROUPS.keys():


        subset = get_group(
            group_name
        )


        median_path = [

            subset[
                f"POST_{minute}"
            ].median()

            for minute in minutes

        ]


        plt.plot(

            minutes,

            median_path,

            linewidth=3,

            label=(
                f"{group_name} "
                f"(N={len(subset)})"
            )

        )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.xlabel(
        "Minutes after 08:30 close"
    )


    plt.ylabel(
        "Median EUR/USD displacement (pips)"
    )


    plt.title(

        f"MEDIAN EUR/USD PRICE PATH\n"
        f"First {horizon} Minutes After the 08:30 Candle Close"

    )


    plt.legend()


    plt.grid(
        True,
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        f"{OUTPUT_DIR}/"
        f"median_path_post_T{horizon}.png",

        dpi=220,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================
# 18. DISTRIBUTION OF TERMINAL PIP RETURNS
# =====================================================================
#
# IMPORTANT:
#
# These graphs are NOT the distribution of every point along
# the overlay.
#
# They show the distribution of the TERMINAL RETURN at the
# selected horizon.
#
#
# Example:
#
# T15 distribution =
#
#     08:45 close - 08:30 close
#
# for every historical event in the group.
#
# =====================================================================

for horizon in POST_HORIZONS:


    # -------------------------------------------------------------
    # Build common histogram range
    # -------------------------------------------------------------

    all_terminal_returns = []


    for group_name in GROUPS.keys():


        values = get_group(
            group_name
        )[
            f"POST_{horizon}"
        ].dropna().values


        all_terminal_returns.extend(
            values
        )


    all_terminal_returns = np.array(
        all_terminal_returns
    )


    low = np.percentile(
        all_terminal_returns,
        1
    )


    high = np.percentile(
        all_terminal_returns,
        99
    )


    # Ensure zero is inside the visible range

    low = min(
        low,
        0
    )


    high = max(
        high,
        0
    )


    histogram_bins = np.linspace(

        low,

        high,

        26

    )


    # -------------------------------------------------------------
    # Four-panel distribution
    # -------------------------------------------------------------

    fig, axes = plt.subplots(

        2,
        2,

        figsize=(18, 12),

        sharex=True,
        sharey=True

    )


    axes = axes.flatten()


    for ax, group_name in zip(
        axes,
        GROUPS.keys()
    ):


        subset = get_group(
            group_name
        )


        values = subset[
            f"POST_{horizon}"
        ].dropna()


        mean_val = values.mean()

        median_val = values.median()


        ax.hist(

            values,

            bins=histogram_bins,

            alpha=0.75,

            edgecolor="black"

        )


        ax.axvline(

            0,

            linestyle="--",

            linewidth=1.5,

            label="0 pips"

        )


        ax.axvline(

            mean_val,

            linewidth=2.5,

            label=(
                f"Mean = "
                f"{mean_val:.1f}"
            )

        )


        ax.axvline(

            median_val,

            linewidth=2.5,

            linestyle=":",

            label=(
                f"Median = "
                f"{median_val:.1f}"
            )

        )


        pct_up = (

            values > 0
        ).mean() * 100


        ax.set_title(

            f"{group_name}\n"
            f"N={len(values)} | "
            f"EUR/USD Up={pct_up:.1f}%"

        )


        ax.set_xlabel(
            f"T{horizon} return from 08:30 close (pips)"
        )


        ax.set_ylabel(
            "Number of releases"
        )


        ax.legend()


        ax.grid(
            True,
            alpha=0.20
        )


    fig.suptitle(

        f"DISTRIBUTION OF EUR/USD T{horizon} RETURNS\n"
        f"Return Measured From the Close of the 08:30 NFP Candle",

        fontsize=16

    )


    plt.tight_layout(
        rect=[
            0,
            0,
            1,
            0.94
        ]
    )


    plt.savefig(

        f"{OUTPUT_DIR}/"
        f"distribution_post_T{horizon}.png",

        dpi=220,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================
# 19. BOX PLOTS — OPTIONAL BUT USEFUL FOR THESIS
# =====================================================================
#
# These make distribution comparisons much easier than histograms
# when comparing all four groups directly.
#
# =====================================================================

for horizon in POST_HORIZONS:


    box_data = []

    labels = []


    for group_name in GROUPS.keys():


        values = get_group(
            group_name
        )[
            f"POST_{horizon}"
        ].dropna()


        box_data.append(
            values.values
        )


        labels.append(
            group_name
        )


    plt.figure(
        figsize=(16, 8)
    )


    plt.boxplot(

        box_data,

        labels=labels,

        showfliers=True,

        showmeans=True

    )


    plt.axhline(
        0,
        linestyle="--"
    )


    plt.ylabel(
        f"T{horizon} EUR/USD return from 08:30 close (pips)"
    )


    plt.title(

        f"DISTRIBUTION COMPARISON — T{horizon}\n"
        f"NFP Beat/Miss and Three-Factor Alignment"

    )


    plt.xticks(
        rotation=15,
        ha="right"
    )


    plt.grid(
        True,
        axis="y",
        alpha=0.25
    )


    plt.tight_layout()


    plt.savefig(

        f"{OUTPUT_DIR}/"
        f"boxplot_post_T{horizon}.png",

        dpi=220,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================
# 20. FINAL MEAN RETURN BY HORIZON
# =====================================================================
#
# One figure showing how terminal mean returns evolve through time.
#
# =====================================================================

plt.figure(
    figsize=(15, 8)
)


for group_name in GROUPS.keys():


    subset = get_group(
        group_name
    )


    means = [

        subset[
            f"POST_{horizon}"
        ].mean()

        for horizon in POST_HORIZONS

    ]


    plt.plot(

        POST_HORIZONS,

        means,

        marker="o",

        linewidth=3,

        label=(
            f"{group_name} "
            f"(N={len(subset)})"
        )

    )


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Minutes after 08:30 close"
)


plt.ylabel(
    "Mean EUR/USD return (pips)"
)


plt.title(

    "MEAN POST-NFP EUR/USD RETURN BY HORIZON\n"
    "Baseline = Close of the First 08:30 Candle"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.tight_layout()


plt.savefig(

    f"{OUTPUT_DIR}/"
    "mean_return_by_horizon.png",

    dpi=220,

    bbox_inches="tight"

)


plt.show()


# =====================================================================
# 21. FINAL MEDIAN RETURN BY HORIZON
# =====================================================================

plt.figure(
    figsize=(15, 8)
)


for group_name in GROUPS.keys():


    subset = get_group(
        group_name
    )


    medians = [

        subset[
            f"POST_{horizon}"
        ].median()

        for horizon in POST_HORIZONS

    ]


    plt.plot(

        POST_HORIZONS,

        medians,

        marker="o",

        linewidth=3,

        label=(
            f"{group_name} "
            f"(N={len(subset)})"
        )

    )


plt.axhline(
    0,
    linestyle="--"
)


plt.xlabel(
    "Minutes after 08:30 close"
)


plt.ylabel(
    "Median EUR/USD return (pips)"
)


plt.title(

    "MEDIAN POST-NFP EUR/USD RETURN BY HORIZON\n"
    "Baseline = Close of the First 08:30 Candle"

)


plt.legend()


plt.grid(
    True,
    alpha=0.25
)


plt.tight_layout()


plt.savefig(

    f"{OUTPUT_DIR}/"
    "median_return_by_horizon.png",

    dpi=220,

    bbox_inches="tight"

)


plt.show()


# =====================================================================
# 22. SAVE TABLES
# =====================================================================

first_summary.to_csv(

    f"{OUTPUT_DIR}/"
    "first_minute_summary.csv",

    index=False

)


post_summary.to_csv(

    f"{OUTPUT_DIR}/"
    "post_first_minute_summary.csv",

    index=False

)


data.to_csv(

    f"{OUTPUT_DIR}/"
    "event_level_paths.csv",

    index=False

)


# =====================================================================
# 23. FINAL MESSAGE
# =====================================================================

print("\n" + "=" * 100)
print("STUDY FINISHED")
print("=" * 100)


print(
    "All charts and CSV files saved to:"
)

print(
    OUTPUT_DIR
)


print(
"""

FILES PRODUCED
==============

FIRST MINUTE
------------

01_first_minute_overlay.png

    Shows the actual first 08:30 announcement candle.
    Baseline = 08:30 OPEN.

02_first_minute_mean_median.png

    Compares average and median first-minute pip return
    across the four macro groups.

03_first_minute_return_distributions.png

    Shows the historical distribution of first-minute returns.


POST-FIRST-MINUTE OVERLAYS
--------------------------

overlay_post_T5.png
overlay_post_T10.png
overlay_post_T15.png
overlay_post_T30.png
overlay_post_T45.png
overlay_post_T60.png

    ALL start at:

        08:30 CLOSE = 0 PIPS

    Therefore these isolate the price action that occurs
    AFTER the first NFP candle has already finished.


MEAN PATHS
----------

mean_path_post_T5.png
mean_path_post_T10.png
mean_path_post_T15.png
mean_path_post_T30.png
mean_path_post_T45.png
mean_path_post_T60.png


MEDIAN PATHS
------------

median_path_post_T5.png
median_path_post_T10.png
median_path_post_T15.png
median_path_post_T30.png
median_path_post_T45.png
median_path_post_T60.png


PIP RETURN DISTRIBUTIONS
------------------------

distribution_post_T5.png
distribution_post_T10.png
distribution_post_T15.png
distribution_post_T30.png
distribution_post_T45.png
distribution_post_T60.png

These are the distributions of TERMINAL returns.

Example:

    distribution_post_T15.png

shows:

    08:45 CLOSE
        minus
    08:30 CLOSE

for every historical NFP release in each category.


BOXPLOTS
--------

boxplot_post_T5.png
boxplot_post_T10.png
boxplot_post_T15.png
boxplot_post_T30.png
boxplot_post_T45.png
boxplot_post_T60.png


SUMMARY FIGURES
---------------

mean_return_by_horizon.png

median_return_by_horizon.png


CSV TABLES
----------

first_minute_summary.csv

post_first_minute_summary.csv

event_level_paths.csv


FOR YOUR THESIS I WOULD ESPECIALLY LOOK AT:

1. 01_first_minute_overlay.png

2. 03_first_minute_return_distributions.png

3. overlay_post_T5.png

4. overlay_post_T15.png

5. overlay_post_T60.png

6. distribution_post_T5.png

7. distribution_post_T15.png

8. distribution_post_T60.png

9. mean_return_by_horizon.png

10. median_return_by_horizon.png

"""
)